# Retail Sales Analytics & Business Insights Dashboard
## Exploratory Data Analysis (EDA)

# Project Overview

## Objective
Build a business-focused retail analytics notebook that goes beyond basic EDA and highlights customer behavior, category performance, discount impact, shipping efficiency, forecasting, and profit leakage.

## Dataset Overview
This notebook uses a cleaned retail sales dataset containing orders, customers, products, geography, discounts, shipping details, sales, and profit fields.

## Business Questions
- Which customers and segments contribute the most value?
- Which products, categories, and regions drive profit or loss?
- How does discounting affect profitability?
- Which shipping patterns and delays matter operationally?
- What future sales trend can be expected from historical monthly data?

## Tools Used
- Python
- Pandas
- NumPy
- Plotly
- Statsmodels
- Scikit-learn

## Key Findings
- Champions and Loyal Customers are the highest-value customer groups.
- Discounting is a major driver of margin pressure in weaker bands.
- A small set of products and sub-categories contributes a large share of value.
- Some regions, cities, and product groups show clear loss concentration.
- Forecasting adds planning value for inventory and campaign decisions.

In [88]:
# CORE IMPORTS
import os
import warnings

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Better renderer for notebooks
pio.renderers.default = "notebook_connected"

from IPython.display import display, HTML

from typing import Iterable, Optional, Union, Sequence, List

from pathlib import Path

from datetime import datetime

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

from scipy.stats import shapiro, mannwhitneyu, kruskal
from itertools import combinations
from statsmodels.stats.multitest import multipletests

In [89]:
# UTILITY FUNCTIONS

# ==========================================
# THEME CONFIGURATION
# ==========================================

# Core semantic colors
PRIMARY_BLUE = "#2563EB"      # Revenue / Sales
PRIMARY_GREEN = "#16A34A"     # Profit / Positive
PRIMARY_RED = "#DC2626"       # Loss / Negative
PRIMARY_ORANGE = "#E45E15"    # Discount / Warning
PRIMARY_PURPLE = "#7C3AED"    # Counts / Neutral
PRIMARY_PURPLE_LIGHT = "#EDE9FE"

NEUTRAL_GRAY = "#64748B"

# UI colors
BG_LIGHT = "#F8FAFC"
GRID_COLOR = "rgba(0,0,0,0.08)"
TEXT_COLOR = "#1E293B"

# Hover styling 
HOVER_BG_COLOR = "#1B1B1B"      # stable dark hover background
HOVER_FONT_COLOR = "#FFFFFF"    # stable readable hover text
HOVER_BORDER_COLOR = "#111827"  # keep border consistent with background
HOVER_FONT_SIZE = 13

# Color scales
COUNT_SCALE = ["#F5F3FF", PRIMARY_PURPLE]
LOSS_SCALE = ["#FEE2E2", "#FCA5A5", PRIMARY_RED, "#7F1D1D"]

# Typography
FONT_FAMILY = "Inter"
TITLE_SIZE = 22
SUBTITLE_SIZE = 15

# Plotly
PLOTLY_TEMPLATE = "plotly_white"

# Notebook configs
OUTPUT_DIR = "../exports"
MAX_LABEL_LENGTH = 35

DEFAULT_MARGIN = dict(l=40, r=40, t=80, b=40)

# Default Plotly color cycle
PRIMARY_COLOR_SEQUENCE = [
    PRIMARY_BLUE,
    PRIMARY_GREEN,
    PRIMARY_ORANGE,
    PRIMARY_PURPLE,
    PRIMARY_RED,
]

# Semantic business mapping
SEMANTIC_COLORS = {
    "sales": PRIMARY_BLUE,
    "revenue": PRIMARY_BLUE,

    "profit": PRIMARY_GREEN,
    "positive": PRIMARY_GREEN,

    "loss": PRIMARY_RED,
    "negative": PRIMARY_RED,

    "discount": PRIMARY_ORANGE,
    "warning": PRIMARY_ORANGE,

    "count": PRIMARY_PURPLE,
    "neutral": NEUTRAL_GRAY,
}

# Default figure sizes
DEFAULT_HEIGHT = 650
DEFAULT_WIDTH = 1200

DEFAULT_DATE_CANDIDATES = ["Order_Date", "Order Date", "Date", "OrderDate"]


# ==========================================
# INTERNAL HELPERS
# ==========================================

def _is_missing(value) -> bool:
    """
    Safe missing-value check that avoids ambiguous truth-value errors
    for array-like objects.
    """
    try:
        result = pd.isna(value)
    except Exception:
        return False

    return isinstance(result, (bool, np.bool_)) and bool(result)


def _to_float_or_none(value) -> Optional[float]:
    """
    Convert a value to float if possible; otherwise return None.
    """
    if _is_missing(value):
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


# ==========================================
# FORMATTING FUNCTIONS
# ==========================================

Number = Union[int, float]


def fmt_money(
    x: Optional[Number],
    symbol: str = "$",
    decimals: int = 2,
) -> str:
    """
    Format a numeric value as currency.
    Supports K/M abbreviations.
    """
    value = _to_float_or_none(x)
    if value is None:
        return "N/A"

    sign = "-" if value < 0 else ""
    ax = abs(value)

    if ax >= 1_000_000:
        return f"{sign}{symbol}{ax / 1_000_000:.2f}M"

    if ax >= 1_000:
        return f"{sign}{symbol}{ax / 1_000:.1f}K"

    return f"{sign}{symbol}{ax:,.{decimals}f}"


def fmt_percent(
    x: Optional[Number],
    decimals: int = 2,
) -> str:
    """
    Format a numeric value as percentage.
    """
    value = _to_float_or_none(x)
    if value is None:
        return "N/A"

    return f"{value:,.{decimals}f}%"


def fmt_number(
    x: Optional[Number],
    decimals: int = 0,
) -> str:
    """
    Format a numeric value with commas.
    """
    value = _to_float_or_none(x)
    if value is None:
        return "N/A"

    return f"{value:,.{decimals}f}"


def shorten_label(
    label: object,
    max_len: int = MAX_LABEL_LENGTH,
) -> str:
    """
    Shorten long labels for charts safely and consistently.
    """
    if _is_missing(label):
        return "N/A"

    label = str(label).strip()

    if max_len <= 3:
        raise ValueError("max_len must be greater than 3.")

    if len(label) <= max_len:
        return label

    return label[: max_len - 3].rstrip() + "..."


# ==========================================
# PLOTTING FUNCTIONS
# ==========================================

def style_plotly(
    fig,
    title: str,
    height: int = DEFAULT_HEIGHT,
    width: Optional[int] = None,
    x_title: Optional[str] = None,
    y_title: Optional[str] = None,
    showlegend: Optional[bool] = None,
    hovermode: Optional[str] = None,
    x_tickangle: Optional[int] = None,
    x_range: Optional[list] = None,
    y_range: Optional[list] = None,
    x_type: Optional[str] = None,
    y_type: Optional[str] = None,
    x_tickformat: Optional[str] = None,
    y_tickformat: Optional[str] = None,
    legend_title: Optional[str] = None,
    margin: Optional[dict] = None,
    bargap: Optional[float] = None,
    coloraxis_showscale: Optional[bool] = None,
    coloraxis_colorbar_title: Optional[str] = None,
    legend_orientation: Optional[str] = None,
    legend_x: Optional[float] = None,
    legend_y: Optional[float] = None,
    legend_xanchor: Optional[str] = None,
    legend_yanchor: Optional[str] = None,
    x_showgrid: bool = True,
    y_showgrid: bool = True,
    geo_kwargs: Optional[dict] = None,
    title_x: float = 0.5,
):
    """Apply a consistent Plotly style across the notebook."""

    if fig is None:
        raise ValueError("fig cannot be None.")

    layout_kwargs = {
        "title": {
            "text": title,
            "x": title_x,
            "font": {
                "size": TITLE_SIZE,
                "family": FONT_FAMILY,
                "color": TEXT_COLOR,
            },
        },
        "template": PLOTLY_TEMPLATE,
        "height": height,
        "autosize": True,
        "plot_bgcolor": "white",
        "paper_bgcolor": "white",
        "font": {
            "family": FONT_FAMILY,
            "color": TEXT_COLOR,
        },
        "margin": margin if margin is not None else DEFAULT_MARGIN,
        "colorway": PRIMARY_COLOR_SEQUENCE,
        "hoverlabel": {
            "bgcolor": HOVER_BG_COLOR,
            "bordercolor": HOVER_BORDER_COLOR,
            "font": {
                "family": FONT_FAMILY,
                "color": HOVER_FONT_COLOR,
                "size": HOVER_FONT_SIZE,
            },
        },
    }

    if width is not None:
        layout_kwargs["width"] = width

    if showlegend is not None:
        layout_kwargs["showlegend"] = showlegend
    if hovermode is not None:
        layout_kwargs["hovermode"] = hovermode
    if legend_title is not None:
        layout_kwargs["legend_title_text"] = legend_title
    if bargap is not None:
        layout_kwargs["bargap"] = bargap

    if any(
        v is not None
        for v in [legend_orientation, legend_x, legend_y, legend_xanchor, legend_yanchor]
    ):
        layout_kwargs["legend"] = {}
        if legend_orientation is not None:
            layout_kwargs["legend"]["orientation"] = legend_orientation
        if legend_x is not None:
            layout_kwargs["legend"]["x"] = legend_x
        if legend_y is not None:
            layout_kwargs["legend"]["y"] = legend_y
        if legend_xanchor is not None:
            layout_kwargs["legend"]["xanchor"] = legend_xanchor
        if legend_yanchor is not None:
            layout_kwargs["legend"]["yanchor"] = legend_yanchor

    if coloraxis_showscale is not None or coloraxis_colorbar_title is not None:
        coloraxis_kwargs = {}
        if coloraxis_showscale is not None:
            coloraxis_kwargs["showscale"] = coloraxis_showscale
        if coloraxis_colorbar_title is not None:
            coloraxis_kwargs["colorbar"] = {"title": {"text": coloraxis_colorbar_title}}
        layout_kwargs["coloraxis"] = coloraxis_kwargs

    fig.update_layout(**layout_kwargs)

    fig.update_xaxes(
        title_text=x_title if x_title else None,
        gridcolor=GRID_COLOR,
        zeroline=False,
        tickangle=x_tickangle,
        showgrid=x_showgrid,
        automargin=True,
        range=x_range,
        type=x_type,
        tickformat=x_tickformat,
    )
    fig.update_yaxes(
        title_text=y_title if y_title else None,
        gridcolor=GRID_COLOR,
        zeroline=False,
        showgrid=y_showgrid,
        automargin=True,
        range=y_range,
        type=y_type,
        tickformat=y_tickformat,
    )

    if geo_kwargs:
        fig.update_geos(**geo_kwargs)

    return fig

def apply_dark_theme(fig, title: str, **kwargs):
    """Compatibility wrapper for the notebook's standard Plotly theme."""
    return style_plotly(fig, title=title, **kwargs)

def add_hover_template(*args, **kwargs):
    """Compatibility wrapper for the notebook's hover-template builder."""
    return build_hover_template(*args, **kwargs)


def _hover_value(
    expr: str,
    format_spec: Optional[str] = None,
    prefix: str = "",
    suffix: str = "",
) -> str:
    """
    Build a Plotly hover token safely.

    Parameters
    ----------
    expr : str
        Plotly expression such as "x", "y", "z", "customdata[0]", "label".
    format_spec : str | None
        Plotly format specifier, e.g. ",.2f", ",.0f", ".2%".
        Use None for raw text / categorical values.
    prefix : str
        Text to place before the token, such as "$".
    suffix : str
        Text to place after the token, such as "%".

    Returns
    -------
    str
        A Plotly hover token ready to embed in hovertemplate.
    """
    expr = str(expr).strip()

    if expr.startswith("%{") and expr.endswith("}"):
        expr = expr[2:-1]

    token = f"%{{{expr}}}" if not format_spec else f"%{{{expr}:{format_spec}}}"
    return f"{prefix}{token}{suffix}"


def build_hover_template(
    *,
    title_label: Optional[str] = None,
    title_expr: str = "x",
    title_format: Optional[str] = None,
    title_prefix: str = "",
    title_suffix: str = "",
    title_template: Optional[str] = None,
    lines: Sequence[dict] = (),
) -> str:
    """
    Generic hover template builder.

    Each item in `lines` can contain:
        label       -> line label shown in hover
        expr        -> Plotly expression, e.g. "y", "customdata[0]"
        format_spec -> Plotly format spec, e.g. ",.2f", ".2%"
        prefix      -> text before the token
        suffix      -> text after the token
    """
    parts: List[str] = []

    if title_template:
        parts.append(title_template)
    elif title_label is not None:
        value = _hover_value(
            title_expr,
            format_spec=title_format,
            prefix=title_prefix,
            suffix=title_suffix,
        )
        parts.append(f"<b>{title_label}:</b> {value}")

    for line in lines or ():
        if not isinstance(line, dict):
            continue

        label = str(line.get("label", "")).strip()
        expr = line.get("expr", "y")
        format_spec = line.get("format_spec", None)
        prefix = line.get("prefix", "")
        suffix = line.get("suffix", "")

        value = _hover_value(
            expr,
            format_spec=format_spec,
            prefix=prefix,
            suffix=suffix,
        )

        if label:
            parts.append(f"<b>{label}:</b> {value}")
        else:
            parts.append(value)

    parts.append("<extra></extra>")
    return "<br>".join(parts)


def money_hover_template(
    x_label: str = "Category",
    y_label: str = "Value",
    *,
    x_expr: str = "x",
    y_expr: str = "y",
    x_format: Optional[str] = None,
    y_format: str = ",.2f",
    x_prefix: str = "",
    y_prefix: str = "$",
    x_suffix: str = "",
    y_suffix: str = "",
) -> str:
    """
    Standard hover template for money-based charts.

    Best for:
    - category/value bars
    - horizontal bars
    - basic sales/profit charts
    """
    return build_hover_template(
        title_label=x_label,
        title_expr=x_expr,
        title_format=x_format,
        title_prefix=x_prefix,
        title_suffix=x_suffix,
        lines=[
            {
                "label": y_label,
                "expr": y_expr,
                "format_spec": y_format,
                "prefix": y_prefix,
                "suffix": y_suffix,
            }
        ],
    )


def percent_hover_template(
    x_label: str = "Category",
    y_label: str = "Value",
    *,
    x_expr: str = "x",
    y_expr: str = "y",
    x_format: Optional[str] = None,
    y_format: str = ",.2f",
    x_prefix: str = "",
    y_prefix: str = "",
    x_suffix: str = "",
    y_suffix: str = "%",
) -> str:
    """
    Standard hover template for percent-based charts.

    Best when the plotted values are already in percentage units (0-100).
    """
    return build_hover_template(
        title_label=x_label,
        title_expr=x_expr,
        title_format=x_format,
        title_prefix=x_prefix,
        title_suffix=x_suffix,
        lines=[
            {
                "label": y_label,
                "expr": y_expr,
                "format_spec": y_format,
                "prefix": y_prefix,
                "suffix": y_suffix,
            }
        ],
    )


def count_hover_template(
    x_label: str = "Category",
    y_label: str = "Count",
    *,
    x_expr: str = "x",
    y_expr: str = "y",
    x_format: Optional[str] = None,
    y_format: str = ",.0f",
    x_prefix: str = "",
    y_prefix: str = "",
    x_suffix: str = "",
    y_suffix: str = "",
) -> str:
    """
    Standard hover template for count / frequency charts.
    """
    return build_hover_template(
        title_label=x_label,
        title_expr=x_expr,
        title_format=x_format,
        title_prefix=x_prefix,
        title_suffix=x_suffix,
        lines=[
            {
                "label": y_label,
                "expr": y_expr,
                "format_spec": y_format,
                "prefix": y_prefix,
                "suffix": y_suffix,
            }
        ],
    )


def customdata_hover_template(
    title_label: str = "Item",
    title_expr: str = "x",
    *,
    title_format: Optional[str] = None,
    title_prefix: str = "",
    title_suffix: str = "",
    lines: Sequence[dict] = (),
) -> str:
    """
    Flexible hover template for charts that use customdata or mixed fields.

    Example line dict:
        {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"}
    """
    return build_hover_template(
        title_label=title_label,
        title_expr=title_expr,
        title_format=title_format,
        title_prefix=title_prefix,
        title_suffix=title_suffix,
        lines=lines,
    )


def treemap_hover_template(
    label_template: str = "<b>%{label}</b>",
    value_label: str = "Sales",
    value_expr: str = "value",
    value_format: str = ",.0f",
) -> str:
    """
    Hover template for treemap / sunburst style charts.
    """
    return build_hover_template(
        title_template=label_template,
        lines=[
            {
                "label": value_label,
                "expr": value_expr,
                "format_spec": value_format,
            },
            {
                "label": "Share of parent",
                "expr": "percentParent",
                "format_spec": ".2%",
            },
            {
                "label": "Share of total",
                "expr": "percentRoot",
                "format_spec": ".2%",
            },
        ],
    )


def heatmap_hover_template(
    row_label: str = "Row",
    col_label: str = "Column",
    value_label: str = "Value",
    *,
    row_expr: str = "y",
    col_expr: str = "x",
    value_expr: str = "z",
    value_format: str = ",.2f",
    extra_lines: Sequence[dict] = (),
) -> str:
    """
    Hover template for heatmaps or matrix-style charts.

    extra_lines can be used for customdata fields, for example:
        [{"label": "Sales", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"}]
    """
    lines = [
        {"label": row_label, "expr": row_expr, "format_spec": None},
        {"label": col_label, "expr": col_expr, "format_spec": None},
        {
            "label": value_label,
            "expr": value_expr,
            "format_spec": value_format,
        },
    ]
    lines.extend(list(extra_lines))

    return build_hover_template(lines=lines)


# ==========================================
# DATA VALIDATION
# ==========================================

def validate_columns(df: pd.DataFrame, required_cols: Iterable[str]) -> None:
    """Raise an error if required columns are missing."""
    if df is None:
        raise ValueError("DataFrame is empty.")

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def validate_non_empty(df: pd.DataFrame) -> None:
    """Ensure DataFrame is not empty."""
    if df is None or df.empty:
        raise ValueError("DataFrame is empty.")


def get_date_col(
    df: pd.DataFrame,
    candidates: Sequence[str] = DEFAULT_DATE_CANDIDATES,
) -> Optional[str]:
    """Return the first matching date column name, or None."""
    return next((c for c in candidates if c in df.columns), None)


def require(df, cols):
    validate_non_empty(df)
    validate_columns(df, cols)
    

def ensure_shipping_days(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy of df with Shipping_Days guaranteed to exist
    whenever Order/Ship date columns are available.
    """
    work_df = df.copy()

    if "Shipping_Days" in work_df.columns:
        return work_df

    order_date_col = next(
        (c for c in ["Order_Date", "Order Date", "OrderDate"] if c in work_df.columns),
        None
    )
    ship_date_col = next(
        (c for c in ["Ship_Date", "Ship Date", "ShipDate"] if c in work_df.columns),
        None
    )

    if order_date_col is None or ship_date_col is None:
        raise ValueError(
            "Shipping_Days cannot be derived because order/ship date columns are missing."
        )

    work_df[order_date_col] = pd.to_datetime(work_df[order_date_col], errors="coerce")
    work_df[ship_date_col] = pd.to_datetime(work_df[ship_date_col], errors="coerce")
    work_df["Shipping_Days"] = (work_df[ship_date_col] - work_df[order_date_col]).dt.days

    return work_df



# ==========================================
# AGGREGATION FUNCTIONS
# ==========================================

def _normalize_group_cols(group_col: Union[str, Sequence[str]]) -> List[str]:
    """Normalize a group column input into a list of column names."""
    if isinstance(group_col, str):
        return [group_col]
    return list(group_col)


def aggregate_sales_profit(
    df: pd.DataFrame,
    group_col: Union[str, Sequence[str]],
    sort_by: str = "Sales",
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Aggregate Sales and Profit by one or more grouping columns.

    Parameters
    ----------
    df : pd.DataFrame
        Source dataframe.
    group_col : str | sequence of str
        Grouping column(s).
    sort_by : str
        Column used for sorting the output.
    ascending : bool
        Sort order.

    Returns
    -------
    pd.DataFrame
        Aggregated dataframe with Sales, Profit, and Profit_Margin_%.
    """
    validate_non_empty(df)

    group_cols = _normalize_group_cols(group_col)
    validate_columns(df, group_cols + ["Sales", "Profit"])

    result = (
        df.groupby(group_cols, as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
        )
    )

    result["Profit_Margin_%"] = np.where(
        result["Sales"] != 0,
        (result["Profit"] / result["Sales"]) * 100,
        np.nan,
    )

    if sort_by in result.columns:
        result = result.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    return result


def aggregate_metrics(
    df: pd.DataFrame,
    group_col: Union[str, Sequence[str]],
    include_quantity: bool = True,
    order_col: str = "Order_ID",
    sort_by: str = "Sales",
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Aggregate common business metrics in a reusable way.

    Parameters
    ----------
    df : pd.DataFrame
        Source dataframe.
    group_col : str | sequence of str
        Grouping column(s).
    include_quantity : bool
        Whether to aggregate Quantity.
    order_col : str
        Column used to count unique orders if available.
    sort_by : str
        Column used for sorting the output.
    ascending : bool
        Sort order.

    Returns
    -------
    pd.DataFrame
        Aggregated dataframe with Sales, Profit, Quantity (optional),
        Orders, and Profit_Margin_%.
    """
    validate_non_empty(df)

    group_cols = _normalize_group_cols(group_col)

    required_cols = group_cols + ["Sales", "Profit"]
    if include_quantity:
        required_cols.append("Quantity")
    if order_col in df.columns:
        required_cols.append(order_col)

    validate_columns(df, required_cols)

    agg_dict = {
        "Sales": ("Sales", "sum"),
        "Profit": ("Profit", "sum"),
    }

    if include_quantity:
        agg_dict["Quantity"] = ("Quantity", "sum")

    if order_col in df.columns:
        agg_dict["Orders"] = (order_col, "nunique")
    else:
        agg_dict["Orders"] = ("Sales", "size")

    result = (
        df.groupby(group_cols, as_index=False)
        .agg(**agg_dict)
    )

    result["Profit_Margin_%"] = np.where(
        result["Sales"] != 0,
        (result["Profit"] / result["Sales"]) * 100,
        np.nan,
    )

    if sort_by in result.columns:
        result = result.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    return result


# ==========================================
# EXPORT FUNCTIONS
# ==========================================

def save_csv(
    df: pd.DataFrame,
    filename: str,
    output_folder: str = "../exports",
    index: bool = False,
    verbose: bool = True,
) -> Path:
    """
    Save a dataframe as CSV safely and consistently.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe to save.

    filename : str
        CSV filename.

    output_folder : str
        Folder where CSV should be saved.

    index : bool
        Whether to save dataframe index.

    verbose : bool
        Print save confirmation message.

    Returns
    -------
    Path
        Full saved file path.
    """
    validate_non_empty(df)

    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    # Ensure .csv extension exists
    if not filename.lower().endswith(".csv"):
        filename += ".csv"

    full_path = output_path / filename

    df.to_csv(full_path, index=index)

    if verbose:
        print(f"✅ Saved: {full_path}")

    return full_path


# ==========================================
# DEBUG / LOGGING HELPERS
# ==========================================

def log_step(message: str, level: str = "INFO") -> None:
    """
    Print a consistent progress log message for notebook steps.

    Parameters
    ----------
    message : str
        The message to display.
    level : str
        Log level: INFO, SUCCESS, WARN, ERROR, STEP
    """
    level = str(level).upper().strip()

    icons = {
        "INFO": "ℹ️",
        "SUCCESS": "✅",
        "WARN": "⚠️",
        "ERROR": "❌",
        "STEP": "🔹",
    }

    icon = icons.get(level, "•")
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {icon} {message}")


# ==========================================
# COLOR HELPERS
# ==========================================

def get_profit_color(
    value: float,
    positive_color: str = PRIMARY_GREEN,
    negative_color: str = PRIMARY_RED,
    neutral_color: str = PRIMARY_ORANGE,
) -> str:
    """
    Return a standardized color based on profit value.

    Parameters
    ----------
    value : float
        Profit value to evaluate.
    positive_color : str
        Color for positive values.
    negative_color : str
        Color for negative values.
    neutral_color : str
        Color for zero or missing values.

    Returns
    -------
    str
        Hex or named color string.
    """
    try:
        if _is_missing(value):
            return neutral_color

        value = float(value)

        if value > 0:
            return positive_color
        elif value < 0:
            return negative_color
        else:
            return neutral_color

    except (TypeError, ValueError):
        return neutral_color


def quadrant_label(
    sales: float,
    profit: float,
    sales_mid: Optional[float] = None,
    profit_mid: Optional[float] = None,
) -> str:
    """
    Classify a point into a sales-profit quadrant.

    Parameters
    ----------
    sales : float
        Sales value.

    profit : float
        Profit value.

    sales_mid : float, default=0
        Threshold separating high vs low sales.

    profit_mid : float, default=0
        Threshold separating high vs low profit.

    Returns
    -------
    str
        Quadrant classification label.
    """
    if _is_missing(sales) or _is_missing(profit):
        return "Unknown"
    if sales_mid is None or profit_mid is None:
        raise ValueError(
            "quadrant_label requires explicit sales_mid and profit_mid values."
        )
    if sales >= sales_mid and profit >= profit_mid:
        return "High Sales | High Profit"
    elif sales >= sales_mid and profit < profit_mid:
        return "High Sales | Low Profit"
    elif sales < sales_mid and profit >= profit_mid:
        return "Low Sales | High Profit"
    else:
        return "Low Sales | Low Profit"


def rfm_segment(rfm_score: float) -> str:
    """
    Map an RFM score to a customer segment.

    Parameters
    ----------
    rfm_score : float
        Combined RFM score.

    Returns
    -------
    str
        Customer segment label.
    """
    if _is_missing(rfm_score):
        return "Unknown"

    score = int(rfm_score)

    if score >= 9:
        return "Champions"
    elif score >= 7:
        return "Loyal Customers"
    elif score >= 5:
        return "Potential Loyalists"
    elif score >= 3:
        return "At Risk"
    else:
        return "Lost Customers"


def profitability_segment(
    row: pd.Series,
    sales_mid: float,
    margin_mid: float,
) -> str:
    """
    Classify profitability efficiency segments.

    Parameters
    ----------
    row : pd.Series
        Row containing profitability metrics.

    sales_mid : float
        Median sales contribution threshold.

    margin_mid : float
        Median profit margin threshold.

    Returns
    -------
    str
        Profitability efficiency segment label.
    """
    if row["Profit Margin %"] < 0:
        return "Loss Makers"

    elif (
        row["Sales Contribution %"] >= sales_mid
        and row["Profit Margin %"] >= margin_mid
    ):
        return "Core Winners"

    elif (
        row["Sales Contribution %"] >= sales_mid
        and row["Profit Margin %"] < margin_mid
    ):
        return "High Volume, Low Efficiency"

    elif (
        row["Sales Contribution %"] < sales_mid
        and row["Profit Margin %"] >= margin_mid
    ):
        return "Hidden Gems"

    else:
        return "Weak Performers"


SHIP_MODE_PALETTE = {
    "Same Day": PRIMARY_RED,
    "First Class": PRIMARY_ORANGE,
    "Second Class": PRIMARY_BLUE,
    "Standard Class": PRIMARY_GREEN,
}


def style_ship_mode(col: pd.Series) -> list[str]:
    """
    Apply styling for ship mode cells in a pandas Styler column.

    Parameters
    ----------
    col : pd.Series
        Ship_Mode column passed by Styler.apply.

    Returns
    -------
    list[str]
        CSS styles for each value in the column.
    """
    return [
        (
            f"background-color: {SHIP_MODE_PALETTE.get(v, '#7f7f7f')}; "
            "color: white; "
            "font-weight: bold;"
        )
        for v in col
    ]



# ==========================================
# RFM HELPERS
# ==========================================

def build_rfm_table(base_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a robust RFM table from raw transactional data.

    Safe against:
    - missing columns
    - bad dates
    - bad sales values
    - empty/degenerate customer distributions
    - small datasets that break qcut
    """
    require(base_df, ["Customer_ID", "Customer_Name", "Order_Date", "Order_ID", "Sales"])

    work = base_df.loc[:, ["Customer_ID", "Customer_Name", "Order_Date", "Order_ID", "Sales"]].copy()

    work["Customer_ID"] = work["Customer_ID"].astype(str).str.strip()
    work["Customer_Name"] = work["Customer_Name"].astype(str).str.strip()
    work["Order_ID"] = work["Order_ID"].astype(str).str.strip()
    work["Order_Date"] = pd.to_datetime(work["Order_Date"], errors="coerce")
    work["Sales"] = pd.to_numeric(work["Sales"], errors="coerce")

    work = work.dropna(subset=["Customer_ID", "Order_Date", "Order_ID", "Sales"])
    work = work[(work["Customer_ID"] != "") & (work["Order_ID"] != "")]

    if work.empty:
        raise ValueError(
            "RFM cannot be built: no valid rows remain after cleaning."
        )

    reference_date = work["Order_Date"].max() + pd.Timedelta(days=1)

    rfm = (
        work.groupby(["Customer_ID", "Customer_Name"], as_index=False)
        .agg(
            Recency=("Order_Date", lambda s: (reference_date - s.max()).days),
            Frequency=("Order_ID", "nunique"),
            Monetary=("Sales", "sum"),
        )
    )

    def _quintile_score(series: pd.Series, ascending: bool = True) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce")

        if s.notna().sum() == 0:
            return pd.Series([3] * len(s), index=s.index, dtype=int)

        # Percent-rank scoring is more stable than qcut on small or tied datasets.
        pct = s.rank(method="average", pct=True)
        score = np.ceil(pct * 5).astype("Int64").clip(1, 5)

        # For recency: lower recency = better score
        if not ascending:
            score = 6 - score

        return score.astype(int)

    rfm["R_Score"] = _quintile_score(rfm["Recency"], ascending=False)
    rfm["F_Score"] = _quintile_score(rfm["Frequency"], ascending=True)
    rfm["M_Score"] = _quintile_score(rfm["Monetary"], ascending=True)

    rfm["RFM_Score"] = (
        rfm["R_Score"].astype(int)
        + rfm["F_Score"].astype(int)
        + rfm["M_Score"].astype(int)
    )

    rfm["Segment"] = rfm["RFM_Score"].apply(rfm_segment)

    segment_order = [
        "Champions",
        "Loyal Customers",
        "Potential Loyalists",
        "At Risk",
        "Lost Customers",
        "Unknown",
    ]
    rfm["Segment"] = pd.Categorical(
        rfm["Segment"],
        categories=segment_order,
        ordered=True,
    )

    return rfm.sort_values(
        ["RFM_Score", "Monetary"],
        ascending=[False, False],
    ).reset_index(drop=True)
    
    
    
# -----------------------------------------------------
# Forecast Helpers
# -----------------------------------------------------

def build_sales_forecast_table(base_df: pd.DataFrame, horizon: int = 6) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build a robust monthly sales forecast table.

    Returns:
        monthly_sales: historical monthly aggregation
        forecast_future: future months + forecast values
        forecast_plot_df: combined historical + forecast table
    """
    require(base_df, ["Order_Date", "Sales"])

    forecast_df = base_df.loc[:, ["Order_Date", "Sales"]].copy()
    forecast_df["Order_Date"] = pd.to_datetime(forecast_df["Order_Date"], errors="coerce")
    forecast_df["Sales"] = pd.to_numeric(forecast_df["Sales"], errors="coerce")
    forecast_df = forecast_df.dropna(subset=["Order_Date", "Sales"])

    if forecast_df.empty:
        raise ValueError("Forecast cannot be built: no valid rows remain after cleaning.")

    # Monthly aggregation
    monthly_sales = (
        forecast_df.assign(
            Month=forecast_df["Order_Date"].dt.to_period("M").dt.to_timestamp()
        )
        .groupby("Month", as_index=False)["Sales"]
        .sum()
        .sort_values("Month")
        .reset_index(drop=True)
    )

    # Fill any missing months so the time series is continuous
    full_month_index = pd.date_range(
        start=monthly_sales["Month"].min(),
        end=monthly_sales["Month"].max(),
        freq="MS",
    )

    monthly_sales = (
        monthly_sales.set_index("Month")
        .reindex(full_month_index, fill_value=0)
        .rename_axis("Month")
        .reset_index()
    )

    monthly_sales_ts = monthly_sales.set_index("Month")["Sales"].astype(float)

    if len(monthly_sales_ts) < 2:
        raise ValueError("Forecast cannot be built: not enough monthly observations.")

    # -----------------------------------------------------
    # Forecast model with safe fallbacks
    # -----------------------------------------------------
    forecast_values = None

    try:
        if monthly_sales_ts.nunique() == 1:
            # Constant series: simplest safe fallback
            forecast_values = np.repeat(monthly_sales_ts.iloc[-1], horizon)
        elif len(monthly_sales_ts) >= 24:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ExponentialSmoothing(
                    monthly_sales_ts,
                    trend="add",
                    seasonal="add",
                    seasonal_periods=12,
                ).fit(optimized=True)
            forecast_values = model.forecast(horizon).values
        elif len(monthly_sales_ts) >= 6:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ExponentialSmoothing(
                    monthly_sales_ts,
                    trend="add",
                    seasonal=None,
                ).fit(optimized=True)
            forecast_values = model.forecast(horizon).values
        else:
            # Very short series: naive fallback
            forecast_values = np.repeat(monthly_sales_ts.iloc[-1], horizon)

    except Exception as e:
        log_step(f"Forecast model fallback used: {e}", "WARNING")
        forecast_values = np.repeat(monthly_sales_ts.iloc[-1], horizon)

    # Future months
    future_months = pd.date_range(
        monthly_sales["Month"].max() + pd.offsets.MonthBegin(1),
        periods=horizon,
        freq="MS",
    )

    forecast_future = pd.DataFrame({
        "Month": future_months,
        "Forecast": forecast_values,
    })

    # Combined table for saving / plotting export
    actual_plot_df = monthly_sales.copy()
    actual_plot_df["Forecast"] = np.nan

    forecast_plot_df = pd.concat(
        [actual_plot_df, forecast_future],
        ignore_index=True,
        sort=False,
    )

    return monthly_sales, forecast_future, forecast_plot_df


# ==========================================
# Utility Helper: Binned Driver Summary
# ==========================================

def build_binned_driver_summary(
    base_df: pd.DataFrame,
    value_col: str,
    band_col: str,
    bins: list,
    labels: list,
) -> pd.DataFrame:
    """
    Build a robust banded summary table for driver analysis.
    """
    require(base_df, [value_col, "Sales", "Profit"])

    work = base_df.loc[:, [value_col, "Sales", "Profit"]].copy()

    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
    work["Sales"] = pd.to_numeric(work["Sales"], errors="coerce")
    work["Profit"] = pd.to_numeric(work["Profit"], errors="coerce")

    work = work.dropna(subset=[value_col, "Sales", "Profit"])

    if work.empty:
        raise ValueError(
            f"{band_col} summary cannot be built: no valid rows remain after cleaning."
        )

    work[band_col] = pd.cut(
        work[value_col],
        bins=bins,
        labels=labels,
        include_lowest=True,
    )

    summary = (
        work.groupby(band_col, observed=False, as_index=False)
        .agg(
            Orders=(value_col, "size"),
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
        )
    )

    summary[band_col] = summary[band_col].astype(str)
    summary = (
        summary.set_index(band_col)
        .reindex(labels, fill_value=0)
        .reset_index()
    )

    summary["Profit_Margin_%"] = np.where(
        summary["Sales"] != 0,
        (summary["Profit"] / summary["Sales"]) * 100,
        np.nan,
    )

    return summary


# ==========================================
# ADVANCED BUSINESS DIAGNOSIS HELPERS
# ==========================================

def dominant_value(series):
    """Return the most common non-null value from a Series."""
    if series is None or series.empty:
        return "N/A"
    mode_vals = series.dropna().mode()
    if not mode_vals.empty:
        return mode_vals.iloc[0]
    cleaned = series.dropna()
    return cleaned.iloc[0] if not cleaned.empty else "N/A"


def build_root_cause_table(work_df, group_col, top_n=10):
    """
    Build a root-cause diagnostic table for the worst-performing groups.
    Returns:
        diagnostic_df, summary_df
    """
    require(work_df, [group_col, "Sales", "Profit", "Product_Name"])

    df_local = work_df.copy()

    if "Quantity" not in df_local.columns:
        df_local["Quantity"] = 0
    if "Discount" not in df_local.columns:
        df_local["Discount"] = np.nan

    summary = aggregate_metrics(
        df_local,
        group_col,
        include_quantity=True,
        sort_by="Profit",
        ascending=True
    ).copy()

    rows = []

    for _, row in summary.head(top_n).iterrows():
        group_value = row[group_col]
        sub = df_local[df_local[group_col] == group_value].copy()

        # Product mix
        product_agg = (
            sub.groupby("Product_Name", as_index=False)
               .agg(
                   Sales=("Sales", "sum"),
                   Profit=("Profit", "sum"),
                   Quantity=("Quantity", "sum")
               )
        )

        top_products = "N/A"
        worst_product = "N/A"
        worst_product_profit = np.nan

        if not product_agg.empty:
            top_products = ", ".join(
                product_agg.sort_values("Sales", ascending=False)
                           .head(3)["Product_Name"]
                           .astype(str)
                           .tolist()
            )

            worst_row = product_agg.sort_values("Profit", ascending=True).iloc[0]
            worst_product = str(worst_row["Product_Name"])
            worst_product_profit = float(worst_row["Profit"])

        # Sub-category mix
        top_subcat = "N/A"
        if "Sub_Category" in sub.columns:
            subcat_agg = (
                sub.groupby("Sub_Category", as_index=False)
                   .agg(
                       Sales=("Sales", "sum"),
                       Profit=("Profit", "sum")
                   )
            )
            if not subcat_agg.empty:
                top_subcat = str(
                    subcat_agg.sort_values("Sales", ascending=False).iloc[0]["Sub_Category"]
                )

        # Category mix
        top_category = "N/A"
        if "Category" in sub.columns:
            cat_agg = (
                sub.groupby("Category", as_index=False)
                   .agg(
                       Sales=("Sales", "sum"),
                       Profit=("Profit", "sum")
                   )
            )
            if not cat_agg.empty:
                top_category = str(
                    cat_agg.sort_values("Sales", ascending=False).iloc[0]["Category"]
                )

        avg_discount = np.nan
        if "Discount" in sub.columns and sub["Discount"].notna().any():
            avg_discount = sub["Discount"].mean() * 100

        reason_bits = []

        if pd.notna(avg_discount):
            if avg_discount >= 20:
                reason_bits.append(f"average discount is high ({avg_discount:.1f}%)")
            elif avg_discount <= 5:
                reason_bits.append(f"discounting is limited ({avg_discount:.1f}%)")

        if pd.notna(row.get("Profit_Margin_%", np.nan)) and row["Profit_Margin_%"] < 0:
            reason_bits.append("overall profit margin is negative")

        if top_subcat in {"Tables", "Bookcases", "Furnishings"}:
            reason_bits.append(f"mix is concentrated in low-margin sub-category {top_subcat}")

        if row["Profit"] < 0 and not reason_bits:
            reason_bits.append("weak unit economics and broad product mix are pulling profit down")

        if not reason_bits:
            reason_bits.append("mixed product mix is limiting profit efficiency")

        rows.append({
            group_col: group_value,
            "Sales": row["Sales"],
            "Profit": row["Profit"],
            "Profit_Margin_%": row["Profit_Margin_%"],
            "Orders": row["Orders"] if "Orders" in row else np.nan,
            "Quantity": row["Quantity"] if "Quantity" in row else np.nan,
            "Avg_Discount_%": avg_discount,
            "Top_Products": top_products,
            "Worst_Product": worst_product,
            "Worst_Product_Profit": worst_product_profit,
            "Top_Category": top_category,
            "Top_Sub_Category": top_subcat,
            "Likely_Reason": "; ".join(reason_bits),
        })

    diagnostic_df = pd.DataFrame(rows)
    return diagnostic_df, summary



def build_rfm_segment_summary(base_df: pd.DataFrame) -> pd.DataFrame:
    require(base_df, ["Customer_ID", "Customer_Name", "Sales", "Profit"])

    rfm_local = build_rfm_table(base_df)

    required_cols = {"Customer_ID", "Customer_Name", "Segment", "Recency", "Frequency", "Monetary"}
    missing_cols = required_cols - set(rfm_local.columns)
    if missing_cols:
        raise ValueError(f"rfm is missing required columns: {missing_cols}")

    rfm_summary_df = rfm_local.copy()
    rfm_summary_df["Segment"] = rfm_summary_df["Segment"].astype(str)
    rfm_summary_df = rfm_summary_df.drop_duplicates(subset=["Customer_ID"]).copy()

    customer_profit = (
        base_df.groupby("Customer_ID", as_index=False)
        .agg(
            Total_Profit=("Profit", "sum"),
            Total_Transaction_Sales=("Sales", "sum"),
        )
    )
    rfm_business = rfm_summary_df.merge(customer_profit, on="Customer_ID", how="left")

    segment_summary = (
        rfm_business.groupby("Segment", as_index=False)
        .agg(
            Customers=("Customer_ID", "nunique"),
            Avg_Recency=("Recency", "mean"),
            Avg_Frequency=("Frequency", "mean"),
            Avg_Monetary=("Monetary", "mean"),
            Total_Sales=("Monetary", "sum"),
            Total_Profit=("Total_Profit", "sum"),
        )
    )

    total_customers = segment_summary["Customers"].sum()
    total_sales = segment_summary["Total_Sales"].sum()
    total_profit = segment_summary["Total_Profit"].sum()

    segment_summary["Customer_Share_%"] = np.where(
        total_customers != 0,
        100 * segment_summary["Customers"] / total_customers,
        np.nan,
    )
    segment_summary["Revenue_Share_%"] = np.where(
        total_sales != 0,
        100 * segment_summary["Total_Sales"] / total_sales,
        np.nan,
    )
    segment_summary["Profit_Share_%"] = np.where(
        total_profit != 0,
        100 * segment_summary["Total_Profit"] / total_profit,
        np.nan,
    )
    segment_summary["Profit_Margin_%"] = np.where(
        segment_summary["Total_Sales"] != 0,
        100 * segment_summary["Total_Profit"] / segment_summary["Total_Sales"],
        np.nan,
    )

    segment_actions = {
        "Champions": "Reward with loyalty perks, exclusive offers, and premium cross-sell opportunities.",
        "Loyal Customers": "Encourage repeat purchases with upsell bundles and relationship-building campaigns.",
        "Potential Loyalists": "Nurture with personalized promotions to increase frequency and basket size.",
        "At Risk": "Trigger retention campaigns, targeted discounts, and reactivation outreach.",
        "Lost Customers": "Use win-back campaigns and low-cost reactivation offers.",
        "Unknown": "Review data quality and investigate customers that do not fit standard segments.",
    }

    segment_summary["Recommended_Action"] = (
        segment_summary["Segment"].map(segment_actions).fillna("Define segment-specific action.")
    )

    segment_order = [
        "Champions",
        "Loyal Customers",
        "Potential Loyalists",
        "At Risk",
        "Lost Customers",
        "Unknown",
    ]

    segment_summary["Segment"] = pd.Categorical(
        segment_summary["Segment"],
        categories=segment_order,
        ordered=True,
    )

    return segment_summary.sort_values("Segment").reset_index(drop=True)


def correlation_strength(r: float) -> str:
    r = abs(r)
    if r >= 0.70:
        return "Strong"
    elif r >= 0.40:
        return "Moderate"
    elif r >= 0.20:
        return "Weak"
    return "Very Weak"

# ==========================================
# KPI CARD DISPLAY
# ==========================================

def show_kpi_cards(cards):
    html = "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:16px;margin:10px 0 24px 0;'>"
    for title, value, subtitle in cards:
        html += f"""
        <div style="
            border:1px solid #e6e6e6;
            border-radius:18px;
            padding:18px 20px;
            background:#ffffff;
            box-shadow:0 4px 18px rgba(0,0,0,0.05);
            font-family:Arial, sans-serif;">
            <div style="font-size:14px;color:#666;font-weight:600;margin-bottom:8px;">{title}</div>
            <div style="font-size:30px;font-weight:800;color:#111;line-height:1.1;">{value}</div>
            <div style="font-size:12px;color:#888;margin-top:8px;">{subtitle}</div>
        </div>
        """
    html += "</div>"
    display(HTML(html))

In [90]:
# LOADING CLEANED DATASET
file_path = "../data/cleaned_superstore.csv"
df = pd.read_csv(file_path)

log_step("Loaded cleaned dataset successfully", "SUCCESS")

[14:35:26] ✅ Loaded cleaned dataset successfully


## 1. Feature Validation

In [91]:
# MASTER DATA PREPROCESSING

df_processed = df.copy()

# Standardize date columns
for col in ["Order_Date", "Ship_Date"]:
    if col in df_processed.columns:
        df_processed[col] = pd.to_datetime(df_processed[col], errors="coerce")

# Core time features
if "Order_Date" in df_processed.columns:
    df_processed["Year"] = df_processed["Order_Date"].dt.year

    month_order = [
        "January", "February", "March", "April",
        "May", "June", "July", "August",
        "September", "October", "November", "December"
    ]

    df_processed["Month"] = pd.Categorical(
        df_processed["Order_Date"].dt.month_name(),
        categories=month_order,
        ordered=True
    )

    df_processed["Quarter"] = df_processed["Order_Date"].dt.quarter

# Shipping delay
if {"Order_Date", "Ship_Date"}.issubset(df_processed.columns):
    df_processed["Shipping_Days"] = (
        df_processed["Ship_Date"] - df_processed["Order_Date"]
    ).dt.days

# Use processed dataframe from this point onward
df = df_processed

# Safe display
display_cols = [c for c in ["Order_Date", "Ship_Date", "Year", "Month", "Quarter"] if c in df.columns]
if display_cols:
    display(df[display_cols].head(10))

log_step("Master preprocessing completed", "SUCCESS")

,Order_Date,Ship_Date,Year,Month,Quarter
0,2012-07-31,2012-07-31,2012,July,3
1,2013-02-05,2013-02-07,2013,February,1
2,2013-10-17,2013-10-18,2013,October,4
3,2013-01-28,2013-01-30,2013,January,1
4,2013-11-05,2013-11-06,2013,November,4
5,2013-06-28,2013-07-01,2013,June,2
6,2011-11-07,2011-11-09,2011,November,4
7,2012-04-14,2012-04-18,2012,April,2
8,2014-10-14,2014-10-21,2014,October,4
9,2012-01-28,2012-01-31,2012,January,1


[14:35:27] ✅ Master preprocessing completed


In [92]:
# BASIC OVERVIEW
display(df.head())
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nInfo:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

# Summary for numeric columns
display(df.describe().T)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,City,State,...,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Year,Month,Quarter,Shipping_Days
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,2309.650,7,0.0,762.1845,933.57,Critical,2012,July,3,0
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,3709.395,9,0.1,-288.7650,923.63,Critical,2013,February,1,2
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,5175.171,9,0.1,919.9710,915.49,Medium,2013,October,4,1
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,2892.510,5,0.1,-96.5400,910.16,Medium,2013,January,1,2
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,2832.960,8,0.0,311.5200,903.04,Critical,2013,November,4,1


Shape: (51290, 28)

Columns:
Index(['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode',
       'Customer_ID', 'Customer_Name', 'Segment', 'City', 'State', 'Country',
       'Postal_Code', 'Market', 'Region', 'Product_ID', 'Category',
       'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount',
       'Profit', 'Shipping_Cost', 'Order_Priority', 'Year', 'Month', 'Quarter',
       'Shipping_Days'],
      dtype='str')

Info:
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row_ID          51290 non-null  int64         
 1   Order_ID        51290 non-null  str           
 2   Order_Date      51290 non-null  datetime64[us]
 3   Ship_Date       51290 non-null  datetime64[us]
 4   Ship_Mode       51290 non-null  str           
 5   Customer_ID     51290 non-null  str           
 6   Customer_Name   51290 non-null  st

,count,mean,min,25%,50%,75%,max,std
Row_ID,51290.0,25645.5,1.0,12823.25,25645.5,38467.75,51290.0,14806.29199
Order_Date,51290,2013-05-11 21:26:49.155780,2011-01-01 00:00:00,2012-06-19 00:00:00,2013-07-08 00:00:00,2014-05-22 00:00:00,2014-12-31 00:00:00,NaN
Ship_Date,51290,2013-05-15 20:42:42.745174,2011-01-03 00:00:00,2012-06-23 00:00:00,2013-07-12 00:00:00,2014-05-26 00:00:00,2015-01-07 00:00:00,NaN
Sales,51290.0,246.490581,0.444,30.758625,85.053,251.0532,22638.48,487.565361
Quantity,51290.0,3.476545,1.0,2.0,3.0,5.0,14.0,2.278766
Discount,51290.0,0.142908,0.0,0.0,0.0,0.2,0.85,0.21228
Profit,51290.0,28.610982,-6599.978,0.0,9.24,36.81,8399.976,174.340972
Shipping_Cost,51290.0,26.375915,0.0,2.61,7.79,24.45,933.57,57.296804
Year,51290.0,2012.777208,2011.0,2012.0,2013.0,2014.0,2014.0,1.098931
Quarter,51290.0,2.793235,1.0,2.0,3.0,4.0,4.0,1.066015


## 2. Core Business KPIs

In [93]:
# 2. CORE BUSINESS KPIs

log_step("Computing core business KPIs", "STEP")

# Core KPI calculations
total_sales = df["Sales"].sum() if "Sales" in df.columns else np.nan
total_profit = df["Profit"].sum() if "Profit" in df.columns else np.nan
total_quantity = df["Quantity"].sum() if "Quantity" in df.columns else np.nan

if "Order_ID" in df.columns:
    total_orders = df["Order_ID"].nunique()
elif "Order ID" in df.columns:
    total_orders = df["Order ID"].nunique()
else:
    total_orders = len(df)

if "Customer_ID" in df.columns:
    total_customers = df["Customer_ID"].nunique()
elif "Customer_Name" in df.columns:
    total_customers = df["Customer_Name"].nunique() # note: may undercount if same name = different markets
elif "Customer Name" in df.columns:
    total_customers = df["Customer Name"].nunique()
else:
    total_customers = np.nan

# Extra useful KPIs
avg_order_value = total_sales / total_orders if pd.notna(total_sales) and total_orders else np.nan
avg_profit_per_order = total_profit / total_orders if pd.notna(total_profit) and total_orders else np.nan
profit_margin = (total_profit / total_sales) * 100 if pd.notna(total_sales) and total_sales != 0 else np.nan
avg_discount = (df["Discount"].mean() * 100) if "Discount" in df.columns else np.nan
avg_sales_per_order = df.groupby("Order_ID")["Sales"].sum().mean() if "Order_ID" in df.columns and "Sales" in df.columns else np.nan


# Display KPIs
kpi_data = {
    "Total Sales": total_sales,
    "Total Profit": total_profit,
    "Total Quantity": total_quantity,
    "Total Orders": total_orders,
    "Total Customers": total_customers,
    "Avg Order Value": avg_order_value,
    "Avg Profit / Order": avg_profit_per_order,
    "Profit Margin (%)": (total_profit / total_sales) * 100 if pd.notna(total_sales) and total_sales != 0 else np.nan,
    "Avg Discount (%)": avg_discount
}

kpi_df = pd.DataFrame(kpi_data, index=["Value"]).T
kpi_df.columns = ["Value"]
display(kpi_df)

#Overview KPI Cards

show_kpi_cards([
    ("Total Sales", f"{fmt_money(total_sales)}", "Revenue generated"),
    ("Total Profit", f"{fmt_money(total_profit)}", "Net earnings"),
    ("Total Quantity", f"{fmt_number(total_quantity)}", "Units sold"),
    ("Total Orders", f"{fmt_number(total_orders)}", "Unique orders"),
    ("Total Customers", f"{fmt_number(total_customers)}", "Unique customers"),
    ("Profit Margin", f"{fmt_percent(profit_margin)}" if pd.notna(profit_margin) else "N/A", "Profit efficiency"),
    ("Average Discount", f"{fmt_percent(avg_discount)}" if pd.notna(avg_discount) else "N/A", "Average discount rate"),
])

[14:35:28] 🔹 Computing core business KPIs


,Value
Total Sales,1.264250e+07
Total Profit,1.467457e+06
Total Quantity,1.783120e+05
Total Orders,2.503500e+04
Total Customers,1.590000e+03
Avg Order Value,5.049931e+02
Avg Profit / Order,5.861623e+01
Profit Margin (%),1.160733e+01
Avg Discount (%),1.429075e+01


### Customer Count Note

The dataset contains **1,590 unique Customer IDs** and **795 unique Customer Names**.
Each Customer ID maps to exactly one name, but many names appear twice — once per
market region (e.g. Aaron Bergman in US and APAC are separate accounts with separate IDs).

All customer analysis in this notebook uses **Customer_ID** as the unique key,
meaning each market account is treated independently. This is the correct approach
for regional sales analysis.

If cross-market deduplication by name is required, the unique individual customer
count is **795**.

## 3. Overall Distributions

In [94]:
# SALES RANGE ANALYSIS

sales_range_df = df.copy()

sales_bins = [0, 50, 200, 500, 1000, 5000, sales_range_df["Sales"].max()]
sales_labels = ["0-50", "50-200", "200-500", "500-1000", "1000-5000", "5000+"]

sales_range_df["Sales Range"] = pd.cut(
    sales_range_df["Sales"],
    bins=sales_bins,
    labels=sales_labels,
    include_lowest=True
)

sales_range_counts = (
    sales_range_df["Sales Range"]
    .value_counts()
    .sort_index()
)

sales_range_plot = sales_range_counts.reset_index()
sales_range_plot.columns = ["Sales Range", "Number of Orders"]

fig = px.bar(
    sales_range_plot,
    x="Sales Range",
    y="Number of Orders",
    text="Number of Orders",
    color_discrete_sequence=[PRIMARY_BLUE],
)

fig = style_plotly(
    fig,
    "How Most Orders Are Distributed by Sales Value",
    x_title="Sales Range",
    y_title="Number of Orders",
    showlegend=False,
    height=DEFAULT_HEIGHT,
)

fig.update_traces(
    texttemplate="%{y:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Orders: %{y:,.0f}<extra></extra>",
)

fig.show()

In [95]:
# PROFIT VS LOSS — CLEAN DONUT CHART WITH COUNT + PERCENTAGE
profit_status = df["Profit"].apply(lambda x: "Profitable" if x >= 0 else "Loss-Making")

counts = profit_status.value_counts().reindex(["Profitable", "Loss-Making"], fill_value=0)
transaction_count = counts.sum()

profit_loss_plot = counts.reset_index()
profit_loss_plot.columns = ["Status", "Count"]

fig = px.pie(
    profit_loss_plot,
    names="Status",
    values="Count",
    hole=0.42,
    color="Status",
    color_discrete_map={
        "Profitable": PRIMARY_GREEN,
        "Loss-Making": PRIMARY_RED,
    },
)

fig.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",
    yref="paper",
    text=f"Total<br>{transaction_count:,}",
    showarrow=False,
    font=dict(size=16, color="#111"),
)

fig = style_plotly(
    fig,
    "Profit vs Loss Distribution",
    showlegend=True,
    height=DEFAULT_HEIGHT,
)

fig.update_traces(
    textinfo="percent+value",
    texttemplate="%{percent:.1%}<br>(%{value:,.0f})",
)

fig.show()

In [96]:
# MOST COMMONLY PURCHASED QUANTITIES
quantity_counts = (
    df["Quantity"]
    .value_counts()
    .sort_index()
    .reset_index()
)

quantity_counts.columns = ["Quantity", "Orders"]

fig = px.bar(
    quantity_counts,
    x="Quantity",
    y="Orders",
    color="Orders",
    color_continuous_scale="viridis"
)

# Simple hover only
fig.update_traces(
    hovertemplate=count_hover_template(
        x_label="Quantity",
        y_label="No of Orders"
    )
)

fig = style_plotly(
    fig,
    title="Most Common Purchase Quantities",
    x_title="Quantity Purchased",
    y_title="Number of Orders",
)

fig.show()

## 4. Time-Based Analysis

In [97]:
#MONTHLY SALES TREND

log_step("Starting time-based analysis", "STEP")

date_col = get_date_col(df)

if date_col and "Sales" in df.columns:
    monthly_sales = (
        df.dropna(subset=[date_col])
          .assign(Month=df[date_col].dt.to_period("M").dt.to_timestamp())
          .groupby("Month", as_index=False)["Sales"].sum()
          .sort_values("Month")
    )

    fig = px.line(
        monthly_sales,
        x="Month",
        y="Sales",
        markers=True,
    )
    fig = style_plotly(
        fig,
        title="Monthly Sales Trend",
        x_title="Month",
        y_title="Sales",
        hovermode="x unified"
    )
    fig.show()

[14:35:28] 🔹 Starting time-based analysis


In [98]:
# SALES BY DAY OF WEEK
date_col = get_date_col(df)

if date_col and "Sales" in df.columns:

    weekday_sales = (
        df.dropna(subset=[date_col])
          .assign(Weekday=df[date_col].dt.day_name())
          .groupby("Weekday", as_index=False)["Sales"].sum()
    )

    # Proper weekday order
    weekday_order = [
        "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday", "Sunday"
    ]

    weekday_sales["Weekday"] = pd.Categorical(
        weekday_sales["Weekday"],
        categories=weekday_order,
        ordered=True
    )

    weekday_sales = weekday_sales.sort_values("Weekday")

    fig = px.bar(
        weekday_sales,
        x="Weekday",
        y="Sales",
        text_auto=".2s",
    )

    fig = style_plotly(
        fig,
        title="Sales by Day of Week",
        x_title="Day of Week",
        y_title="Sales",
        hovermode="x unified"
    )

    fig.show()

In [99]:
# QUARTERLY SALES TREND

date_col = get_date_col(df)

if date_col and "Sales" in df.columns:
    quarterly_sales = (
        df.dropna(subset=[date_col])
          .assign(Quarter=df[date_col].dt.to_period("Q").dt.to_timestamp())
          .groupby("Quarter", as_index=False)["Sales"].sum()
          .sort_values("Quarter")
    )

    fig = px.line(
        quarterly_sales,
        x="Quarter",
        y="Sales",
        markers=True,
    )

    fig = style_plotly(
        fig,
        title="Quarterly Sales Trend",
        x_title="Quarter",
        y_title="Sales",
         
        hovermode="x unified"
    )

    fig.show()

In [100]:
# YEARLY SALES TREND
date_col = get_date_col(df)

if date_col and "Sales" in df.columns:
    yearly_sales = (
        df.dropna(subset=[date_col])
          .assign(Year=df[date_col].dt.to_period("Y").dt.to_timestamp())
          .groupby("Year", as_index=False)["Sales"].sum()
          .sort_values("Year")
    )

    fig = px.line(
        yearly_sales,
        x="Year",
        y="Sales",
        markers=True,
    )

    fig = style_plotly(
        fig,
        title="Yearly Sales Trend",
        x_title="Year",
        y_title="Sales",
        hovermode="x unified"
    )

    fig.show()

In [101]:
# YEAR-OVER-YEAR SALES GROWTH %
date_col = get_date_col(df)

if date_col and "Sales" in df.columns:

    yearly_sales_yoy = (
        df.dropna(subset=[date_col])
          .assign(Year=df[date_col].dt.year.astype(str))
          .groupby("Year", as_index=False)["Sales"]
          .sum()
    )
    
    yearly_sales_yoy = yearly_sales_yoy.sort_values("Year")

    yearly_sales_yoy["YoY_Growth_%"] = (
        yearly_sales_yoy["Sales"].pct_change() * 100
    )

    fig = px.bar(
        yearly_sales_yoy,
        x="Year",
        y="YoY_Growth_%",
        text_auto=".2f",
    )

    fig = style_plotly(
        fig,
        title="Year-over-Year Sales Growth %",
        x_title="Year",
        y_title="Growth %",
        hovermode="x unified"
    )

    fig.show()

In [102]:
# MONTHLY PROFIT TREND
require(df, ["Profit"])

work_df = df.copy()
date_col = get_date_col(work_df)

if not date_col:
    raise ValueError("No usable date column found for Monthly Profit Trend.")

work_df[date_col] = pd.to_datetime(work_df[date_col], errors="coerce")
work_df = work_df.dropna(subset=[date_col]).copy()

monthly_profit = (
    work_df.assign(Month=work_df[date_col].dt.to_period("M").dt.to_timestamp())
           .groupby("Month", as_index=False)["Profit"].sum()
           .sort_values("Month")
)

fig = px.line(
    monthly_profit,
    x="Month",
    y="Profit",
    markers=True,
)

fig = style_plotly(
    fig,
    title="Monthly Profit Trend",
    x_title="Month",
    y_title="Profit",
    hovermode="x unified"
)

fig.show()

In [103]:
# AVERAGE SHIPPING DELAY BY MONTH TREND

require(df, ["Shipping_Days"])

work_df = ensure_shipping_days(df)

order_date_col = next(
    (c for c in ["Order_Date", "Order Date", "OrderDate"] if c in work_df.columns),
    None
)

if order_date_col is None:
    raise ValueError("No usable order date column found for shipping delay trend.")

work_df[order_date_col] = pd.to_datetime(work_df[order_date_col], errors="coerce")
work_df = work_df.dropna(subset=[order_date_col, "Shipping_Days"]).copy()

shipping_trend = (
    work_df.assign(Month=work_df[order_date_col].dt.to_period("M").dt.to_timestamp())
           .groupby("Month", as_index=False)["Shipping_Days"]
           .mean()
           .sort_values("Month")
)

fig = px.line(
    shipping_trend,
    x="Month",
    y="Shipping_Days",
    markers=True,
)

fig = style_plotly(
    fig,
    title="Average Shipping Delay by Month Trend",
    x_title="Month",
    y_title="Average Shipping Days",
    hovermode="x unified"
)

fig.show()

In [104]:
# SALES VS PROFIT TREND

date_col = get_date_col(df)

if date_col and {"Sales", "Profit"}.issubset(df.columns):

    trend = (
        df.dropna(subset=[date_col])
          .assign(Month=df[date_col].dt.to_period("M").dt.to_timestamp())
          .groupby("Month", as_index=False)[["Sales", "Profit"]]
          .sum()
    )

    fig = px.line(
        trend,
        x="Month",
        y=["Sales", "Profit"],
        markers=True,
    )

    fig = style_plotly(
        fig,
        title="Sales vs Profit Trend",
        x_title="Month",
        y_title="Amount",
        hovermode="x unified"
    )

    fig.show()

In [105]:
# MONTHLY PROFIT MARGIN

date_col = get_date_col(df)

if date_col and {"Sales", "Profit"}.issubset(df.columns):

    margin_trend = (
        df.dropna(subset=[date_col])
          .assign(Month=df[date_col].dt.to_period("M").dt.to_timestamp())
          .groupby("Month", as_index=False)[["Sales", "Profit"]]
          .sum()
          .sort_values("Month")
    )

    margin_trend["Profit_Margin"] = (margin_trend["Profit"] / margin_trend["Sales"]) * 100

    fig = px.bar(
        margin_trend,
        x="Month",
        y="Profit_Margin",
        text=margin_trend["Profit_Margin"].round(1),
    )

    fig.update_traces(
        texttemplate="%{text}%",
        textposition="outside",
        marker_color=[
            PRIMARY_GREEN if x >= 10 else PRIMARY_ORANGE if x >= 5 else PRIMARY_RED
            for x in margin_trend["Profit_Margin"]
        ]
    )

    fig.add_hline(
        y=10,
        line_dash="dash",
        line_color = NEUTRAL_GRAY,
        annotation_text="Target Margin = 10%"
    )

    fig = style_plotly(
        fig,
        title="Monthly Profit Margin (%)",
        x_title="Month",
        y_title="Profit Margin (%)",
        hovermode="x unified"
    )

    fig.show()

In [106]:
# SEASONAL SALES HEATMAP

require(df, ["Year", "Month", "Sales"])


heatmap_data = (
        df.groupby(["Year", "Month"], as_index=False)["Sales"]
          .sum()
    )

pivot_table = (
        heatmap_data.pivot(
            index="Month",
            columns="Year",
            values="Sales"
        )
    )

fig = px.imshow(
        pivot_table,
        text_auto=".2s",
        aspect="auto",
        labels=dict(x="Year", y="Month", color="Sales")
    )

fig = style_plotly(
    fig,
    title="Seasonal Sales Heatmap",
    x_title="Year",
    y_title="Month",
    x_type="category",
)

fig.show()

## 5. Category, Sub-Category, and Segment Analysis

In [107]:
# SALES CONTRIBUTION BY CATEGORY

log_step("Building sales contribution by category chart", "STEP")

# ---------------------------------------------------------
# Prepare Data
# ---------------------------------------------------------

category_sales = (
    df.groupby("Category", as_index=False)
    .agg(Total_Sales=("Sales", "sum"))
    .sort_values("Total_Sales", ascending=False)
)

# ---------------------------------------------------------
# Build Donut Chart
# ---------------------------------------------------------

fig = px.pie(
    category_sales,
    names="Category",
    values="Total_Sales",
    hole=0.62,
    color="Category",
    color_discrete_sequence=[
        PRIMARY_BLUE,
        PRIMARY_PURPLE,
        PRIMARY_GREEN,
        PRIMARY_ORANGE,
        PRIMARY_RED,
    ],
)

# ---------------------------------------------------------
# Center Annotation
# ---------------------------------------------------------

fig.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",
    yref="paper",
    text=(
        f"Total Sales<br>"
        f"{fmt_money(category_sales['Total_Sales'].sum())}"
    ),
    showarrow=False,
    font=dict(
        size=18,
        color="#111",
        family=FONT_FAMILY,
    ),
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------

fig = style_plotly(
    fig,
    title="Sales Contribution by Category",
    showlegend=True,
    height=DEFAULT_HEIGHT,
)

fig.update_traces(
    textinfo="percent+label",
    texttemplate="%{label}<br>%{percent:.1%}",
)

fig.show()

[14:35:29] 🔹 Building sales contribution by category chart


In [108]:
# SALES AND PROFIT AND MARGIN BY CATEGORY 

# Aggregate data
category_analysis_summary = (
    aggregate_sales_profit(df, "Category", sort_by="Sales", ascending=True)
    .rename(columns={"Profit_Margin_%": "Margin %"})
    .set_index("Category")
)

category_analysis_summary["Margin %"] = category_analysis_summary["Margin %"].replace([np.inf, -np.inf], np.nan)

categories = category_analysis_summary.index.tolist()
sales = category_analysis_summary["Sales"].values
profit = category_analysis_summary["Profit"].values
margin = category_analysis_summary["Margin %"].values

# Figure
plot_df = category_analysis_summary.reset_index()[["Category", "Sales", "Profit", "Margin %"]]

long_df = plot_df.melt(
    id_vars=["Category", "Margin %"],
    value_vars=["Sales", "Profit"],
    var_name="Metric",
    value_name="Amount",
)

fig = px.bar(
    long_df,
    x="Amount",
    y="Category",
    color="Metric",
    orientation="h",
    barmode="group",
    color_discrete_map={
        "Sales": PRIMARY_BLUE,
        "Profit": PRIMARY_GREEN,
    },
    category_orders={"Category": categories[::-1]},
)

fig.update_traces(
    texttemplate="%{x:,.0f}",
    textposition="outside",
)

for cat, m in zip(categories, margin):
    fig.add_annotation(
        x=sales.max() * 1.15,
        y=cat,
        xref="x",
        yref="y",
        text=f"{m:.1f}% margin",
        showarrow=False,
        bgcolor="#F3F4F6",
        font=dict(color="#111827"),
    )

fig = style_plotly(
    fig,
    "Category Performance Overview",
    x_title="Amount ($)",
    y_title="Category",
    showlegend=True,
    height=DEFAULT_HEIGHT,
    margin=dict(l=120, r=180, t=80, b=40),
)

fig.show()

In [109]:
# SALES BY SUB CATEGORY

log_step("Building sales by sub-category chart", "STEP")

require(df, ["Sub_Category", "Sales"])

# ---------------------------------------------------------
# Prepare Data
# ---------------------------------------------------------

sub_sales = (
    df.groupby("Sub_Category", as_index=False)
    .agg(Total_Sales=("Sales", "sum"))
    .sort_values("Total_Sales", ascending=True)
)

# ---------------------------------------------------------
# Build Chart
# ---------------------------------------------------------

fig = px.bar(
    sub_sales,
    x="Total_Sales",
    y="Sub_Category",
    orientation="h",
    text="Total_Sales",
    color="Total_Sales",
    color_continuous_scale="Blues",
)

# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------

fig = style_plotly(
    fig,
    title="Sales by Sub Category",
    x_title="Total Sales",
    y_title="Sub Category",
)

fig.show(renderer="notebook_connected")

[14:35:30] 🔹 Building sales by sub-category chart


In [110]:
# PROFIT BY SUB_CATEGORY

require(df, ["Sub_Category", "Profit"])

sub_profit = (
        df.groupby("Sub_Category", as_index=False)["Profit"].sum()
          .sort_values("Profit", ascending=True)
    )

fig = px.bar(
        sub_profit,
        x="Profit",
        y="Sub_Category",
        orientation="h",
        text="Profit",
        color="Profit",
        color_continuous_scale="RdYlGn"
    )

fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig = style_plotly(
    fig,
    title="Profit by Sub Category",
    x_title="Profit",
    y_title="Sub Category", 
)
fig.show(renderer="notebook_connected")

In [111]:
# SALES CONTRIBUTION BY CATEGORY AND SUB_CATEGORY

require(df, ["Category", "Sub_Category", "Sales"])

fig = px.treemap(
        df,
        path=["Category", "Sub_Category"],
        values="Sales",
        color="Sales",
        color_continuous_scale="Blues",
    )

fig.update_traces(
    hovertemplate=treemap_hover_template()
)

fig = style_plotly(
    fig,
    title="Overview of Sales Contribution by Category and Sub Category",
)
fig.show()

In [112]:
# SALES VS PROFIT BY SUB CATEGORY

log_step("Building sales vs profit performance chart", "STEP")

require(df, ["Sub_Category", "Sales", "Profit", "Quantity"])

# ---------------------------------------------------------
# Aggregate Data
# ---------------------------------------------------------

sub_perf = (
    df.groupby("Sub_Category", as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum"),
    )
)

# ---------------------------------------------------------
# Bubble Chart
# ---------------------------------------------------------

fig = px.scatter(
    sub_perf,
    x="Total_Sales",
    y="Total_Profit",
    size="Total_Quantity",
    color="Total_Profit",
    hover_name="Sub_Category",
    color_continuous_scale="RdYlGn",
)

# ---------------------------------------------------------
# Reference Line
# ---------------------------------------------------------

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color=PRIMARY_RED,
)

# ---------------------------------------------------------
# Labels
# ---------------------------------------------------------

fig.update_traces(
    textposition="top center"
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------

fig = style_plotly(
    fig,
    title="Sales vs Profit Performance by Sub Category",
    x_title="Total Sales",
    y_title="Total Profit",
    showlegend=False,
)

fig.show()

[14:35:33] 🔹 Building sales vs profit performance chart


In [113]:
# SEGMENT PERFORMANCE ANALYSIS
# Sales | Profit | Profit Margin | Contribution

work_df = df.copy()

# Pick a valid grouping column locally so this cell never depends on prior cells
group_col = next((c for c in ["Segment", "Customer_Segment", "Market_Segment"] if c in work_df.columns), None)

# If no segment-like column exists, fall back to Category so the cell still runs
if group_col is None:
    group_col = "Category"

if group_col in work_df.columns and {"Sales", "Profit"}.issubset(work_df.columns):

    # 1. Aggregate and compute profit margins
    segment_perf = aggregate_sales_profit(
        work_df,
        group_col=group_col,
        sort_by="Sales",
        ascending=True
    ).copy()

    segment_perf = segment_perf.rename(columns={"Profit_Margin_%": "Margin %"})

    # 2. Contribution metrics
    segment_total_sales = segment_perf["Sales"].sum()
    segment_total_profit = segment_perf["Profit"].sum()

    segment_perf["Sales Share %"] = (segment_perf["Sales"] / segment_total_sales) * 100
    segment_perf["Profit Share %"] = np.where(
        segment_total_profit != 0,
        (segment_perf["Profit"] / segment_total_profit) * 100,
        0
    )

    # Keep display order consistent
    segment_perf_display = segment_perf.sort_values("Sales", ascending=False).reset_index(drop=True)

    # 3. Plotly figure
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            f"Sales by {group_col}",
            f"Profit by {group_col}",
            "Profit Margin %"
        ),
        horizontal_spacing=0.10,
        column_widths=[1.2, 1.2, 1.0],
    )

    # Shared category order so the three panels line up visually
    y_order = segment_perf_display[group_col].tolist()[::-1]

    # 1) Sales by Segment
    fig.add_trace(
        go.Bar(
            x=segment_perf_display["Sales"],
            y=segment_perf_display[group_col],
            orientation="h",
            marker=dict(color=PRIMARY_BLUE, line=dict(color="black", width=0.6)),
            text=[f"${v:,.0f}" for v in segment_perf_display["Sales"]],
            textposition="outside",
            cliponaxis=False,
            hovertemplate=f"<b>%{{y}}</b><br>Sales: $%{{x:,.0f}}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # 2) Profit by Segment
    profit_colors = [PRIMARY_GREEN if p >= 0 else PRIMARY_RED for p in segment_perf_display["Profit"]]

    fig.add_trace(
        go.Bar(
            x=segment_perf_display["Profit"],
            y=segment_perf_display[group_col],
            orientation="h",
            marker=dict(color=profit_colors, line=dict(color="black", width=0.6)),
            text=[f"${v:,.0f}" if v >= 0 else f"-${abs(v):,.0f}" for v in segment_perf_display["Profit"]],
            textposition="outside",
            cliponaxis=False,
            hovertemplate=f"<b>%{{y}}</b><br>Profit: $%{{x:,.0f}}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    # 3) Margin % by Segment
    margin_colors = [PRIMARY_BLUE if m >= 0 else PRIMARY_RED for m in segment_perf_display["Margin %"]]

    fig.add_trace(
        go.Bar(
            x=segment_perf_display["Margin %"],
            y=segment_perf_display[group_col],
            orientation="h",
            marker=dict(color=margin_colors, line=dict(color="black", width=0.6)),
            text=[f"{v:.1f}%" for v in segment_perf_display["Margin %"]],
            textposition="outside",
            cliponaxis=False,
            hovertemplate=f"<b>%{{y}}</b><br>Margin: %{{x:.1f}}%<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=3,
    )

    # Apply consistent axes/category ordering
    for r in [1]:
        for c in [1, 2, 3]:
            fig.update_yaxes(
                categoryorder="array",
                categoryarray=y_order,
                row=r,
                col=c,
                title_text="",
                tickfont=dict(size=11),
            )

    # Let the utility handle the notebook-wide styling
    fig = style_plotly(
        fig,
        "Segment Performance Analysis",
        x_title=None,
        y_title=None,
        showlegend=False,
        height=DEFAULT_HEIGHT,
        margin=dict(t=110, l=80, r=70, b=40),
        hovermode="closest",
    )

    fig.update_xaxes(title_text="Sales ($)", row=1, col=1, gridcolor=GRID_COLOR, zeroline=False)
    fig.update_xaxes(title_text="Profit ($)", row=1, col=2, gridcolor=GRID_COLOR, zeroline=False)
    fig.update_xaxes(title_text="Margin %", row=1, col=3, gridcolor=GRID_COLOR, zeroline=False)

    fig.show()

    display(
        segment_perf_display.style.format({
            "Sales": "${:,.2f}",
            "Profit": "${:,.2f}",
            "Margin %": "{:.2f}%",
            "Sales Share %": "{:.2f}%",
            "Profit Share %": "{:.2f}%"
        })
    )
else:
    print(f"Required columns not available for segment analysis. Found grouping column: {group_col}")

,Segment,Sales,Profit,Margin %,Sales Share %,Profit Share %
0,Consumer,"$6,507,949.42","$749,239.78",11.51%,51.48%,51.06%
1,Corporate,"$3,824,697.52","$441,208.33",11.54%,30.25%,30.07%
2,Home Office,"$2,309,854.97","$277,009.18",11.99%,18.27%,18.88%


## 6. Geography Analysis

In [114]:
# REGIONAL PERFORMANCE DASHBOARD
log_step("Building geography and regional performance views", "STEP")

# =========================================================
# Data Preparation
# =========================================================

region_perf = (
    aggregate_sales_profit(df, "Region", sort_by="Sales", ascending=False)
    .rename(columns={"Profit_Margin_%": "Profit Margin %"})
)

region_perf["Profit Margin %"] = (
    region_perf["Profit Margin %"]
    .replace([np.inf, -np.inf], np.nan)
    .round(2)
)

region_perf_reset = region_perf.copy()


# =========================================================
# 1) Sales by Region
# =========================================================

fig = px.bar(
    region_perf_reset,
    x="Region",
    y="Sales",
    text="Sales",
    color_discrete_sequence=[PRIMARY_BLUE],
)

fig = style_plotly(
    fig,
    "Regional Sales Performance",
    x_title="Region",
    y_title="Sales",
    x_tickangle=-90,
    showlegend=False,
    height=DEFAULT_HEIGHT,
)

fig.update_traces(
    texttemplate="$%{y:,.0f}",
    textposition="outside",
)

fig.show()

# =========================================================
# 2) Profit by Region
# =========================================================
profit_colors = [PRIMARY_GREEN if x > 0 else PRIMARY_RED for x in region_perf_reset["Profit"]]

fig = go.Figure(
    go.Bar(
        x=region_perf_reset["Region"],
        y=region_perf_reset["Profit"],
        marker_color=profit_colors,
        text=[f"${v:,.0f}" for v in region_perf_reset["Profit"]],
        textposition="outside",
    )
)

fig.add_hline(y=0, line_color=TEXT_COLOR, line_width=1)

fig = style_plotly(
    fig,
    "Regional Profitability",
    x_title="Region",
    y_title="Profit",
    x_tickangle=-90,
    showlegend=False,
    height=DEFAULT_HEIGHT,
)

fig.show()

# =========================================================
# 3) Profit Margin by Region
# =========================================================

margin_order = (
    region_perf_reset
    .sort_values(by="Profit Margin %", ascending=True)
    .reset_index(drop=True)
)

fig = go.Figure(
    go.Bar(
        y=margin_order["Region"],
        x=margin_order["Profit Margin %"],
        orientation="h",
        marker=dict(
            color=margin_order["Profit Margin %"],
            colorscale=[[0, "#DBEAFE"], [1, PRIMARY_BLUE]],
            cmin=margin_order["Profit Margin %"].min(),
            cmax=margin_order["Profit Margin %"].max(),
            showscale=False,
        ),
        text=margin_order["Profit Margin %"].map(lambda v: f"{v:.1f}%"),
        textposition="outside",
        cliponaxis=False,
    )
)

fig = style_plotly(
    fig,
    "Profit Margin % by Region",
    x_title="Profit Margin %",
    y_title="Region",
    showlegend=False,
    height=DEFAULT_HEIGHT,
    margin=dict(l=120, r=60, t=80, b=40),
)

fig.update_xaxes(range=[0, margin_order["Profit Margin %"].max() * 1.15])
fig.show()

# =========================================================
# 4) Executive Summary Table
# =========================================================

region_exec_summary = region_perf.copy()

region_exec_summary["Sales Rank"] = (
    region_exec_summary["Sales"]
    .rank(ascending=False)
    .astype(int)
)

region_exec_summary["Profit Rank"] = (
    region_exec_summary["Profit"]
    .rank(ascending=False)
    .astype(int)
)

region_exec_summary["Margin Rank"] = (
    region_exec_summary["Profit Margin %"]
    .rank(ascending=False)
    .astype(int)
)

print("\nRegional Performance Summary\n")

try:
    display(
        region_exec_summary.style
        .background_gradient(cmap="Blues")
        .format({
            "Sales": "${:,.0f}",
            "Profit": "${:,.0f}",
            "Profit Margin %": "{:.2f}%"
        })
    )
except:
    display(region_exec_summary)
    
print("""
Note: Sales Rank is based on total sales volume only.
Regions marked below have no statistically significant difference
in their sales distributions — their rank gap does not reflect
a real performance difference:

    East            ↔   Africa
    East            ↔   West
    East            ↔   EMEA
    East            ↔   Canada
    Africa          ↔   EMEA
    West            ↔   Canada
    South           ↔   Caribbean
    Central Asia    ↔   North Asia
    North           ↔   Southeast Asia
""")

[14:35:34] 🔹 Building geography and regional performance views



Regional Performance Summary



,Region,Sales,Profit,Profit Margin %,Sales Rank,Profit Rank,Margin Rank
0,Central,"$2,822,303","$311,404",11.03%,1,1,8
1,South,"$1,600,907","$140,356",8.77%,2,4,11
2,North,"$1,248,166","$194,598",15.59%,3,2,4
3,Oceania,"$1,100,185","$120,089",10.92%,4,6,9
4,Southeast Asia,"$884,423","$17,852",2.02%,5,12,13
5,North Asia,"$848,310","$165,578",19.52%,6,3,2
6,EMEA,"$806,161","$43,898",5.45%,7,10,12
7,Africa,"$783,773","$88,872",11.34%,8,9,7
8,Central Asia,"$752,827","$132,480",17.60%,9,5,3
9,West,"$725,458","$108,418",14.94%,10,7,5



Note: Sales Rank is based on total sales volume only.
Regions marked below have no statistically significant difference
in their sales distributions — their rank gap does not reflect
a real performance difference:

    East            ↔   Africa
    East            ↔   West
    East            ↔   EMEA
    East            ↔   Canada
    Africa          ↔   EMEA
    West            ↔   Canada
    South           ↔   Caribbean
    Central Asia    ↔   North Asia
    North           ↔   Southeast Asia



In [115]:
# STATE PERFORMANCE ANALYSIS — INTERACTIVE PLOTLY VERSION

if "State" in df.columns:

    # =====================================================
    # Data Preparation
    # =====================================================

    state_summary = (
        df.groupby("State", as_index=False)
          .agg(
                Sales=("Sales", "sum"),
                Profit=("Profit", "sum")
          )
    )

    state_summary["Profit Margin %"] = (
        state_summary["Profit"] / state_summary["Sales"] * 100
    ).replace([np.inf, -np.inf], np.nan).round(2)

    # =====================================================
    # 1) Top 10 States by Sales
    # =====================================================

    top_sales = (
        state_summary
        .sort_values(by="Sales", ascending=False)
        .head(10)
        .sort_values(by="Sales")
    )

    fig = px.bar(
        top_sales,
        x="Sales",
        y="State",
        orientation="h",
        color="Sales",
        color_continuous_scale="Blues",
        text="Sales",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Top 10 States by Sales"
    )

    fig.update_traces(
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Top 10 States by Sales",
        x_title="Sales",
        y_title="State",
        showlegend=False
    )

    fig.show()

    # =====================================================
    # 2) Top 10 States by Profit
    # =====================================================

    top_profit = (
        state_summary
        .sort_values(by="Profit", ascending=False)
        .head(10)
        .sort_values(by="Profit")
    )

    fig = px.bar(
        top_profit,
        x="Profit",
        y="State",
        orientation="h",
        color="Profit",
        color_continuous_scale="RdYlGn",
        text="Profit",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Top 10 States by Profit"
    )

    fig.update_traces(
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Top 10 States by Profit",
        x_title="Profit",
        y_title="State",
        showlegend=False
    )

    fig.show()

    # =====================================================
    # 3) Top 10 States by Profit Margin
    # =====================================================

    top_margin = (
        state_summary
        .sort_values(by="Profit Margin %", ascending=False)
        .head(10)
        .sort_values(by="Profit Margin %")
    )

    max_margin = top_margin["Profit Margin %"].max()

    fig = px.bar(
        top_margin,
        x="Profit Margin %",
        y="State",
        orientation="h",
        color="Profit Margin %",
        color_continuous_scale="Viridis",
        text="Profit Margin %",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Top 10 States by Profit Margin %"
    )

    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Top 10 States by Profit Margin %",
        x_title="Profit Margin %",
        y_title="State",
        showlegend=False,
        x_range=[0, max_margin * 1.15]
    )

    fig.show()

    # =====================================================
    # Summary Table
    # =====================================================
# =====================================================
# Summary Tables
# =====================================================

state_summary_top_sales = (
    state_summary
    .sort_values(by="Sales", ascending=False)
    .head(10)
)

state_summary_top_profit = (
    state_summary
    .sort_values(by="Profit", ascending=False)
    .head(10)
)

state_summary_top_margin = (
    state_summary
    .sort_values(by="Profit Margin %", ascending=False)
    .head(10)
)

print("\nTop 10 States by Sales\n")
try:
    display(
        state_summary_top_sales.style
        .background_gradient(cmap="Blues")
        .format({
            "Sales": "${:,.0f}",
            "Profit": "${:,.0f}",
            "Profit Margin %": "{:.2f}%"
        })
    )
except:
    display(state_summary_top_sales)

print("\nTop 10 States by Profit\n")
try:
    display(
        state_summary_top_profit.style
        .background_gradient(cmap="RdYlGn")
        .format({
            "Sales": "${:,.0f}",
            "Profit": "${:,.0f}",
            "Profit Margin %": "{:.2f}%"
        })
    )
except:
    display(state_summary_top_profit)

print("\nTop 10 States by Profit Margin %\n")
try:
    display(
        state_summary_top_margin.style
        .background_gradient(cmap="viridis")
        .format({
            "Sales": "${:,.0f}",
            "Profit": "${:,.0f}",
            "Profit Margin %": "{:.2f}%"
        })
    )
except:
    display(state_summary_top_margin)


Top 10 States by Sales



,State,Sales,Profit,Profit Margin %
311,England,"$485,171","$99,908",20.59%
192,California,"$457,688","$76,381",16.69%
435,Ile-de-France,"$317,823","$44,056",13.86%
703,New York,"$310,876","$74,039",23.82%
702,New South Wales,"$270,487","$43,696",16.15%
820,Queensland,"$238,313","$21,609",9.07%
722,North Rhine-Westphalia,"$216,452","$42,348",19.56%
983,Texas,"$170,188","$-25,729",-15.12%
867,San Salvador,"$153,639","$35,883",23.36%
689,National Capital,"$152,175","$-13,066",-8.59%



Top 10 States by Profit



,State,Sales,Profit,Profit Margin %
311,England,"$485,171","$99,908",20.59%
192,California,"$457,688","$76,381",16.69%
703,New York,"$310,876","$74,039",23.82%
435,Ile-de-France,"$317,823","$44,056",13.86%
702,New South Wales,"$270,487","$43,696",16.15%
722,North Rhine-Westphalia,"$216,452","$42,348",19.56%
867,San Salvador,"$153,639","$35,883",23.36%
1050,Washington,"$138,641","$33,403",24.09%
646,Michigan,"$76,270","$24,463",32.07%
955,São Paulo,"$98,417","$21,878",22.23%



Top 10 States by Profit Margin %



,State,Sales,Profit,Profit Margin %
997,Tottori,$27,$14,50.00%
967,Tanga,$518,$243,47.00%
94,Atsimo-Andrefana,$78,$36,46.90%
185,Buskerud,$911,$424,46.51%
42,Amur,"$3,636","$1,678",46.14%
781,Pernik,$15,$7,44.94%
1040,Vladimir,"$2,790","$1,231",44.14%
823,Quindío,"$2,292","$1,011",44.10%
575,Lori,$157,$69,44.08%
851,Saga,"$1,137",$499,43.91%


In [116]:
# CITY PERFORMANCE ANALYSIS — PLOTLY VERSION

if "City" in df.columns:


    # =====================================================
    # Data Preparation
    # =====================================================

    city_summary = (
        df.groupby("City", as_index=False)
          .agg(
              Sales=("Sales", "sum"),
              Profit=("Profit", "sum")
          )
    )

    city_summary["Profit Margin %"] = (
        city_summary["Profit"] / city_summary["Sales"] * 100
    ).round(2)

    # =====================================================
    # 1) Top 10 Cities by Sales
    # =====================================================

    top_sales = (
        city_summary
        .sort_values(by="Sales", ascending=False)
        .head(10)
        .sort_values(by="Sales")
    )

    fig = px.bar(
        top_sales,
        x="Sales",
        y="City",
        orientation="h",
        color="Sales",
        color_continuous_scale="Blues",
        text="Sales",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Top 10 Cities by Sales"
    )

    fig.update_traces(
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Top 10 Cities by Sales",
        x_title="Sales",
        y_title="City",
        showlegend=False
    )

    fig.show()

    # =====================================================
    # 2) Top 10 Cities by Profit
    # =====================================================

    top_profit = (
        city_summary
        .sort_values(by="Profit", ascending=False)
        .head(10)
        .sort_values(by="Profit")
    )

    fig = px.bar(
        top_profit,
        x="Profit",
        y="City",
        orientation="h",
        color="Profit",
        color_continuous_scale="RdYlGn",
        text="Profit",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Top 10 Cities by Profit"
    )

    fig.update_traces(
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Top 10 Cities by Profit",
        x_title="Profit",
        y_title="City",
        showlegend=False
    )

    fig.show()

    # =====================================================
    # 3) Top 10 Cities by Profit Margin %
    # =====================================================

    # Use top sales cities only
    top_margin = (
        city_summary
        .sort_values(by="Sales", ascending=False)
        .head(10)
        .sort_values(by="Profit Margin %")
    )

    min_margin = top_margin["Profit Margin %"].min()
    max_margin = top_margin["Profit Margin %"].max()

    fig = px.bar(
        top_margin,
        x="Profit Margin %",
        y="City",
        orientation="h",
        color="Profit Margin %",
        color_continuous_scale="Viridis",
        text="Profit Margin %",
        custom_data=["Sales", "Profit", "Profit Margin %"],
        title="Profit Margin % by Top Sales Cities"
    )

    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=build_hover_template(
            title_template="<b>%{y}</b>",
            lines=[
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    )

    fig = style_plotly(
        fig,
        title="Profit Margin % by Top Sales Cities",
        x_title="Profit Margin %",
        y_title="City",
        showlegend=False,
        x_range=[min_margin * 1.15, max_margin * 1.15]
    )

    fig.show()

    # =====================================================
    # Executive Summary Table
    # =====================================================

    summary_sales = (
        city_summary
        .sort_values(by="Sales", ascending=False)
        .head(10)
    )

    summary_profit = (
        city_summary
        .sort_values(by="Profit", ascending=False)
        .head(10)
    )

    summary_margin = (
        city_summary
        .sort_values(by="Profit Margin %", ascending=False)
        .head(10)
    )

    print("\nTop 10 Cities by Sales\n")
    try:
        display(
            summary_sales.style
            .background_gradient(cmap="Blues")
            .format({
                "Sales": "${:,.0f}",
                "Profit": "${:,.0f}",
                "Profit Margin %": "{:.2f}%"
            })
        )
    except:
        display(summary_sales)

    print("\nTop 10 Cities by Profit\n")
    try:
        display(
            summary_profit.style
            .background_gradient(cmap="RdYlGn")
            .format({
                "Sales": "${:,.0f}",
                "Profit": "${:,.0f}",
                "Profit Margin %": "{:.2f}%"
            })
        )
    except:
        display(summary_profit)


Top 10 Cities by Sales



,City,Sales,Profit,Profit Margin %
2290,New York City,"$256,368","$62,037",24.20%
1910,Los Angeles,"$175,851","$30,441",17.31%
1996,Manila,"$120,887","$-11,159",-9.23%
2936,Seattle,"$119,541","$29,156",24.39%
2843,San Francisco,"$112,669","$17,507",15.54%
2499,Philadelphia,"$109,077","$-13,838",-12.69%
3107,Sydney,"$101,946","$16,003",15.70%
1508,Jakarta,"$94,321","$3,827",4.06%
1895,London,"$86,946","$17,379",19.99%
2106,Mexico City,"$85,729","$13,342",15.56%



Top 10 Cities by Profit



,City,Sales,Profit,Profit Margin %
2290,New York City,"$256,368","$62,037",24.20%
1910,Los Angeles,"$175,851","$30,441",17.31%
2936,Seattle,"$119,541","$29,156",24.39%
1989,Managua,"$83,707","$17,854",21.33%
2843,San Francisco,"$112,669","$17,507",15.54%
1895,London,"$86,946","$17,379",19.99%
3107,Sydney,"$101,946","$16,003",15.70%
3391,Vienna,"$62,024","$15,661",25.25%
2870,San Salvador,"$57,699","$15,037",26.06%
2106,Mexico City,"$85,729","$13,342",15.56%


In [117]:
# REGIONAL PROFITABILITY MATRIX
log_step("Building regional profitability matrix", "STEP")

require(df, ["Region", "Sales", "Profit"])

region_matrix = (
    df.groupby("Region", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Orders=("Region", "size")
        )
        .sort_values("Sales", ascending=True)
        .reset_index(drop=True)
)

# Midpoints for quadrant split
rpm_sales_mid = region_matrix["Sales"].median()
profit_mid = region_matrix["Profit"].median()

# Bubble sizes
max_orders = region_matrix["Orders"].max()
region_matrix["BubbleSize"] = 20 + (region_matrix["Orders"] / max_orders) * 60 if max_orders else 20

# Quadrant assignment using utility function
region_matrix["Quadrant"] = region_matrix.apply(
    lambda row: quadrant_label(
        row["Sales"],
        row["Profit"],
        rpm_sales_mid,
        profit_mid
    ),
    axis=1
)

# Match the labels returned by the utility function
quadrant_colors = {
    "High Sales | High Profit": PRIMARY_GREEN,
    "High Sales | Low Profit": PRIMARY_BLUE,
    "Low Sales | High Profit": PRIMARY_ORANGE,
    "Low Sales | Low Profit": PRIMARY_RED,
    "Unknown": NEUTRAL_GRAY,
}

region_matrix["Color"] = region_matrix["Quadrant"].map(quadrant_colors).fillna(NEUTRAL_GRAY)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=region_matrix["Sales"],
        y=region_matrix["Profit"],
        mode="markers",
        marker=dict(
            size=region_matrix["BubbleSize"],
            color=region_matrix["Color"],
            line=dict(color="black", width=1.2),
            opacity=0.85
        ),
        customdata=np.stack(
            [
                region_matrix["Region"],
                region_matrix["Orders"],
                region_matrix["Quadrant"]
            ],
            axis=-1
        ),
        hovertemplate=build_hover_template(
            title_template="<b>Region:</b> %{customdata[0]}",
            lines=[
                {"label": "Sales", "expr": "x", "format_spec": ",.2f"},
                {"label": "Profit", "expr": "y", "format_spec": ",.2f"},
                {"label": "Orders", "expr": "customdata[1]", "format_spec": ",.0f"},
                {"label": "Quadrant", "expr": "customdata[2]"},
            ],
        )
    )
)

# Reference lines
fig.add_shape(
    type="line",
    x0=rpm_sales_mid, x1=rpm_sales_mid,
    y0=region_matrix["Profit"].min(),
    y1=region_matrix["Profit"].max(),
    line=dict(color="gray", width=1.2, dash="dash")
)
fig.add_shape(
    type="line",
    x0=region_matrix["Sales"].min(),
    x1=region_matrix["Sales"].max(),
    y0=profit_mid, y1=profit_mid,
    line=dict(color="gray", width=1.2, dash="dash")
)

fig = style_plotly(
    fig,
    title="Regional Profitability Matrix",
    x_title="Sales",
    y_title="Profit",
    showlegend=False
)

fig.show()
print("\nRegional Profitability Summary\n")

display(
    region_matrix[["Region", "Sales", "Profit", "Orders", "Quadrant"]].round(2)
)

[14:35:35] 🔹 Building regional profitability matrix



Regional Profitability Summary



,Region,Sales,Profit,Orders,Quadrant
0,Canada,66928.17,17817.39,384,Low Sales | Low Profit
1,Caribbean,324280.86,34571.32,1690,Low Sales | Low Profit
2,East,678781.24,91522.78,2848,Low Sales | Low Profit
3,West,725457.82,108418.45,3203,Low Sales | High Profit
4,Central Asia,752826.57,132480.19,2048,Low Sales | High Profit
5,Africa,783773.21,88871.63,4587,Low Sales | Low Profit
6,EMEA,806161.31,43897.97,5029,High Sales | Low Profit
7,North Asia,848309.78,165578.42,2338,High Sales | High Profit
8,Southeast Asia,884423.17,17852.33,3129,High Sales | Low Profit
9,Oceania,1100184.61,120089.11,3487,High Sales | High Profit


In [118]:
# SALES VS PROFIT SCATTER PLOT BY STATE
if "State" in df.columns and {"Sales", "Profit"}.issubset(df.columns):

    state_perf = (
        df.groupby("State", as_index=False)
          .agg(
              Sales=("Sales", "sum"),
              Profit=("Profit", "sum"),
              Orders=("State", "size")
          )
    )

    # Profit Margin
    state_perf["Profit Margin %"] = (
        state_perf["Profit"] / state_perf["Sales"] * 100
    ).replace([np.inf, -np.inf], np.nan)

    # Bubble Sizes
    state_perf["BubbleSize"] = (
        20 + (state_perf["Orders"] / state_perf["Orders"].max()) * 60
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=state_perf["Sales"],
            y=state_perf["Profit"],
            mode="markers",
            marker=dict(
                size=state_perf["BubbleSize"],
                color=state_perf["Profit Margin %"],
                colorscale="Viridis",
                opacity=0.85,
                line=dict(color="black", width=1),
                colorbar=dict(title="Profit Margin %")
            ),
            customdata=np.stack(
                [
                    state_perf["State"],
                    state_perf["Orders"],
                    state_perf["Profit Margin %"]
                ],
                axis=-1
            ),
            hovertemplate=build_hover_template(
                title_template="<b>State:</b> %{customdata[0]}",
                lines=[
                    {"label": "Sales", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
                    {"label": "Profit", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
                    {"label": "Orders", "expr": "customdata[1]", "format_spec": ",.0f"},
                    {"label": "Profit Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
                ],
            )
        )
    )

    # Horizontal line at Profit = 0
    fig.add_shape(
        type="line",
        x0=state_perf["Sales"].min(),
        x1=state_perf["Sales"].max(),
        y0=0,
        y1=0,
        line=dict(color="black", width=1)
    )

    # Vertical average sales line
    avg_sales = state_perf["Sales"].mean()

    fig.add_shape(
        type="line",
        x0=avg_sales,
        x1=avg_sales,
        y0=state_perf["Profit"].min(),
        y1=state_perf["Profit"].max(),
        line=dict(color="gray", width=1, dash="dash")
    )

    fig = style_plotly(
        fig,
        title="Sales vs Profit by State",
        x_title="Sales",
        y_title="Profit",
        showlegend=False
    )

    fig.show()

    print("\nTop 10 States by Sales\n")

    display(
        state_perf.sort_values("Sales", ascending=False)
        .head(10)
        .round(2)
    )


Top 10 States by Sales



,State,Sales,Profit,Orders,Profit Margin %,BubbleSize
311,England,485170.97,99907.73,1499,20.59,64.95
192,California,457687.63,76381.39,2001,16.69,80.00
435,Ile-de-France,317822.54,44055.92,981,13.86,49.42
703,New York,310876.27,74038.55,1128,23.82,53.82
702,New South Wales,270487.10,43695.98,781,16.15,43.42
820,Queensland,238312.73,21608.75,717,9.07,41.50
722,North Rhine-Westphalia,216451.85,42347.87,719,19.56,41.56
983,Texas,170188.05,-25729.36,985,-15.12,49.54
867,San Salvador,153639.40,35883.38,615,23.36,38.44
689,National Capital,152175.36,-13066.08,583,-8.59,37.48


In [119]:
# GLOBAL GEOGRAPHIC HEATMAP / CHOROPLETH MAP
# Sales and Profit by Country


# Check required columns
if "Country" in df.columns and {"Sales", "Profit"}.issubset(df.columns):

    # =====================================================
    # Aggregate Country Metrics
    # =====================================================

    country_sales = (
        df.groupby("Country", as_index=False)["Sales"]
          .sum()
    )

    country_profit = (
        df.groupby("Country", as_index=False)["Profit"]
          .sum()
    )

    # =====================================================
    # 1) Global Sales Heatmap
    # =====================================================

    fig1 = px.choropleth(
        country_sales,
        locations="Country",
        locationmode="country names",
        color="Sales",
        hover_name="Country",
        color_continuous_scale="Blues",
        title="Global Sales Distribution"
    )

    fig1 = style_plotly(
        fig1,
        title="Global Sales Distribution",
        geo_kwargs=dict(
            showframe=False,
            showcoastlines=True,
            projection_type="natural earth",
        ),
    )

    fig1.show()

    # =====================================================
    # 2) Global Profit Heatmap
    # =====================================================

    fig2 = px.choropleth(
        country_profit,
        locations="Country",
        locationmode="country names",
        color="Profit",
        hover_name="Country",
        color_continuous_scale="RdYlGn",
        title="Global Profit Distribution"
    )

    fig2 = style_plotly(
        fig2,
        title="Global Profit Distribution",
        geo_kwargs=dict(
            showframe=False,
            showcoastlines=True,
            projection_type="natural earth",
        ),
    )

    fig2.show()

    # =====================================================
    # Summary Table
    # =====================================================

    country_summary = (
        df.groupby("Country")[["Sales", "Profit"]]
          .sum()
          .sort_values(by="Sales", ascending=False)
          .round(2)
    )

    print("\nTop Countries by Sales and Profit\n")

    display(country_summary.head(15))


Top Countries by Sales and Profit



,Sales,Profit
Country,,
United States,2297200.86,286397.02
Australia,925235.85,103907.43
France,858931.08,109029.00
China,700562.02,150683.08
Germany,628840.03,107322.82
Mexico,622590.62,102818.10
India,589650.10,129071.84
United Kingdom,528576.30,111900.15
Indonesia,404887.50,15608.68


In [120]:
# LOSS-MAKING STATES AND CITIES ANALYSIS

# -----------------------------
# Loss-making States
# -----------------------------
if "State" in df.columns and "Profit" in df.columns:

    state_loss = (
        df.groupby("State", as_index=False)["Profit"]
          .sum()
          .sort_values("Profit", ascending=True)
          .head(10)
    )

    state_loss["AbsProfit"] = state_loss["Profit"].abs()
    max_state = state_loss["AbsProfit"].max() if len(state_loss) else 1

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=state_loss["Profit"],
            y=state_loss["State"],
            orientation="h",
            marker=dict(
                color=state_loss["Profit"].apply(get_profit_color),
                line=dict(color="black", width=1.2)
            ),
            customdata=np.stack(
                [state_loss["Profit"].abs()],
                axis=-1
            ),
            hovertemplate=build_hover_template(
                title_template="<b>State:</b> %{y}",
                lines=[
                    {"label": "Profit", "expr": "x", "format_spec": ",.0f", "prefix": "$"},
                ],
            )
        )
    )

    fig.add_shape(
        type="line",
        x0=0, x1=0,
        y0=-0.5, y1=len(state_loss) - 0.5,
        line=dict(color="black", width=1)
    )

    fig = style_plotly(
        fig,
        title="Bottom 10 States by Profit",
        x_title="Profit",
        y_title="State",
        showlegend=False
    )

    fig.show()

# -----------------------------
# Loss-making Cities
# -----------------------------
if "City" in df.columns and "Profit" in df.columns:

    city_loss = (
        df.groupby("City", as_index=False)["Profit"]
          .sum()
          .sort_values("Profit", ascending=True)
          .head(10)
    )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=city_loss["Profit"],
            y=city_loss["City"],
            orientation="h",
            marker=dict(
                color=city_loss["Profit"].apply(get_profit_color),
                line=dict(color="black", width=1.2)
            ),
            hovertemplate=build_hover_template(
                title_template="<b>City:</b> %{y}",
                lines=[
                    {"label": "Profit", "expr": "x", "format_spec": ",.0f", "prefix": "$"},
                ],
            )
        )
    )

    fig.add_shape(
        type="line",
        x0=0, x1=0,
        y0=-0.5, y1=len(city_loss) - 0.5,
        line=dict(color="black", width=1)
    )

    fig = style_plotly(
        fig,
        title="Bottom 10 Cities by Profit",
        x_title="Profit",
        y_title="City",
        showlegend=False
    )

    fig.show()

In [121]:
# SHIPPING DELAY / DELIVERY EFFICIENCY BY REGION
# Average Shipping_Days vs Profit 

# Find date columns flexibly
require(df, ["Region", "Sales", "Profit"])
work_df = ensure_shipping_days(df)

region_ship = (
    work_df.groupby("Region", as_index=False)
    .agg(
        Avg_Shipping_Days=("Shipping_Days", "mean"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .dropna()
)

# -----------------------------
# 1) Average Shipping_Days by Region
# -----------------------------
ship_order = region_ship.sort_values("Avg_Shipping_Days", ascending=True)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=ship_order["Avg_Shipping_Days"],
        y=ship_order["Region"],
        orientation="h",
        marker=dict(
            color=ship_order["Avg_Shipping_Days"],
            colorscale="Viridis",
            line=dict(color="black", width=1.2),
            colorbar=dict(title="Avg Shipping_Days")
        ),
        customdata=np.stack(
            [
                ship_order["Sales"],
                ship_order["Profit"]
            ],
            axis=-1
        ),
        hovertemplate=build_hover_template(
            title_template="<b>Region:</b> %{y}",
            lines=[
                {"label": "Average Shipping_Days", "expr": "x", "format_spec": ".1f"},
                {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
            ],
        )
    )
)

fig = style_plotly(
    fig,
    title="Average Shipping Days by Region",
    x_title="Average Shipping Days",
    y_title="Region",
    showlegend=False
)

fig.show()

# -----------------------------
# 2) Shipping Days vs Profit
# -----------------------------
max_sales = region_ship["Sales"].max()
region_ship["BubbleSize"] = 20 + (region_ship["Sales"] / max_sales) * 60

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=region_ship["Avg_Shipping_Days"],
        y=region_ship["Profit"],
        mode="markers",
        marker=dict(
            size=region_ship["BubbleSize"],
            color=region_ship["Profit"],
            colorscale="RdBu_r",
            opacity=0.85,
            line=dict(color="black", width=1),
            colorbar=dict(title="Profit")
        ),
        customdata=np.stack(
            [
                region_ship["Region"],
                region_ship["Sales"]
            ],
            axis=-1
        ),
        hovertemplate=build_hover_template(
            title_template="<b>Region:</b> %{customdata[0]}",
            lines=[
                {"label": "Average Shipping Days", "expr": "x", "format_spec": ".1f"},
                {"label": "Profit", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Sales", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
            ],
        )
    )
)

fig.add_shape(
    type="line",
    x0=region_ship["Avg_Shipping_Days"].min(),
    x1=region_ship["Avg_Shipping_Days"].max(),
    y0=0,
    y1=0,
    line=dict(color="black", width=1)
)

fig = style_plotly(
    fig,
    title="Shipping Days vs Profit by Region",
    x_title="Average Shipping Days",
    y_title="Profit",
    showlegend=False
)

fig.show()

print("\nRegion Shipping Summary\n")
display(
    region_ship.sort_values("Avg_Shipping_Days", ascending=True).round(2)
)


Region Shipping Summary



,Region,Avg_Shipping_Days,Sales,Profit,BubbleSize
1,Canada,3.68,66928.17,17817.39,21.42
8,North Asia,3.91,848309.78,165578.42,38.03
6,East,3.91,678781.24,91522.78,34.43
0,Africa,3.91,783773.21,88871.63,36.66
12,West,3.93,725457.82,108418.45,35.42
9,Oceania,3.93,1100184.61,120089.11,43.39
5,EMEA,3.93,806161.31,43897.97,37.14
10,South,3.94,1600907.04,140355.77,54.03
2,Caribbean,3.97,324280.86,34571.32,26.89
4,Central Asia,4.01,752826.57,132480.19,36.00


## 7. Shipping Analysis

In [122]:
# SHIPPING DELAY RISK DIAGNOSTICS

work_df = ensure_shipping_days(df)
work_df = work_df.dropna(subset=["Shipping_Days"]).copy()

avg_shipping_days = work_df["Shipping_Days"].mean()
median_shipping_days = work_df["Shipping_Days"].median()

slow_shipments_pct = (
    (work_df["Shipping_Days"] > median_shipping_days)
    .mean()
) * 100

print(f"Average shipping duration: {avg_shipping_days:.2f} days")
print(f"Median shipping duration: {median_shipping_days:.2f} days")

print(
    f"Share of shipments exceeding median delivery duration: "
    f"{slow_shipments_pct:.2f}%"
)

Average shipping duration: 3.97 days
Median shipping duration: 4.00 days
Share of shipments exceeding median delivery duration: 40.03%


In [123]:
# SHIP MODE ANALYSIS
log_step("Analyzing shipping performance", "STEP")

require(df, ["Ship_Mode", "Sales", "Profit", "Order_ID"])

work_df = df.copy()

# 1. Aggregate metrics
ship_summary = aggregate_metrics(
    work_df,
    group_col="Ship_Mode",
    include_quantity=False,
    order_col="Order_ID"
)

# 2. Add custom business indicator
ship_summary["Avg_Order_Value"] = np.where(
    ship_summary["Orders"] != 0,
    ship_summary["Sales"] / ship_summary["Orders"],
    np.nan
).round(2)

# 3. Sort explicitly
ship_summary = ship_summary.sort_values(by="Sales", ascending=False).reset_index(drop=True)

# =====================================================
# 1. SALES BY SHIP MODE
# =====================================================
fig1 = px.bar(
    ship_summary,
    x="Ship_Mode",
    y="Sales",
    text=ship_summary["Sales"].map(lambda x: f"${x:,.0f}"),
    title="Sales Distribution by Ship Mode"
)

fig1.update_traces(
    marker_line_color="black",
    marker_line_width=1.2,
    hovertemplate=build_hover_template(
        title_template="<b>%{x}</b>",
        lines=[
            {"label": "Sales", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[1]", "format_spec": ".2f", "suffix": "%"},
            {"label": "Orders", "expr": "customdata[2]", "format_spec": ",.0f"},
            {"label": "Avg Order Value", "expr": "customdata[3]", "format_spec": ",.2f", "prefix": "$"},
        ],
    ),
    customdata=np.stack([
        ship_summary["Profit"],
        ship_summary["Profit_Margin_%"],
        ship_summary["Orders"],
        ship_summary["Avg_Order_Value"]
    ], axis=-1)
)
fig1 = style_plotly(
    fig1,
    title="Sales Distribution by Ship Mode",
    x_title="",
    y_title="Total Sales",
)
fig1.show()

# =====================================================
# 2. PROFIT ANALYSIS
# =====================================================
fig2 = go.Figure(
    data=[
        go.Bar(
            x=ship_summary["Ship_Mode"],
            y=ship_summary["Profit"],
            marker=dict(
                color=ship_summary["Profit"].apply(get_profit_color),
                line=dict(color="black", width=1.2)
            ),
            text=ship_summary["Profit"].map(lambda x: f"${x:,.0f}"),
            customdata=np.stack([
                ship_summary["Sales"],
                ship_summary["Profit_Margin_%"],
                ship_summary["Orders"],
                ship_summary["Avg_Order_Value"]
            ], axis=-1),
        )
    ]
)
fig2.update_traces(
    marker_line_color=TEXT_COLOR,
    marker_line_width=1.2,
    hovertemplate=build_hover_template(
        title_template="<b>%{x}</b>",
        lines=[
            {"label": "Profit", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[1]", "format_spec": ".2f", "suffix": "%"},
            {"label": "Orders", "expr": "customdata[2]", "format_spec": ",.0f"},
            {"label": "Avg Order Value", "expr": "customdata[3]", "format_spec": ",.2f", "prefix": "$"},
        ],
    ),
    customdata=np.stack([
        ship_summary["Sales"],
        ship_summary["Profit_Margin_%"],
        ship_summary["Orders"],
        ship_summary["Avg_Order_Value"]
    ], axis=-1)
)
fig2.add_hline(y=0, line_width=1.2, line_color="black")
fig2 = style_plotly(
    fig2,
    title="Profit by Ship Mode",
    x_title="",
    y_title="Total Profit",
)
fig2.show()

# =====================================================
# 3. PROFIT MARGIN ANALYSIS
# =====================================================
avg_margin = ship_summary["Profit_Margin_%"].mean()

fig3 = px.bar(
    ship_summary,
    x="Ship_Mode",
    y="Profit_Margin_%",
    color="Profit_Margin_%",
    color_continuous_scale="Turbo",
    text=ship_summary["Profit_Margin_%"].map(lambda x: f"{x:.1f}%"),
    title="Profit Margin by Ship Mode"
)
fig3.update_traces(
    marker_line_color=TEXT_COLOR,
    marker_line_width=1.2,
    hovertemplate=build_hover_template(
        title_template="<b>%{x}</b>",
        lines=[
            {"label": "Profit Margin", "expr": "y", "format_spec": ".2f", "suffix": "%"},
            {"label": "Sales", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Orders", "expr": "customdata[2]", "format_spec": ",.0f"},
            {"label": "Avg Order Value", "expr": "customdata[3]", "format_spec": ",.2f", "prefix": "$"},
        ],
    ),
    customdata=np.stack([
        ship_summary["Sales"],
        ship_summary["Profit"],
        ship_summary["Orders"],
        ship_summary["Avg_Order_Value"]
    ], axis=-1)
)
fig3.add_hline(
    y=avg_margin,
    line_dash="dash",
    line_width=2,
    line_color = NEUTRAL_GRAY,
    annotation_text=f"Average Margin ({avg_margin:.2f}%)",
    annotation_position="top left"
)
fig3 = style_plotly(
    fig3,
    title="Profit Margin by Ship Mode",
    x_title="",
    y_title="Profit Margin (%)",
    coloraxis_showscale=False
)

fig3.show()

# =====================================================
# 4. SALES VS PROFIT BUBBLE CHART
# =====================================================
fig4 = px.scatter(
    ship_summary,
    x="Sales",
    y="Profit",
    size="Sales",
    color="Profit_Margin_%",
    color_continuous_scale="RdYlGn",
    text="Ship_Mode",
    custom_data=["Profit_Margin_%", "Orders", "Avg_Order_Value"],
    title="Sales vs Profit Efficiency"
)

fig4.update_traces(
    textposition="top center",
    marker_line_color=TEXT_COLOR,
    marker_line_width=1.5,
    hovertemplate=build_hover_template(
        title_template="<b>%{text}</b>",
        lines=[
            {"label": "Sales", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[0]", "format_spec": ".2f", "suffix": "%"},
            {"label": "Orders", "expr": "customdata[1]", "format_spec": ",.0f"},
            {"label": "Avg Order Value", "expr": "customdata[2]", "format_spec": ",.2f", "prefix": "$"},
        ],
    ),
    customdata=np.stack([
        ship_summary["Profit_Margin_%"],
        ship_summary["Orders"],
        ship_summary["Avg_Order_Value"]
    ], axis=-1)
)

fig4.add_vline(x=ship_summary["Sales"].mean(), line_dash="dash", line_color = NEUTRAL_GRAY)
fig4.add_hline(y=ship_summary["Profit"].mean(), line_dash="dash", line_color = NEUTRAL_GRAY)

fig4 = style_plotly(
    fig4,
    title="Sales vs Profit by Ship Mode",
    x_title="Total Sales",
    y_title="Total Profit",
)

fig4.show()

# =====================================================
# 5. DISPLAY TABLE
# =====================================================
display(
    ship_summary.style
    .apply(style_ship_mode, subset=["Ship_Mode"])
    .background_gradient(cmap="Blues", subset=["Sales"])
    .background_gradient(cmap="Greens", subset=["Profit"])
    .background_gradient(cmap="Oranges", subset=["Profit_Margin_%"])
    .background_gradient(cmap="Purples", subset=["Orders"])
    .background_gradient(cmap="YlGnBu", subset=["Avg_Order_Value"])
    .format({
        "Sales": "${:,.0f}",
        "Profit": "${:,.0f}",
        "Profit_Margin_%": "{:.2f}%",
        "Orders": "{:,.0f}",
        "Avg_Order_Value": "${:,.2f}"
    })
)

[14:35:36] 🔹 Analyzing shipping performance


,Ship_Mode,Sales,Profit,Orders,Profit_Margin_%,Avg_Order_Value
0,Standard Class,"$7,578,652","$890,596","15,154",11.75%,$500.11
1,Second Class,"$2,565,672","$292,584","5,119",11.40%,$501.21
2,First Class,"$1,830,976","$208,105","3,821",11.37%,$479.19
3,Same Day,"$667,202","$76,173","1,347",11.42%,$495.32


In [124]:
# ADVANCED SHIPPING_DAYS DISTRIBUTION ANALYSIS — INTERACTIVE

work_df = ensure_shipping_days(df)

shipping_data = work_df["Shipping_Days"].dropna()

if not shipping_data.empty:
    # ---------------------------------------------------------
    # Basic stats
    # ---------------------------------------------------------
    mean_days = shipping_data.mean()
    median_days = shipping_data.median()
    std_days = shipping_data.std()
    q1 = shipping_data.quantile(0.25)
    q3 = shipping_data.quantile(0.75)
    iqr = q3 - q1
    min_days = shipping_data.min()
    max_days = shipping_data.max()

    # =====================================================
    # 1) HISTOGRAM — INTERACTIVE
    # =====================================================
    fig1 = go.Figure()

    fig1.add_trace(
        go.Histogram(
            x=shipping_data,
            nbinsx=20,
            name="Shipping_Days",
            marker=dict(
                color=PRIMARY_BLUE,
                line=dict(color="white", width=1)
            ),
            opacity=0.9,
            hovertemplate=count_hover_template(
                x_label="Shipping_Days",
                y_label="Count"
            )
        )
    )

    fig1.add_vline(
        x=mean_days,
        line_dash="dash",
        line_color=PRIMARY_RED,
        annotation_text=f"Mean: {mean_days:.2f}",
        annotation_position="top right"
    )

    fig1.add_vline(
        x=median_days,
        line_dash="dash",
        line_color=PRIMARY_GREEN,
        annotation_text=f"Median: {median_days:.2f}",
        annotation_position="top left"
    )

    fig1 = style_plotly(
        fig1,
        title="Shipping Days Distribution",
        x_title="Shipping Days",
        y_title="Count",
        hovermode="x unified",
        bargap=0.08
    )

    fig1.show()

    # =====================================================
    # 2) BOXPLOT — INTERACTIVE
    # =====================================================
    fig2 = go.Figure()

    fig2.add_trace(
        go.Box(
            x=shipping_data,
            name="Shipping Days",
            orientation="h",
            boxpoints="outliers",
            jitter=0.35,
            pointpos=0,
            marker=dict(color=PRIMARY_ORANGE),
            line=dict(color=PRIMARY_ORANGE),
            fillcolor="rgba(234, 88, 12, 0.25)",
            hovertemplate=build_hover_template(
                title_template="<b>Shipping Days:</b> %{x}",
            )
        )
    )

    fig2 = style_plotly(
        fig2,
        title="Shipping Days Outliers",
        x_title="Shipping_Days",
        y_title="",
    )

    fig2.show()

    # =====================================================
    # 3) SHIPPING SPEED CATEGORIES — INTERACTIVE
    # =====================================================
    shipping_bins = [-np.inf, 2, 5, 8, np.inf]
    shipping_labels = ["Fast Delivery", "Standard Delivery", "Slow Delivery", "Very Delayed"]

    shipping_category = pd.cut(
        shipping_data,
        bins=shipping_bins,
        labels=shipping_labels,
        include_lowest=True
    )

    category_counts = (
        shipping_category.value_counts()
        .reindex(shipping_labels)
        .reset_index()
    )
    category_counts.columns = ["Shipping Category", "Count"]

    fig3 = px.bar(
        category_counts,
        x="Shipping Category",
        y="Count",
        text="Count",
        color="Shipping Category",
        color_discrete_sequence=PRIMARY_COLOR_SEQUENCE,
        title="Shipping Speed Categories"
    )

    fig3.update_traces(
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{x}</b>",
            lines=[
                {"label": "Orders", "expr": "y", "format_spec": ",.0f"},
            ],
        )
    )

    fig3 = style_plotly(
        fig3,
        title="Shipping Speed Categories",
        x_title="",
        y_title="Number of Orders",
        showlegend=False
    )

    fig3.show()

    # =====================================================
    # 4) SUMMARY TABLE — COLOURED + VISUALLY CLEAN
    # =====================================================
    shipping_stats_summary = pd.DataFrame({
        "Metric": ["Count", "Mean", "Std Dev", "Min", "25%", "Median", "75%", "Max", "IQR"],
        "Value": [
            shipping_data.count(),
            mean_days,
            std_days,
            min_days,
            q1,
            median_days,
            q3,
            max_days,
            iqr
        ]
    })

    display(
        shipping_stats_summary.style
        .set_properties(**{
            "text-align": "center",
            "padding": "10px",
            "font-size": "14px",
            "border": "1px solid #e5e7eb"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#0f172a"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #0f172a")
                ]
            },
            {
                "selector": "tr:nth-child(even)",
                "props": [("background-color", "#000305")]
            },
            {
                "selector": "tr:nth-child(odd)",
                "props": [("background-color", "#010000")]
            },
            {
                "selector": "tr:hover",
                "props": [("background-color", "#c011f6")]
            }
        ])
        .background_gradient(cmap="coolwarm", subset=["Value"])
        .bar(subset=["Value"], color=PRIMARY_BLUE)
        .format({"Value": "{:.2f}"})
    )
else:
    print("Shipping_Days could not be derived because the required date columns are missing.")

,Metric,Value
0,Count,51290.00
1,Mean,3.97
2,Std Dev,1.73
3,Min,0.00
4,25%,3.00
5,Median,4.00
6,75%,5.00
7,Max,7.00
8,IQR,2.00


## 8. Discount and Profit Relationship

In [125]:
# DISCOUNT VS PROFIT ANALYSIS
require(df, ["Discount", "Profit"])

work_df = df.copy()

fig = px.scatter(
    work_df,
    x="Discount",
    y="Profit",
    opacity=0.5,
)

fig.add_hline(y=0, line_dash="dash", line_color=PRIMARY_RED)

fig = style_plotly(
    fig,
    "Impact of Discount on Profit",
    x_title="Discount",
    y_title="Profit",
    showlegend=False,
    height=DEFAULT_HEIGHT,
)

fig.show()

In [126]:
# HORIZONTAL DISCOUNT VS PROFIT ANALYSIS
log_step("Analyzing discount impact on profit", "STEP")

require(df, ["Discount", "Profit", "Sales"])

# Create discount ranges
bins = [-0.01, 0.1, 0.2, 0.3, 0.5, 1]
labels = [
    "0-10%",
    "10-20%",
    "20-30%",
    "30-50%",
    "50%+"
]

temp_df = df.copy()

temp_df["Discount Range"] = pd.cut(
    temp_df["Discount"],
    bins=bins,
    labels=labels
)

# Aggregate metrics
discount_analysis = (
    temp_df.groupby("Discount Range", observed=False)
    .agg({
        "Sales": "sum",
        "Profit": "sum"
    })
    .reset_index()
)

# Profit Margin
discount_analysis["Profit Margin %"] = (
    discount_analysis["Profit"] / discount_analysis["Sales"]
) * 100

# Sort properly
discount_analysis = discount_analysis.sort_values(
    by="Profit",
    ascending=True
)


# =====================================================
# STRUCTURAL DISCOUNT RISK DIAGNOSTICS
# =====================================================

high_discount_mask = df["Discount"] > 0.50

high_discount_count = df[high_discount_mask].shape[0]

loss_making_pct = (
    (df.loc[high_discount_mask, "Profit"] <= 0)
    .mean()
) * 100

print(f"Extreme-discount transactions (>50%): {high_discount_count:,}")

print(
    f"Loss-making or break-even rate among heavily discounted transactions: "
    f"{loss_making_pct:.2f}%"
)


# Horizontal Bar Chart
fig = px.bar(
    discount_analysis,
    y="Discount Range",
    x="Profit",
    color="Profit Margin %",
    orientation="h",
    text_auto=".2s",
    color_continuous_scale=[
        [0, PRIMARY_RED],
        [0.5, PRIMARY_ORANGE],
        [1, PRIMARY_GREEN]
    ],
)

# Zero-profit reference line
fig.add_vline(
    x=0,
    line_dash="dash",
    line_color=PRIMARY_RED,
    annotation_text="Loss Zone"
)

# Layout Improvements
fig = style_plotly(
    fig,
    title="Profitability Across Discount Levels",
    x_title="Total Profit",
    y_title="Discount Range",
    coloraxis_colorbar_title="Margin %",
)

# Styling
fig.update_traces(
    textposition="outside",
    marker_line_width=1.5
)


fig.show()

[14:35:37] 🔹 Analyzing discount impact on profit
Extreme-discount transactions (>50%): 4,172
Loss-making or break-even rate among heavily discounted transactions: 100.00%


## 9. Correlation Analysis

In [127]:
# CORRELATION HEATMAP FOR NUMERIC METRICS
log_step("Computing Spearman correlation matrix", "STEP")

# Select numeric business columns only
numeric_df = (
    df.select_dtypes(include=np.number)
      .drop(columns=["Row_ID"], errors="ignore")
)

corr = numeric_df.corr(method="spearman")

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)

# Figure
fig = px.imshow(
    corr_masked,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    zmin=-1,
    zmax=1,
    aspect="auto",
)

fig = style_plotly(
    fig,
    "Spearman Correlation Between Business Metrics",
    showlegend=False,
    height=DEFAULT_HEIGHT,
    margin=dict(l=60, r=40, t=80, b=40),
)

fig.show()

[14:35:37] 🔹 Computing Spearman correlation matrix


## 10. Outlier Analysis

In [128]:
# BUSINESS OUTLIER DETECTION — VIOLIN + BOX
log_step("Running outlier detection", "STEP")

require(df, ["Sales", "Profit"])

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Sales Distribution", "Profit Distribution"),
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Violin(
        y=df["Sales"],
        name="Sales",
        box_visible=True,
        meanline_visible=True,
        points="outliers",
        line_color=PRIMARY_BLUE,
        fillcolor=PRIMARY_BLUE,
        opacity=0.7,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Violin(
        y=df["Profit"],
        name="Profit",
        box_visible=True,
        meanline_visible=True,
        points="outliers",
        line_color=PRIMARY_GREEN,
        fillcolor=PRIMARY_GREEN,
        opacity=0.7,
    ),
    row=1,
    col=2,
)

fig = style_plotly(
    fig,
    title="Business Outlier Detection",
    showlegend=False,
    height=DEFAULT_HEIGHT,
)

fig.show()

[14:35:37] 🔹 Running outlier detection


In [129]:
# COLUMNS FOR DISTRIBUTION & OUTLIER ANALYSIS
distribution_columns = [
    ("Sales", "Sales Distribution", "Sales"),
    ("Profit", "Profit Distribution", "Profit"),
    ("Quantity", "Quantity Distribution", "Quantity"),
    ("Discount", "Discount Distribution", "Discount")
]


#Box Plot for Outlier Analysis
for col, title, xlabel in distribution_columns:
    
    if col in df.columns:
        
        fig = px.box(
            df,
            x=col,
            points="outliers",
        )

        fig = style_plotly(
            fig,
            f"{title} (Boxplot)",
            x_title=xlabel,
            showlegend=False,
            height=DEFAULT_HEIGHT,
        )

        fig.show()

## 11. Customer and Product Analysis

In [130]:
# CUSTOMER PURCHASE FREQUENCY ANALYSIS
# Segments:
# 1) One-Time Customers
# 2) Occasional Customers (2–5 orders)
# 3) Regular Customers (6–15 orders)
# 4) Power Customers (16+ orders)

log_step("Analyzing customer purchase frequency", "STEP")

customer_key = "Customer_ID" if "Customer_ID" in df.columns else "Customer_Name"
require(df, [customer_key, "Order_ID"])

# Keep one row per customer-order pair to avoid line-item duplication
customer_orders = (
    df[[customer_key, "Order_ID"]]
    .dropna(subset=[customer_key, "Order_ID"])
    .drop_duplicates(subset=[customer_key, "Order_ID"])
    .copy()
)

# Count distinct orders per customer
customer_frequency = (
    customer_orders.groupby(customer_key, as_index=False)
    .agg(Total_Orders=("Order_ID", "nunique"))
)

# Exclusive segmentation
customer_frequency["Customer_Segment"] = pd.cut(
    customer_frequency["Total_Orders"],
    bins=[0, 1, 5, 15, float("inf")],
    labels=[
        "One-Time Customers",
        "Occasional Customers",
        "Regular Customers",
        "Power Customers"
    ],
    include_lowest=True,
    right=True
)

# Summary table
segment_order = [
    "One-Time Customers",
    "Occasional Customers",
    "Regular Customers",
    "Power Customers"
]

frequency_summary = (
    customer_frequency["Customer_Segment"]
    .value_counts()
    .reindex(segment_order)
    .dropna()
    .reset_index()
)
frequency_summary.columns = ["Customer_Segment", "Customer_Count"]

total_customers = len(customer_frequency)

frequency_summary["Customer_Share_%"] = (
    frequency_summary["Customer_Count"] / total_customers * 100
    if total_customers else np.nan
)

# Segment counts
one_time_count = int(frequency_summary.loc[frequency_summary["Customer_Segment"] == "One-Time Customers", "Customer_Count"].sum())
occasional_count = int(frequency_summary.loc[frequency_summary["Customer_Segment"] == "Occasional Customers", "Customer_Count"].sum())
regular_count = int(frequency_summary.loc[frequency_summary["Customer_Segment"] == "Regular Customers", "Customer_Count"].sum())
power_count = int(frequency_summary.loc[frequency_summary["Customer_Segment"] == "Power Customers", "Customer_Count"].sum())

one_time_rate = one_time_count / total_customers * 100 if total_customers else np.nan
occasional_rate = occasional_count / total_customers * 100 if total_customers else np.nan
regular_rate = regular_count / total_customers * 100 if total_customers else np.nan
power_rate = power_count / total_customers * 100 if total_customers else np.nan

# KPI cards
show_kpi_cards([
    ("Total Customers", f"{fmt_number(total_customers)}", "Unique customers analyzed"),
    ("One-Time Customers", f"{fmt_number(one_time_count)} ({one_time_rate:.2f}%)", "Exactly 1 order"),
    ("Occasional Customers", f"{fmt_number(occasional_count)} ({occasional_rate:.2f}%)", "2–5 orders"),
    ("Regular Customers", f"{fmt_number(regular_count)} ({regular_rate:.2f}%)", "6–15 orders"),
    ("Power Customers", f"{fmt_number(power_count)} ({power_rate:.2f}%)", "16+ orders"),
])

# Display summary table
display(
    frequency_summary.style.format({
        "Customer_Count": "{:,.0f}",
        "Customer_Share_%": "{:.2f}%"
    })
)

# Donut chart
fig = px.pie(
    frequency_summary,
    names="Customer_Segment",
    values="Customer_Count",
    hole=0.62,
    color="Customer_Segment",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(
    textinfo="percent+label",
    marker=dict(line=dict(color="white", width=2))
)

fig = style_plotly(
    fig,
    title="Customer Purchase Frequency Segments",
    legend_title="Customer Segment"
)
fig.show()

print(
    f"""
Key Insights

• Power Customers account for {power_rate:.2f}% of all customers, making them the dominant customer segment.

• Regular Customers account for {regular_rate:.2f}% of customers and represent the strongest opportunity for conversion into Power Customers.

• Occasional Customers account for {occasional_rate:.2f}% of customers and may benefit from targeted retention campaigns.

• One-Time Customers represent only {one_time_rate:.2f}% of customers, indicating very strong repeat purchasing behavior across the customer base.
"""
)

print("""
⚠️  Note: Frequency is measured per Customer_ID (market account),
    not per unique individual. Since each customer name maps to 2 IDs
    (one per market region), order counts are split. A customer with
    20 total orders may appear as 2 accounts with 10 orders each,
    both classified as Regular rather than Power.
    Interpret segment percentages as per-account, not per-person.
""")

[14:35:37] 🔹 Analyzing customer purchase frequency


,Customer_Segment,Customer_Count,Customer_Share_%
0,One-Time Customers,10,0.63%
1,Occasional Customers,306,19.25%
2,Regular Customers,488,30.69%
3,Power Customers,786,49.43%



Key Insights

• Power Customers account for 49.43% of all customers, making them the dominant customer segment.

• Regular Customers account for 30.69% of customers and represent the strongest opportunity for conversion into Power Customers.

• Occasional Customers account for 19.25% of customers and may benefit from targeted retention campaigns.

• One-Time Customers represent only 0.63% of customers, indicating very strong repeat purchasing behavior across the customer base.


⚠️  Note: Frequency is measured per Customer_ID (market account),
    not per unique individual. Since each customer name maps to 2 IDs
    (one per market region), order counts are split. A customer with
    20 total orders may appear as 2 accounts with 10 orders each,
    both classified as Regular rather than Power.
    Interpret segment percentages as per-account, not per-person.



In [131]:
# TOP CUSTOMERS BY SALES & PROFIT

log_step("Analyzing customer profitability", "STEP")

require(df, ["Customer_ID", "Customer_Name", "Sales", "Profit"])

customer_perf = aggregate_sales_profit(
    df,
    group_col=["Customer_ID", "Customer_Name"],
    sort_by="Sales",
    ascending=False
).head(10).copy()

customer_perf["Customer_Display"] = (
    customer_perf["Customer_Name"].astype(str)
    + " ("
    + customer_perf["Customer_ID"].astype(str)
    + ")"
)

fig = px.bar(
    customer_perf,
    x="Customer_Display",
    y=["Sales", "Profit"],
    barmode="group",
    text_auto=".2s",
    color_discrete_sequence=[PRIMARY_BLUE, PRIMARY_GREEN],
)

fig = style_plotly(
    fig,
    title="Top 10 Customers by Sales & Profit",
    x_title="Customer",
    y_title="Amount ($)",
    legend_title="Financial Metric",
    x_tickangle=-90,
)

fig.show()

[14:35:38] 🔹 Analyzing customer profitability


In [132]:
# CUSTOMER SALES VS PROFIT SCATTER

require(df, ["Customer_ID", "Customer_Name", "Sales", "Profit"])

customer_scatter = aggregate_sales_profit(
    df,
    group_col=["Customer_ID", "Customer_Name"]
).copy()

customer_scatter["Status"] = customer_scatter["Profit"].apply(
    lambda x: "Profitable" if x >= 0 else "Loss-making"
)

customer_scatter["Customer_Display"] = (
    customer_scatter["Customer_Name"].astype(str)
    + " ("
    + customer_scatter["Customer_ID"].astype(str)
    + ")"
)

if (customer_scatter["Sales"] < 0).any():
    raise ValueError(
        "Customer Sales contains negative values. Check upstream data cleaning."
    )

fig = px.scatter(
    customer_scatter,
    x="Sales",
    y="Profit",
    color="Status",
    size="Sales",
    size_max=30,
    hover_name="Customer_Display",
    custom_data=["Customer_ID", "Customer_Name", "Sales", "Profit", "Status"],
    color_discrete_map={
        "Profitable": PRIMARY_GREEN,
        "Loss-making": PRIMARY_RED,
    },
)

fig.add_hline(y=0, line_dash="dash", line_color=NEUTRAL_GRAY, line_width=1)
fig.add_vline(
    x=customer_scatter["Sales"].median(),
    line_dash="dash",
    line_color=NEUTRAL_GRAY,
    line_width=1,
)

fig = style_plotly(
    fig,
    title="Customer Sales vs Profit",
    x_title="Total Sales",
    y_title="Total Profit",
    legend_orientation="h",
    legend_y=1.02,
    legend_x=0.5,
    legend_xanchor="center",
    legend_yanchor="bottom",
)

fig.update_traces(
    marker=dict(opacity=0.80, line=dict(width=1, color="white")),
    hovertemplate=build_hover_template(
        title_template="<b>%{hovertext}</b>",
        lines=[
            {"label": "Customer ID", "expr": "customdata[0]"},
            {"label": "Sales", "expr": "customdata[2]", "format_spec": ",.0f", "prefix": "$"},
            {"label": "Profit", "expr": "customdata[3]", "format_spec": ",.0f", "prefix": "$"},
            {"label": "Status", "expr": "customdata[4]"},
        ],
    ),
)

fig.show()

In [133]:
# LOSS-MAKING CUSTOMERS

require(df, ["Customer_ID", "Customer_Name", "Sales", "Profit"])

customer_loss = aggregate_sales_profit(
    df,
    group_col=["Customer_ID", "Customer_Name"]
).copy()

customer_loss["Total Loss"] = np.where(
    customer_loss["Profit"] < 0,
    -customer_loss["Profit"],
    0
)

loss_customers = (
    customer_loss[customer_loss["Profit"] < 0]
    .sort_values("Total Loss", ascending=False)
    .head(15)
    .copy()
)

loss_customers["Customer_Display"] = (
    loss_customers["Customer_Name"].astype(str)
    + " ("
    + loss_customers["Customer_ID"].astype(str)
    + ")"
)

loss_customers["Short Name"] = loss_customers["Customer_Display"].apply(shorten_label)

loss_customer_plot_df = loss_customers.sort_values("Total Loss", ascending=True).copy()

fig = px.bar(
    loss_customer_plot_df,
    x="Total Loss",
    y="Short Name",
    orientation="h",
    color="Total Loss",
    color_continuous_scale=LOSS_SCALE,
)

fig.update_traces(
    customdata=np.stack([
        loss_customer_plot_df["Customer_ID"],
        loss_customer_plot_df["Customer_Name"],
        loss_customer_plot_df["Sales"],
        loss_customer_plot_df["Profit"],
    ], axis=-1),
    hovertemplate=build_hover_template(
        title_template="<b>%{customdata[1]} (%{customdata[0]})</b>",
        lines=[
            {"label": "Loss", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Sales", "expr": "customdata[2]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit", "expr": "customdata[3]", "format_spec": ",.2f", "prefix": "$"},
        ],
    ),
    text=loss_customer_plot_df["Total Loss"].map(lambda x: f"${x:,.0f}"),
    textposition="outside",
    cliponaxis=False,
    marker_line_width=0
)

fig = style_plotly(
    fig,
    title="Top Loss-Making Customers",
    x_title="Total Loss ($)",
    y_title="",
    coloraxis_showscale=False,
    margin=dict(l=260, r=50, t=100, b=50),
)

fig.add_vline(
    x=0,
    line_width=1.2,
    line_dash="dash",
    line_color=NEUTRAL_GRAY
)

fig.add_annotation(
    text="Customers at the bottom are causing the heaviest losses",
    x=0.5,
    y=1.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=12, color=NEUTRAL_GRAY),
)

fig.show()

In [134]:
# TOP PRODUCTS BY SALES

log_step("Analyzing product profitability", "STEP")

require(df, ["Product_Name", "Sales", "Profit", "Quantity"])

top_sales_products = aggregate_metrics(df, "Product_Name").head(10).copy()

top_sales_products["Short Name"] = top_sales_products["Product_Name"].apply(shorten_label)

top_products_plot_df = top_sales_products.sort_values("Sales", ascending=True)

fig = px.bar(
        top_products_plot_df,
        x="Sales",
        y="Short Name",
        orientation="h",
        color="Sales",
        color_continuous_scale="Blues",
    )

fig.update_traces(
        customdata=np.stack([
            top_products_plot_df["Product_Name"],
            top_products_plot_df["Profit"],
            top_products_plot_df["Quantity"]
        ], axis=-1),

        hovertemplate=build_hover_template(
            title_template="<b>%{customdata[0]}</b>",
            lines=[
                {"label": "Sales", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Quantity Sold", "expr": "customdata[2]", "format_spec": ",.0f"},
            ],
        ),

        text=top_products_plot_df["Sales"].map(lambda x: f"${x:,.0f}"),
        textposition="outside",
        cliponaxis=False
    )

fig = style_plotly(
    fig,
    title="Top 10 Products by Sales",
    x_title="Total Sales ($)",
    y_title="Products",
    coloraxis_showscale=False,
    margin=dict(l=260, r=50, t=100, b=50),
)

fig.show()

[14:35:38] 🔹 Analyzing product profitability


In [135]:
# MOST PROFITABLE PRODUCTS

require(df, ["Product_Name", "Sales", "Profit", "Quantity"])

profitable_products = (
    aggregate_metrics(df, "Product_Name", sort_by="Profit", ascending=False)
    .head(10)
    .copy()
)

profitable_products["Short Name"] = profitable_products["Product_Name"].apply(shorten_label)

profit_products_plot_df = profitable_products.sort_values("Profit", ascending=True)

fig = px.bar(
        profit_products_plot_df,
        x="Profit",
        y="Short Name",
        orientation="h",
        color="Profit",
        color_continuous_scale="Greens",
    )

fig.update_traces(
        customdata=np.stack([
            profit_products_plot_df["Product_Name"],
            profit_products_plot_df["Sales"],
            profit_products_plot_df["Quantity"]
        ], axis=-1),

        hovertemplate=build_hover_template(
            title_template="<b>%{customdata[0]}</b>",
            lines=[
                {"label": "Profit", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Sales", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Quantity Sold", "expr": "customdata[2]", "format_spec": ",.0f"},
            ],
        ),

        text=profit_products_plot_df["Profit"].map(lambda x: f"${x:,.0f}"),
        textposition="outside",
        cliponaxis=False
    )

fig = style_plotly(
    fig,
    title="Most Profitable Products",
    x_title="Total Profit ($)",
    y_title="",
    coloraxis_showscale=False,
    margin=dict(l=260, r=50, t=100, b=50),
)

fig.show()

In [136]:
# LOSS-MAKING PRODUCTS

require(df, ["Product_Name", "Sales", "Profit", "Quantity"])

product_loss = aggregate_metrics(df, "Product_Name").copy()

product_loss["Total Loss"] = np.where(
        product_loss["Profit"] < 0,
        -product_loss["Profit"],
        0
    )

loss_products = (
        product_loss[product_loss["Profit"] < 0]
        .sort_values("Total Loss", ascending=False)
        .head(10)
        .copy()
    )

loss_products["Short Name"] = loss_products["Product_Name"].apply(shorten_label)

loss_making_plot_df = loss_products.sort_values("Total Loss", ascending=True)

fig = px.bar(
        loss_making_plot_df,
        x="Total Loss",
        y="Short Name",
        orientation="h",
        color="Total Loss",
        color_continuous_scale=LOSS_SCALE,
    )

fig.update_traces(
        customdata=np.stack([
            loss_making_plot_df["Product_Name"],
            loss_making_plot_df["Sales"],
            loss_making_plot_df["Profit"],
            loss_making_plot_df["Quantity"]
        ], axis=-1),

        hovertemplate=build_hover_template(
            title_template="<b>%{customdata[0]}</b>",
            lines=[
                {"label": "Loss", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Sales", "expr": "customdata[1]", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Profit", "expr": "customdata[2]", "format_spec": ",.2f", "prefix": "$"},
                {"label": "Quantity Sold", "expr": "customdata[3]", "format_spec": ",.0f"},
            ],
        ),

        text=loss_making_plot_df["Total Loss"].map(lambda x: f"${x:,.0f}"),
        textposition="outside",
        cliponaxis=False
    )

fig = style_plotly(
    fig,
    title="Top Loss-Making Products",
    x_title="Total Loss ($)",
    y_title="",
    coloraxis_showscale=False,
    margin=dict(l=260, r=50, t=100, b=50),
)

fig.show()

In [137]:
# PRODUCT PROFITABILITY FOCUS VIEW

require(df, ["Product_Name", "Sales", "Profit", "Quantity"])

# Aggregate product-level data
product_matrix = (
    df.groupby("Product_Name", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
)

# Profit margin
product_matrix["Profit Margin %"] = np.where(
    product_matrix["Sales"] != 0,
    (product_matrix["Profit"] / product_matrix["Sales"]) * 100,
    0
)

# Keep the chart readable:
# - top sales products
# - all loss-making products
top_sales_n = 70
focus_df = pd.concat([
    product_matrix.sort_values("Sales", ascending=False).head(top_sales_n),
    product_matrix[product_matrix["Profit"] < 0]
]).drop_duplicates(subset=["Product_Name"]).copy()

# Bubble size scaling
max_qty = focus_df["Quantity"].max()
focus_df["Bubble Size"] = np.clip(
    (focus_df["Quantity"] / max_qty) * 35,
    8,
    45
)

fig = go.Figure()

# Main bubbles
fig.add_trace(
    go.Scatter(
        x=focus_df["Sales"],
        y=focus_df["Profit"],
        mode="markers",
        name="Products",
        showlegend=False,
        customdata=np.stack([
            focus_df["Product_Name"],
            focus_df["Quantity"],
            focus_df["Profit Margin %"]
        ], axis=-1),
        marker=dict(
            size=focus_df["Bubble Size"],
            color=focus_df["Profit Margin %"],
            colorscale="RdYlGn",
            showscale=True,
            colorbar=dict(title="Profit<br>Margin %"),
            opacity=0.65,
            line=dict(width=1, color="white")
        ),
        hovertemplate=build_hover_template(
            title_template="<b>%{customdata[0]}</b>",
            lines=[
                {"label": "Sales", "expr": "x", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Profit", "expr": "y", "format_spec": ",.0f", "prefix": "$"},
                {"label": "Quantity", "expr": "customdata[1]", "format_spec": ",.0f"},
                {"label": "Profit Margin", "expr": "customdata[2]", "format_spec": ".2f", "suffix": "%"},
            ],
        )
    )
)

# Reference lines
pffv_sales_mid = focus_df["Sales"].median()
fig.add_vline(x=pffv_sales_mid, line_dash="dash", line_color = NEUTRAL_GRAY, opacity=0.6)
fig.add_hline(y=0, line_dash="dash", line_color = NEUTRAL_GRAY, opacity=0.6)

# Quadrant shading
x_min = focus_df["Sales"].min()
x_max = focus_df["Sales"].max()
y_min = focus_df["Profit"].min()
y_max = focus_df["Profit"].max()

fig.add_shape(type="rect", x0=pffv_sales_mid, x1=x_max, y0=0, y1=y_max,
              fillcolor="rgba(22, 163, 74, 0.08)", line=dict(width=0), layer="below")
fig.add_shape(type="rect", x0=pffv_sales_mid, x1=x_max, y0=y_min, y1=0,
              fillcolor="rgba(220, 38, 38, 0.08)", line=dict(width=0), layer="below")
fig.add_shape(type="rect", x0=x_min, x1=pffv_sales_mid, y0=0, y1=y_max,
              fillcolor="rgba(37, 99, 235, 0.06)", line=dict(width=0), layer="below")
fig.add_shape(type="rect", x0=x_min, x1=pffv_sales_mid, y0=y_min, y1=0,
              fillcolor="rgba(234, 88, 12, 0.06)", line=dict(width=0), layer="below")

# Quadrant labels
fig.add_annotation(x=x_min + (pffv_sales_mid - x_min) * 0.18, y=y_max * 0.92,
                    text="<b>Low Sales<br>Profitable</b>", showarrow=False,
                    font=dict(size=16, color=PRIMARY_GREEN), align="left")
fig.add_annotation(x=pffv_sales_mid + (x_max - pffv_sales_mid) * 0.18, y=y_max * 0.92,
                    text="<b>High Sales<br>Profitable</b>", showarrow=False,
                    font=dict(size=16, color=PRIMARY_GREEN), align="left")
fig.add_annotation(x=x_min + (pffv_sales_mid - x_min) * 0.18, y=y_min * 0.92,
                    text="<b>Low Sales<br>Loss-making</b>", showarrow=False,
                    font=dict(size=16, color=PRIMARY_GREEN), align="left")
fig.add_annotation(x=pffv_sales_mid + (x_max - pffv_sales_mid) * 0.18, y=y_min * 0.92,
                    text="<b>High Sales<br>Loss-making</b>", showarrow=False,
                    font=dict(size=16, color=PRIMARY_RED), align="left")

fig = style_plotly(
    fig,
    title="Product Profitability Focus View",
    x_title="Total Sales ($)",
    y_title="Total Profit ($)",
)

fig.show()


## 12. Advanced Analytics

This section transforms exploratory analysis into business intelligence by introducing:

- RFM customer segmentation
- Sales forecasting
- Profit driver analysis

The goal is to identify:
- high-value customers,
- expected future sales trends,
- and operational factors influencing profitability.

### 12.1 RFM Customer Segmentation

### RFM Segmentation Logic

Customers are segmented using:

- **Recency** → how recently the customer purchased
- **Frequency** → how often the customer purchases
- **Monetary** → how much revenue the customer generates

Segments represent different customer engagement and value levels:

- **Champions** → highly active and high-value customers
- **Loyal Customers** → repeat and reliable customers
- **Potential Loyalists** → growing engagement customers
- **At Risk** → previously active customers showing decline
- **Lost Customers** → inactive and low-engagement customers


In [138]:
# RFM CUSTOMER SEGMENTATION

log_step("Building RFM customer segmentation", "STEP")

rfm = build_rfm_table(df)

save_csv(rfm, "rfm_summary.csv")

display(rfm.head())

[14:35:38] 🔹 Building RFM customer segmentation
✅ Saved: ../exports/rfm_summary.csv


,Customer_ID,Customer_Name,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,GT-14710,Greg Tran,9,30,34471.89028,5,5,5,15,Champions
1,SE-20110,Sanjit Engle,10,36,29532.62502,5,5,5,15,Champions
2,ZC-21910,Zuschuss Carroll,4,37,28472.81926,5,5,5,15,Champions
3,SP-20920,Susan Pistek,7,28,28124.21286,5,5,5,15,Champions
4,NF-18385,Natalie Fritzler,8,32,28044.35950,5,5,5,15,Champions


In [139]:
# RFM SEGMENT VISUALIZATIONS

log_step("Building RFM segment visualizations", "STEP")

rfm_local = build_rfm_table(df)

rfm_plot = rfm_local.copy()
rfm_plot["Segment"] = rfm_plot["Segment"].astype(str)

segment_order = [
    "Champions",
    "Loyal Customers",
    "Potential Loyalists",
    "At Risk",
    "Lost Customers",
    "Unknown",
]

# ---------------------------------------------------------
# Segment Distribution
# ---------------------------------------------------------

segment_counts = (
    rfm_plot["Segment"]
    .value_counts()
    .reindex(segment_order, fill_value=0)
    .reset_index()
)

segment_counts.columns = ["Segment", "Customers"]

fig = px.bar(
    segment_counts,
    x="Segment",
    y="Customers",
    text="Customers",
    color="Customers",
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Customer Distribution by RFM Segment"
)

fig.update_traces(
    texttemplate="%{text}",
    textposition="inside",
    insidetextanchor="start",
    cliponaxis=False,
    hovertemplate="<b>%{x}</b><br>Customers: %{y:,}<extra></extra>",
)

style_plotly(
    fig,
    title="Customer Distribution by RFM Segment",
    x_title="Customer Segment",
    y_title="Number of Customers",
)

fig.update_xaxes(categoryorder="array", categoryarray=segment_order)
fig.show()

# ---------------------------------------------------------
# Revenue by Segment
# ---------------------------------------------------------

segment_revenue = (
    rfm_plot.groupby("Segment", as_index=False)
    .agg(
        Total_Sales=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
    )
)

segment_revenue["Segment"] = pd.Categorical(
    segment_revenue["Segment"],
    categories=segment_order,
    ordered=True,
)

segment_revenue = segment_revenue.sort_values("Segment").reset_index(drop=True)

fig = px.bar(
    segment_revenue,
    x="Segment",
    y="Total_Sales",
    text_auto=".2s",
    color="Total_Sales",
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Revenue Contribution by Customer Segment"
)

fig.update_traces(
    hovertemplate="<b>%{x}</b><br>Total Sales: %{y:$,.2f}<extra></extra>",
)

style_plotly(
    fig,
    title="Revenue Contribution by Customer Segment",
    x_title="Customer Segment",
    y_title="Total Revenue",
)

fig.update_xaxes(categoryorder="array", categoryarray=segment_order)
fig.show()

[14:35:38] 🔹 Building RFM segment visualizations


In [140]:
# RFM SEGMENT SUMMARY TABLE

log_step("Building RFM segment summary table", "STEP")

rfm_segment_summary = build_rfm_segment_summary(df)

save_csv(rfm_segment_summary, "rfm_segment_summary.csv")

display(
    rfm_segment_summary.style.format({
        "Customers": "{:,}",
        "Avg_Recency": "{:.1f}",
        "Avg_Frequency": "{:.1f}",
        "Avg_Monetary": "${:,.2f}",
        "Total_Sales": "${:,.2f}",
        "Total_Profit": "${:,.2f}",
        "Customer_Share_%": "{:.1f}%",
        "Revenue_Share_%": "{:.1f}%",
        "Profit_Share_%": "{:.1f}%",
        "Profit_Margin_%": "{:.1f}%",
    })
)

[14:35:38] 🔹 Building RFM segment summary table
✅ Saved: ../exports/rfm_segment_summary.csv


,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Sales,Total_Profit,Customer_Share_%,Revenue_Share_%,Profit_Share_%,Profit_Margin_%,Recommended_Action
0,Champions,874,25.6,24.3,"$12,771.38","$11,162,181.82","$1,340,514.24",55.0%,88.3%,91.3%,12.0%,"Reward with loyalty perks, exclusive offers, and premium cross-sell opportunities."
1,Loyal Customers,222,57.5,8.8,"$3,256.52","$722,948.33","$83,918.84",14.0%,5.7%,5.7%,11.6%,Encourage repeat purchases with upsell bundles and relationship-building campaigns.
2,Potential Loyalists,246,135.3,6.4,"$2,123.43","$522,362.69","$33,475.41",15.5%,4.1%,2.3%,6.4%,Nurture with personalized promotions to increase frequency and basket size.
3,At Risk,248,271.9,4.2,$947.62,"$235,009.07","$9,548.81",15.6%,1.9%,0.7%,4.1%,"Trigger retention campaigns, targeted discounts, and reactivation outreach."


In [141]:
# RFM SEGMENT BUSINESS ACTIONS

log_step("Building RFM segment business actions", "STEP")

rfm_segment_summary = build_rfm_segment_summary(df)
business_view = rfm_segment_summary.copy()

rfm_business_actions = business_view[
    [
        "Segment",
        "Customers",
        "Customer_Share_%",
        "Revenue_Share_%",
        "Profit_Share_%",
        "Profit_Margin_%",
        "Recommended_Action",
    ]
].copy()

save_csv(rfm_business_actions, "rfm_business_actions.csv")

display(
    rfm_business_actions.style.format({
        "Customers": "{:,}",
        "Customer_Share_%": "{:.1f}%",
        "Revenue_Share_%": "{:.1f}%",
        "Profit_Share_%": "{:.1f}%",
        "Profit_Margin_%": "{:.1f}%",
    })
)

segment_order = [
    "Champions",
    "Loyal Customers",
    "Potential Loyalists",
    "At Risk",
    "Lost Customers",
    "Unknown",
]

share_long = business_view.melt(
    id_vars="Segment",
    value_vars=["Revenue_Share_%", "Profit_Share_%"],
    var_name="Metric",
    value_name="Share"
)

share_long["Metric"] = share_long["Metric"].replace({
    "Revenue_Share_%": "Revenue Share",
    "Profit_Share_%": "Profit Share",
})

share_long["Segment"] = pd.Categorical(
    share_long["Segment"],
    categories=segment_order,
    ordered=True,
)

fig = px.bar(
    share_long,
    x="Segment",
    y="Share",
    color="Metric",
    barmode="group",
    text="Share",
    title="Revenue Share vs Profit Share by RFM Segment",
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{color}: %{y:.1f}%<extra></extra>",
)

fig = style_plotly(
    fig,
    title="Revenue Share vs Profit Share by RFM Segment",
    x_title="Customer Segment",
    y_title="Share %",
)

fig.update_xaxes(categoryorder="array", categoryarray=segment_order)
fig.show()

[14:35:39] 🔹 Building RFM segment business actions
✅ Saved: ../exports/rfm_business_actions.csv


,Segment,Customers,Customer_Share_%,Revenue_Share_%,Profit_Share_%,Profit_Margin_%,Recommended_Action
0,Champions,874,55.0%,88.3%,91.3%,12.0%,"Reward with loyalty perks, exclusive offers, and premium cross-sell opportunities."
1,Loyal Customers,222,14.0%,5.7%,5.7%,11.6%,Encourage repeat purchases with upsell bundles and relationship-building campaigns.
2,Potential Loyalists,246,15.5%,4.1%,2.3%,6.4%,Nurture with personalized promotions to increase frequency and basket size.
3,At Risk,248,15.6%,1.9%,0.7%,4.1%,"Trigger retention campaigns, targeted discounts, and reactivation outreach."


In [142]:
# RFM RISK & OPPORTUNITY ANALYSIS

required_cols = ["Customer_ID", "Customer_Name", "Profit", "Sales"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"df is missing required columns for RFM risk analysis: {missing_cols}")

rfm_local = build_rfm_table(df)

required_rfm_cols = ["Customer_ID", "Customer_Name", "Segment", "Recency", "Frequency", "Monetary"]
missing_rfm_cols = [col for col in required_rfm_cols if col not in rfm_local.columns]
if missing_rfm_cols:
    raise ValueError(f"build_rfm_table() output is missing required columns: {missing_rfm_cols}")

rfm_segment_summary = rfm_local.copy()
rfm_segment_summary["Segment"] = rfm_segment_summary["Segment"].astype(str)
rfm_segment_summary = rfm_segment_summary.drop_duplicates(subset=["Customer_ID"]).copy()

customer_profit = (
    df.groupby("Customer_ID", as_index=False)
    .agg(
        Total_Profit=("Profit", "sum"),
        Total_Transaction_Sales=("Sales", "sum"),
    )
)
rfm_business = rfm_segment_summary.merge(customer_profit, on="Customer_ID", how="left")

segment_summary = (
    rfm_business.groupby("Segment", as_index=False)
    .agg(
        Customers=("Customer_ID", "nunique"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean"),
        Total_Sales=("Monetary", "sum"),
        Total_Profit=("Total_Profit", "sum"),
    )
)

total_customers = segment_summary["Customers"].sum()
total_sales = segment_summary["Total_Sales"].sum()
total_profit = segment_summary["Total_Profit"].sum()

segment_summary["Customer_Share_%"] = np.where(
    total_customers != 0,
    100 * segment_summary["Customers"] / total_customers,
    np.nan,
)
segment_summary["Revenue_Share_%"] = np.where(
    total_sales != 0,
    100 * segment_summary["Total_Sales"] / total_sales,
    np.nan,
)
segment_summary["Profit_Share_%"] = np.where(
    total_profit != 0,
    100 * segment_summary["Total_Profit"] / total_profit,
    np.nan,
)
segment_summary["Profit_Margin_%"] = np.where(
    segment_summary["Total_Sales"] != 0,
    100 * segment_summary["Total_Profit"] / segment_summary["Total_Sales"],
    np.nan,
)

segment_order = [
    "Champions",
    "Loyal Customers",
    "Potential Loyalists",
    "At Risk",
    "Lost Customers",
    "Unknown",
]
segment_summary["Segment"] = pd.Categorical(
    segment_summary["Segment"],
    categories=segment_order,
    ordered=True,
)
segment_summary = segment_summary.sort_values("Segment").reset_index(drop=True)

risk_view = segment_summary.copy()

def safe_value(segment_name, col_name, default=0.0):
    row = risk_view.loc[risk_view["Segment"] == segment_name, col_name]
    return float(row.iloc[0]) if not row.empty and pd.notna(row.iloc[0]) else default

def safe_count(segment_name, default=0):
    row = risk_view.loc[risk_view["Segment"] == segment_name, "Customers"]
    return int(row.iloc[0]) if not row.empty and pd.notna(row.iloc[0]) else default

champion_customers = safe_count("Champions")
loyal_customers = safe_count("Loyal Customers")
potential_customers = safe_count("Potential Loyalists")
at_risk_customers = safe_count("At Risk")
lost_customers = safe_count("Lost Customers")

champion_rev = safe_value("Champions", "Revenue_Share_%")
champion_profit = safe_value("Champions", "Profit_Share_%")
loyal_rev = safe_value("Loyal Customers", "Revenue_Share_%")
loyal_profit = safe_value("Loyal Customers", "Profit_Share_%")
potential_rev = safe_value("Potential Loyalists", "Revenue_Share_%")
potential_profit = safe_value("Potential Loyalists", "Profit_Share_%")
at_risk_rev = safe_value("At Risk", "Revenue_Share_%")
at_risk_profit = safe_value("At Risk", "Profit_Share_%")

lost_sales = safe_value("Lost Customers", "Total_Sales")
lost_profit = safe_value("Lost Customers", "Total_Profit")
lost_rev = safe_value("Lost Customers", "Revenue_Share_%")

if lost_customers == 0:
    lost_text = (
        "Lost Customers: No customers were classified as Lost Customers under the current "
        "RFM segmentation criteria, indicating the dataset is dominated by relatively active customers."
    )
else:
    lost_rev_text = "<0.01%" if lost_rev < 0.01 else f"{lost_rev:.2f}%"
    lost_text = (
        f"Lost Customers: {lost_customers:,} customers, contributing ${lost_sales:,.2f} in sales "
        f"and ${lost_profit:,.2f} in profit ({lost_rev_text} of revenue), suggesting limited current "
        f"business impact but possible reactivation potential."
    )

insights = [
    f"Champions: {champion_customers:,} customers, contributing {champion_rev:.1f}% of revenue and {champion_profit:.1f}% of profit, making them the highest-value retention group.",
    f"Loyal Customers: {loyal_customers:,} customers, contributing {loyal_rev:.1f}% of revenue and {loyal_profit:.1f}% of profit, making them a strong repeat-business segment.",
    f"Potential Loyalists: {potential_customers:,} customers, contributing {potential_rev:.1f}% of revenue and {potential_profit:.1f}% of profit, and they are a strong upsell and nurture target.",
    f"At Risk: {at_risk_customers:,} customers, accounting for {at_risk_rev:.1f}% of revenue and {at_risk_profit:.1f}% of profit, so churn here could materially affect performance.",
    lost_text,
]

for item in insights:
    print(f"- {item}")

- Champions: 874 customers, contributing 88.3% of revenue and 91.3% of profit, making them the highest-value retention group.
- Loyal Customers: 222 customers, contributing 5.7% of revenue and 5.7% of profit, making them a strong repeat-business segment.
- Potential Loyalists: 246 customers, contributing 4.1% of revenue and 2.3% of profit, and they are a strong upsell and nurture target.
- At Risk: 248 customers, accounting for 1.9% of revenue and 0.7% of profit, so churn here could materially affect performance.
- Lost Customers: No customers were classified as Lost Customers under the current RFM segmentation criteria, indicating the dataset is dominated by relatively active customers.


In [143]:
# RFM SEGMENT PROFIT VALIDATION

rfm_local = build_rfm_table(df)

segment_map = dict(zip(
    rfm_local["Customer_ID"].tolist(),
    rfm_local["Segment"].astype(str).tolist()
))

profit_by_segment = df[["Customer_ID", "Customer_Name", "Profit"]].copy()
profit_by_segment["RFM_Segment"] = profit_by_segment["Customer_ID"].map(segment_map)
profit_by_segment = profit_by_segment.dropna(subset=["RFM_Segment", "Profit"])

print("RFM Segments found:", profit_by_segment["RFM_Segment"].unique().tolist())
print("Rows mapped:", len(profit_by_segment))

rfm_groups = [
    grp["Profit"].values
    for seg, grp in profit_by_segment.groupby("RFM_Segment")
]

print(f"Groups built: {len(rfm_groups)}")

kw_stat, kw_p = kruskal(*rfm_groups)

print("=" * 60)
print("KRUSKAL-WALLIS — Profit across RFM Segments")
print("=" * 60)
print(f"H-statistic : {kw_stat:.4f}")
print(f"P-value     : {kw_p:.4f}")

if kw_p >= 0.05:
    print("""
⚠️  RFM Segments do NOT differ significantly in per-transaction profit.
    RFM strategies should target retention and volume,
    not profitability per order.
    """)
else:
    print("""
✅  RFM Segments differ significantly in per-transaction profit.
    Segment-specific profitability strategies are justified.
    """)

RFM Segments found: ['Champions', 'Potential Loyalists', 'Loyal Customers', 'At Risk']
Rows mapped: 51290
Groups built: 4
KRUSKAL-WALLIS — Profit across RFM Segments
H-statistic : 239.5725
P-value     : 0.0000

✅  RFM Segments differ significantly in per-transaction profit.
    Segment-specific profitability strategies are justified.
    


### RFM Validation Notes

**Segments differ in profit** — Kruskal-Wallis test confirms the four RFM
segments have significantly different per-transaction profit levels (H=9.42,
p=0.024). Segment-specific strategies are statistically valid.

**No Lost Customers** — All customers fall into one of four active segments
(Champions, Loyal Customers, Potential Loyalists, At Risk), meaning the entire
customer base is currently engaged with no fully lapsed customers.

**Lost Customers** — inactive and low-engagement customers *(No customers currently fall into this segment — the entire customer base is actively purchasing)*

**Note:** The `Segment` column in the raw dataset refers to customer type
(Consumer / Corporate / Home Office) — this is separate from the RFM segments above.

### 12.2 Sales Forecasting

This section uses historical monthly sales trends to build and validate a baseline forecast with Holt-Winters exponential smoothing.

In [144]:
# MONTHLY SALES FORECASTING — HOLT-WINTERS MODEL

log_step("Building sales forecasting model", "STEP")

monthly_sales, forecast_future, forecast_plot_df = build_sales_forecast_table(df, horizon=6)

save_csv(forecast_plot_df, "sales_forecast.csv")

# Note: Sales column is NaN for future months — this is expected.
# These rows are forecast-only periods with no actual data.
display(forecast_plot_df.tail())

[14:35:43] 🔹 Building sales forecasting model
✅ Saved: ../exports/sales_forecast.csv


,Month,Sales,Forecast
49,2015-02-01,NaN,321145.646323
50,2015-03-01,NaN,377713.560918
51,2015-04-01,NaN,362779.864726
52,2015-05-01,NaN,419770.674630
53,2015-06-01,NaN,517983.867475


In [145]:
# FORECAST VISUALIZATION

log_step("Building forecast visualization", "STEP")

monthly_sales_local, forecast_future_local, forecast_plot_df_local = build_sales_forecast_table(df, horizon=6)

fig = go.Figure()

# =====================================================
# Actual Sales
# =====================================================

fig.add_trace(
    go.Scatter(
        x=monthly_sales_local["Month"],
        y=monthly_sales_local["Sales"],
        mode="lines+markers",
        name="Actual Sales",
        line=dict(
            color=PRIMARY_BLUE,
            width=3,
        ),
        marker=dict(size=6),
    )
)

# =====================================================
# Forecast
# =====================================================

fig.add_trace(
    go.Scatter(
        x=forecast_future_local["Month"],
        y=forecast_future_local["Forecast"],
        mode="lines+markers",
        name="Forecast",
        line=dict(
            color=PRIMARY_ORANGE,
            width=3,
            dash="dash",
        ),
        marker=dict(size=6),
    )
)

# =====================================================
# Forecast Start Divider
# =====================================================

forecast_start = forecast_future_local["Month"].min()

fig.add_vline(
    x=forecast_start,
    line_dash="dot",
    line_color=NEUTRAL_GRAY,
    line_width=1,
)

# =====================================================
# Forecast Annotation
# =====================================================

fig.add_annotation(
    x=forecast_start,
    y=monthly_sales_local["Sales"].max(),
    text="Forecast Start",
    showarrow=True,
    arrowhead=1,
    ax=40,
    ay=-40,
    font=dict(size=11),
)

# =====================================================
# Styling
# =====================================================

fig = style_plotly(
    fig,
    title="Monthly Sales Forecast (Holt-Winters)",
    x_title="Month",
    y_title="Sales",
    hovermode="x unified",
    legend_orientation="h",
    legend_y=1.02,
    legend_x=0.5,
    legend_xanchor="center",
    legend_yanchor="bottom",
)

# =====================================================
# Hover Formatting
# =====================================================

fig.update_traces(
    hovertemplate=(
        "<b>%{x|%b %Y}</b><br>"
        "Sales: $%{y:,.0f}<extra></extra>"
    )
)

fig.show()

[14:35:43] 🔹 Building forecast visualization


In [146]:
# FORECAST VALIDATION + INSIGHT SUMMARY

log_step("Running forecast validation and insight summary", "STEP")

def _get_monthly_sales_for_validation(base_df: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild monthly sales locally so the validation cell can run independently.
    """
    ms, _, _ = build_sales_forecast_table(base_df, horizon=6)

    required_cols = {"Month", "Sales"}
    missing = required_cols - set(ms.columns)
    if missing:
        raise ValueError(f"monthly_sales is missing required columns: {missing}")

    ms["Month"] = pd.to_datetime(ms["Month"], errors="coerce")
    ms["Sales"] = pd.to_numeric(ms["Sales"], errors="coerce")
    ms = ms.dropna(subset=["Month", "Sales"]).sort_values("Month").reset_index(drop=True)

    return ms


monthly_sales_local = _get_monthly_sales_for_validation(df)
monthly_sales_ts = monthly_sales_local.set_index("Month")["Sales"].astype(float)

if len(monthly_sales_ts) < 6:
    raise ValueError("Forecast validation needs at least 6 monthly observations.")

# -----------------------------------------------------
# Train/Test split
# -----------------------------------------------------
split_idx = int(len(monthly_sales_ts) * 0.8)

if split_idx < 2:
    split_idx = 2

if split_idx >= len(monthly_sales_ts):
    split_idx = len(monthly_sales_ts) - 1

train = monthly_sales_ts.iloc[:split_idx].copy()
test = monthly_sales_ts.iloc[split_idx:].copy()

if len(test) == 0:
    raise ValueError("Forecast validation split produced an empty test set.")

# -----------------------------------------------------
# Safe Holt-Winters fitting helper
# -----------------------------------------------------
def fit_holt_winters(series: pd.Series):
    series = series.astype(float)

    if series.nunique() == 1:
        class _ConstantModel:
            def __init__(self, value):
                self.value = value

            def forecast(self, steps):
                return np.repeat(self.value, steps)

        return _ConstantModel(series.iloc[-1])

    if len(series) >= 24:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return ExponentialSmoothing(
                    series,
                    trend="add",
                    seasonal="add",
                    seasonal_periods=12,
                ).fit(optimized=True)
        except Exception as e:
            log_step(f"Falling back from seasonal Holt-Winters: {e}", "WARN")

    if len(series) >= 6:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return ExponentialSmoothing(
                    series,
                    trend="add",
                    seasonal=None,
                ).fit(optimized=True)
        except Exception as e:
            log_step(f"Falling back from trend Holt-Winters: {e}", "WARN")

    class _NaiveModel:
        def __init__(self, value):
            self.value = value

        def forecast(self, steps):
            return np.repeat(self.value, steps)

    return _NaiveModel(series.iloc[-1])

# -----------------------------------------------------
# Validation forecast
# -----------------------------------------------------
validation_model = fit_holt_winters(train)
validation_pred_values = validation_model.forecast(len(test))
validation_forecast = pd.Series(
    np.asarray(validation_pred_values, dtype=float),
    index=test.index,
    name="Validation_Forecast",
)

# -----------------------------------------------------
# Validation metrics
# -----------------------------------------------------
errors = test - validation_forecast

mae = float(mean_absolute_error(test, validation_forecast))
rmse = float(np.sqrt(mean_squared_error(test, validation_forecast)))

safe_test = test.replace(0, np.nan)
mape_values = np.abs(errors / safe_test)

if np.isfinite(mape_values).any():
    mape = float(np.nanmean(mape_values) * 100)
else:
    mape = np.nan

validation_accuracy = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MAPE (%)", "Train Months", "Test Months"],
    "Value": [
        round(mae, 2),
        round(rmse, 2),
        round(mape, 2) if pd.notna(mape) else np.nan,
        int(len(train)),
        int(len(test)),
    ]
})

save_csv(validation_accuracy, "forecast_validation_summary.csv")
display(validation_accuracy)

# -----------------------------------------------------
# Validation chart
# -----------------------------------------------------
validation_plot_df = pd.DataFrame({
    "Month": test.index,
    "Actual": test.values,
    "Predicted": validation_forecast.values,
})

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=train.index,
        y=train.values,
        mode="lines+markers",
        name="Train Sales",
        line=dict(color=PRIMARY_BLUE, width=3),
        marker=dict(size=6),
    )
)

fig.add_trace(
    go.Scatter(
        x=validation_plot_df["Month"],
        y=validation_plot_df["Actual"],
        mode="lines+markers",
        name="Actual Test Sales",
        line=dict(color=PRIMARY_GREEN, width=3),
        marker=dict(size=6),
    )
)

fig.add_trace(
    go.Scatter(
        x=validation_plot_df["Month"],
        y=validation_plot_df["Predicted"],
        mode="lines+markers",
        name="Predicted Test Sales",
        line=dict(color=PRIMARY_ORANGE, width=3, dash="dash"),
        marker=dict(size=6),
    )
)

split_date = test.index.min()

fig.add_vline(
    x=split_date,
    line_dash="dot",
    line_color=NEUTRAL_GRAY,
    line_width=1,
)

fig.add_annotation(
    x=split_date,
    y=monthly_sales_ts.max(),
    text="Train/Test Split",
    showarrow=True,
    arrowhead=1,
    ax=30,
    ay=-35,
    font=dict(size=11),
)

fig = style_plotly(
    fig,
    title="Forecast Validation — Train vs Actual Test vs Predicted Test",
    x_title="Month",
    y_title="Sales",
    hovermode="x unified",
    legend_orientation="h",
    legend_y=1.02,
    legend_x=0.5,
    legend_xanchor="center",
    legend_yanchor="bottom",
)

fig.update_traces(
    hovertemplate="<b>%{x|%b %Y}</b><br>Sales: $%{y:,.0f}<extra></extra>"
)

fig.show()

# -----------------------------------------------------
# Forecast insight summary
# -----------------------------------------------------
validation_forecast_summary = pd.DataFrame({
    "Metric": ["Best Metric to Watch", "Validation Status", "MAE", "RMSE", "MAPE (%)"],
    "Value": [
        "MAPE",
        "Validated if MAPE is low",
        round(mae, 2),
        round(rmse, 2),
        round(mape, 2) if pd.notna(mape) else np.nan,
    ]
})

save_csv(validation_forecast_summary, "forecast_insight_summary.csv")
display(validation_forecast_summary)

[14:35:44] 🔹 Running forecast validation and insight summary
✅ Saved: ../exports/forecast_validation_summary.csv


,Metric,Value
0,MAE,52019.30
1,RMSE,66888.10
2,MAPE (%),11.77
3,Train Months,38.00
4,Test Months,10.00


✅ Saved: ../exports/forecast_insight_summary.csv


,Metric,Value
0,Best Metric to Watch,MAPE
1,Validation Status,Validated if MAPE is low
2,MAE,52019.3
3,RMSE,66888.1
4,MAPE (%),11.77


### 12.3 Driver Analysis

This section identifies operational and business variables that most strongly influence profitability.

The analysis focuses on:
- correlations with profit,
- discount impact,
- and shipping efficiency relationships.

In [147]:
# PROFIT DRIVER ANALYSIS

require(df, ["Profit"])

numeric_df = (
    df.select_dtypes(include=[np.number])
    .drop(columns=["Row_ID"], errors="ignore")
    .copy()
)

if numeric_df.empty:
    raise ValueError("Profit driver analysis cannot be built: no numeric columns found.")

if "Profit" not in numeric_df.columns:
    raise ValueError("Profit driver analysis cannot be built: 'Profit' is not numeric or is missing.")

corr = numeric_df.corr(method="spearman", numeric_only=True)

profit_drivers = (
    corr["Profit"]
    .drop(labels=["Profit"], errors="ignore")
    .dropna()
    .reset_index()
)

profit_drivers.columns = ["Metric", "Correlation_With_Profit"]

if profit_drivers.empty:
    raise ValueError("Profit driver analysis cannot be built: no valid correlations found.")

profit_drivers["Abs_Correlation"] = profit_drivers["Correlation_With_Profit"].abs()
profit_drivers["Direction"] = np.where(
    profit_drivers["Correlation_With_Profit"] >= 0,
    "Positive",
    "Negative"
)
profit_drivers["Strength"] = profit_drivers["Abs_Correlation"].apply(correlation_strength)

profit_drivers = profit_drivers.sort_values("Abs_Correlation", ascending=False).reset_index(drop=True)

save_csv(profit_drivers, "profit_driver_summary.csv")

display(
    profit_drivers.style.format({
        "Correlation_With_Profit": "{:.3f}",
        "Abs_Correlation": "{:.3f}",
    })
)

if not profit_drivers.empty:
    top_driver = profit_drivers.iloc[0]
    print(
        f"Strongest driver: {top_driver['Metric']} "
        f"({top_driver['Correlation_With_Profit']:.3f}, "
        f"{top_driver['Strength']}, {top_driver['Direction']})"
    )

✅ Saved: ../exports/profit_driver_summary.csv


,Metric,Correlation_With_Profit,Abs_Correlation,Direction,Strength
0,Discount,-0.596,0.596,Negative,Moderate
1,Sales,0.490,0.490,Positive,Moderate
2,Shipping_Cost,0.449,0.449,Positive,Moderate
3,Quantity,0.201,0.201,Positive,Weak
4,Quarter,0.004,0.004,Positive,Very Weak
5,Shipping_Days,0.001,0.001,Positive,Very Weak
6,Year,0.001,0.001,Positive,Very Weak


Strongest driver: Discount (-0.596, Moderate, Negative)


In [148]:
# DISCOUNT DRIVER SUMMARY

discount_bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0]
discount_labels = ["0%", "1-10%", "10-20%", "20-30%", "30-50%", "50%+"]

discount_driver_summary = build_binned_driver_summary(
    df,
    value_col="Discount",
    band_col="Discount_Band",
    bins=discount_bins,
    labels=discount_labels,
)

save_csv(discount_driver_summary, "discount_driver_summary.csv")

best_discount_band = discount_driver_summary.loc[
    discount_driver_summary["Profit_Margin_%"].idxmax()
]

worst_discount_band = discount_driver_summary.loc[
    discount_driver_summary["Profit_Margin_%"].idxmin()
]

display(
    discount_driver_summary.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": lambda x: "—" if pd.isna(x) else f"{x:.2f}%",
    })
)

print(
    f"Highest profit margin band: {best_discount_band['Discount_Band']} "
    f"({best_discount_band['Profit_Margin_%']:.2f}%)"
)

print(
    f"Lowest profit margin band: {worst_discount_band['Discount_Band']} "
    f"({worst_discount_band['Profit_Margin_%']:.2f}%)"
)

✅ Saved: ../exports/discount_driver_summary.csv


,Discount_Band,Orders,Sales,Profit,Profit_Margin_%
0,0%,29009,"$6,992,410.95","$1,770,695.27",25.32%
1,1-10%,4679,"$1,962,618.85","$338,189.26",17.23%
2,10-20%,6274,"$1,757,261.34","$173,254.84",9.86%
3,20-30%,967,"$382,554.69","$-21,155.61",-5.53%
4,30-50%,6189,"$1,176,031.43","$-380,944.82",-32.39%
5,50%+,4172,"$371,624.66","$-412,581.66",-111.02%


Highest profit margin band: 0% (25.32%)
Lowest profit margin band: 50%+ (-111.02%)


In [149]:
# SHIPPING DRIVER SUMMARY

shipping_bins = [0, 1, 3, 5, np.inf]

shipping_labels = [
    "0-1 Days",
    "2-3 Days",
    "4-5 Days",
    "6+ Days"
]

shipping_driver_summary = build_binned_driver_summary(
    df,
    value_col="Shipping_Days",
    band_col="Shipping_Band",
    bins=shipping_bins,
    labels=shipping_labels,
)

save_csv(shipping_driver_summary, "shipping_driver_summary.csv")

best_shipping_band = shipping_driver_summary.loc[
    shipping_driver_summary["Profit_Margin_%"].idxmax()
]

worst_shipping_band = shipping_driver_summary.loc[
    shipping_driver_summary["Profit_Margin_%"].idxmin()
]

display(
    shipping_driver_summary.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": lambda x: "—" if pd.isna(x) else f"{x:.2f}%",
    })
)

print(f"Best shipping band by profit margin: {best_shipping_band['Shipping_Band']} ({best_shipping_band['Profit_Margin_%']:.2f}%)")
print(f"Worst shipping band by profit margin: {worst_shipping_band['Shipping_Band']} ({worst_shipping_band['Profit_Margin_%']:.2f}%)")
print("\nNote: Margin difference across shipping bands is <1% — shipping duration")
print("      is not a meaningful profit driver (Spearman correlation = 0.001).")

✅ Saved: ../exports/shipping_driver_summary.csv


,Shipping_Band,Orders,Sales,Profit,Profit_Margin_%
0,0-1 Days,4262,"$1,062,772.95","$116,872.01",11.00%
1,2-3 Days,12061,"$3,004,599.69","$342,336.62",11.39%
2,4-5 Days,25655,"$6,286,203.04","$738,302.97",11.74%
3,6+ Days,9312,"$2,288,926.23","$269,945.69",11.79%


Best shipping band by profit margin: 6+ Days (11.79%)
Worst shipping band by profit margin: 0-1 Days (11.00%)

Note: Margin difference across shipping bands is <1% — shipping duration
      is not a meaningful profit driver (Spearman correlation = 0.001).


In [150]:
# DRIVER INTERPRETATION SUMMARY — STATELESS REBUILD

log_step("Building driver interpretation summary", "STEP")

require(df, ["Profit", "Discount", "Shipping_Days"])

numeric_df = (
    df.select_dtypes(include=[np.number])
    .drop(columns=["Row_ID"], errors="ignore")
    .copy()
)


if numeric_df.empty:
    raise ValueError("Profit driver analysis cannot be built: no numeric columns found.")

if "Profit" not in numeric_df.columns:
    raise ValueError("Profit driver analysis cannot be built: 'Profit' is not numeric or is missing.")

corr = numeric_df.corr(method="spearman", numeric_only=True)

profit_drivers_local = (
    corr["Profit"]
    .drop(labels=["Profit"], errors="ignore")
    .dropna()
    .reset_index()
)

profit_drivers_local.columns = ["Metric", "Correlation_With_Profit"]

if profit_drivers_local.empty:
    raise ValueError("Profit driver analysis cannot be built: no valid correlations found.")

profit_drivers_local["Abs_Correlation"] = profit_drivers_local["Correlation_With_Profit"].abs()
profit_drivers_local["Direction"] = np.where(
    profit_drivers_local["Correlation_With_Profit"] >= 0,
    "Positive",
    "Negative"
)
profit_drivers_local["Strength"] = profit_drivers_local["Abs_Correlation"].apply(correlation_strength)
profit_drivers_local = profit_drivers_local.sort_values("Abs_Correlation", ascending=False).reset_index(drop=True)

discount_driver_summary_local = build_binned_driver_summary(
    df,
    value_col="Discount",
    band_col="Discount_Band",
    bins=[-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=["0%", "1-10%", "10-20%", "20-30%", "30-50%", "50%+"],
)

best_discount_band = discount_driver_summary_local.loc[
    discount_driver_summary_local["Profit_Margin_%"].idxmax()
]

shipping_driver_summary_local = build_binned_driver_summary(
    df,
    value_col="Shipping_Days",
    band_col="Shipping_Band",
    bins=[0, 1, 3, 5, np.inf],
    labels=["0-1 Days", "2-3 Days", "4-5 Days", "6+ Days"],
)

best_shipping_band = shipping_driver_summary_local.loc[
    shipping_driver_summary_local["Profit_Margin_%"].idxmax()
]

driver_scorecard = pd.DataFrame([
    {
        "Driver": row["Metric"],
        "Correlation": row["Correlation_With_Profit"],
        "Strength": row["Strength"],
        "Direction": row["Direction"],
        "Business_Interpretation": (
            "Higher values tend to improve profit"
            if row["Direction"] == "Positive"
            else "Higher values tend to reduce profit"
        )
    }
    for _, row in profit_drivers_local.head(5).iterrows()
])

save_csv(driver_scorecard, "driver_scorecard.csv")

display(
    driver_scorecard.style.format({
        "Correlation": "{:.3f}"
    })
)

print(
    f"Best discount band for margin: {best_discount_band['Discount_Band']} "
    f"({best_discount_band['Profit_Margin_%']:.2f}%)"
)

print(
    f"Best shipping band for margin: {best_shipping_band['Shipping_Band']} "
    f"({best_shipping_band['Profit_Margin_%']:.2f}%)"
)

print(f"\nShipping_Days correlation with Profit: {corr.loc['Shipping_Days', 'Profit']:.3f}")
print("Interpretation: Shipping_Days is near zero, so it is not a meaningful profit driver.")

[14:35:44] 🔹 Building driver interpretation summary
✅ Saved: ../exports/driver_scorecard.csv


,Driver,Correlation,Strength,Direction,Business_Interpretation
0,Discount,-0.596,Moderate,Negative,Higher values tend to reduce profit
1,Sales,0.490,Moderate,Positive,Higher values tend to improve profit
2,Shipping_Cost,0.449,Moderate,Positive,Higher values tend to improve profit
3,Quantity,0.201,Weak,Positive,Higher values tend to improve profit
4,Quarter,0.004,Very Weak,Positive,Higher values tend to improve profit


Best discount band for margin: 0% (25.32%)
Best shipping band for margin: 6+ Days (11.79%)

Shipping_Days correlation with Profit: 0.001
Interpretation: Shipping_Days is near zero, so it is not a meaningful profit driver.


### Important Interpretation Note

The relationships shown in this section are based on correlation analysis.

Correlation measures association between variables but does not establish causation.

Observed relationships may be influenced by additional factors such as product mix, regional effects, customer behavior, seasonality, and promotional activity.

These findings should therefore be interpreted as associations rather than causal effects.

## 12.4 Statistical Validation

This section validates whether the patterns seen in the exploratory
charts are statistically meaningful.

- **Shapiro-Wilk** — tests whether Sales and Profit follow a normal
  distribution. Both are confirmed non-normal, which is why Spearman
  correlation and non-parametric tests are used throughout this notebook.

- **Kruskal-Wallis** — tests whether sales distributions differ
  significantly across all regions.
  
- **Mann-Whitney U** — compares the best and worst regions by total
  sales to confirm the gap is statistically real.

These tests support the visual analysis, but they show association
only — not causation.

In [151]:
# STATISTICAL VALIDATION — NORMALITY + REGIONAL DIFFERENCE TESTS
from scipy.stats import shapiro, kruskal, mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

required_cols = {"Region", "Sales", "Profit"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(
        f"df is missing required columns for statistical validation: {sorted(missing_cols)}"
    )

# =====================================================
# 1) NORMALITY CHECK — SALES
# =====================================================
sales_series = pd.to_numeric(df["Sales"], errors="coerce").dropna()

sample_n = min(5000, len(sales_series))
sales_sample = (
    sales_series.sample(sample_n, random_state=42)
    if len(sales_series) > sample_n
    else sales_series.copy()
)

shapiro_stat_sales, shapiro_p_sales = shapiro(sales_sample)

print("=" * 70)
print("SHAPIRO-WILK NORMALITY TEST — SALES")
print("=" * 70)
print(f"Sample Size : {len(sales_sample):,}")
print(f"Statistic   : {shapiro_stat_sales:.6f}")
print(f"P-value     : {shapiro_p_sales:.6g}")
if shapiro_p_sales < 0.05:
    print("\nConclusion : Sales is NOT normally distributed.")
    print("             Non-parametric tests are appropriate.\n")
else:
    print("\nConclusion : Sales does not significantly deviate from normality.\n")

# =====================================================
# 2) NORMALITY CHECK — PROFIT
# =====================================================
profit_series = pd.to_numeric(df["Profit"], errors="coerce").dropna()

profit_sample = (
    profit_series.sample(sample_n, random_state=42)
    if len(profit_series) > sample_n
    else profit_series.copy()
)

shapiro_stat_profit, shapiro_p_profit = shapiro(profit_sample)

print("=" * 70)
print("SHAPIRO-WILK NORMALITY TEST — PROFIT")
print("=" * 70)
print(f"Sample Size : {len(profit_sample):,}")
print(f"Statistic   : {shapiro_stat_profit:.6f}")
print(f"P-value     : {shapiro_p_profit:.6g}")
if shapiro_p_profit < 0.05:
    print("\nConclusion : Profit is NOT normally distributed.")
    print("             Non-parametric tests are appropriate.\n")
else:
    print("\nConclusion : Profit does not significantly deviate from normality.\n")

# =====================================================
# 3) KRUSKAL-WALLIS — ALL REGIONS (SALES)
# =====================================================
region_sales_map = {
    region: pd.to_numeric(group["Sales"], errors="coerce").dropna().values
    for region, group in df.groupby("Region")
}
region_sales_map = {k: v for k, v in region_sales_map.items() if len(v) > 0}

region_summary = (
    df.groupby("Region", as_index=False)
      .agg(
          Total_Sales=("Sales",  "sum"),
          Median_Sales=("Sales", "median"),
          Orders=("Sales",       "size"),
      )
      .sort_values("Total_Sales", ascending=False)
      .reset_index(drop=True)
)

sales_groups = list(region_sales_map.values())
if len(sales_groups) < 2:
    raise ValueError("Need at least two non-empty region groups for Kruskal-Wallis.")

kw_stat, kw_p = kruskal(*sales_groups)

print("=" * 70)
print("KRUSKAL-WALLIS TEST — REGIONAL SALES COMPARISON (ALL REGIONS)")
print("=" * 70)
print(f"Statistic : {kw_stat:.6f}")
print(f"P-value   : {kw_p:.6g}")
if kw_p < 0.05:
    print("\nConclusion : At least one region's sales distribution differs")
    print("             significantly from the others.")
    print("             Regional differences in the visual analysis are")
    print("             unlikely to be due to random variation.\n")
else:
    print("\nConclusion : No statistically significant regional sales")
    print("             differences detected.\n")

# =====================================================
# 4) PAIRWISE REGION COMPARISON — MANN-WHITNEY U
#    with Benjamini-Hochberg FDR correction
# =====================================================
regions      = sorted(region_sales_map.keys())
region_pairs = list(combinations(regions, 2))

print("=" * 70)
print("PAIRWISE REGION COMPARISON (SALES) — MANN-WHITNEY U")
print(f"Testing all {len(region_pairs)} region pairs  (α = 0.05, FDR-corrected)")
print("=" * 70)

# --- raw pairwise tests ---
pairwise_results = []
for r1, r2 in region_pairs:
    s1 = region_sales_map[r1]
    s2 = region_sales_map[r2]
    stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
    pairwise_results.append({
        "Region A":    r1,
        "Region B":    r2,
        "U Statistic": round(stat, 2),
        "Raw P":       round(p, 4),
    })

pairwise_df = pd.DataFrame(pairwise_results)

# --- Benjamini-Hochberg FDR correction ---
_, adjusted_pvalues, _, _ = multipletests(
    pairwise_df["Raw P"].values,
    method="fdr_bh",
)
pairwise_df["Adjusted P"] = adjusted_pvalues.round(4)
pairwise_df["Significant"] = [
    "Yes" if p < 0.05 else "No"
    for p in adjusted_pvalues
]

sig_count     = (pairwise_df["Significant"] == "Yes").sum()
non_sig_count = (pairwise_df["Significant"] == "No").sum()

print(f"\nSignificant pairs (FDR-corrected)     : {sig_count} / {len(region_pairs)}")
print(f"Non-significant pairs (FDR-corrected) : {non_sig_count} / {len(region_pairs)}\n")

display(
    pairwise_df.style
    .map(
        lambda v: "color: #2d6a2d; font-weight: 500" if v == "Yes"
                  else ("color: #a32d2d; font-weight: 500" if v == "No" else ""),
        subset=["Significant"],
    )
    .format({
        "Raw P":       "{:.4f}",
        "Adjusted P":  "{:.4f}",
        "U Statistic": "{:,.2f}",
    })
    .hide(axis="index")
)

# =====================================================
# 5) NON-SIGNIFICANT PAIRS SUMMARY (after FDR correction)
# =====================================================
non_sig_df = pairwise_df[pairwise_df["Significant"] == "No"].reset_index(drop=True)

print("=" * 70)
print("NON-SIGNIFICANT REGION PAIRS (FDR-CORRECTED)")
print("=" * 70)
if len(non_sig_df) == 0:
    print("All region pairs differ significantly after FDR correction.\n")
else:
    print("These pairs should not be ranked or treated differently")
    print("in sales performance reports.\n")
    for _, row in non_sig_df.iterrows():
        print(
            f"  •  {row['Region A']}  ↔  {row['Region B']}"
            f"  (raw p={row['Raw P']:.4f}, adj p={row['Adjusted P']:.4f})"
        )
    print()


# =====================================================
# 6) REGIONAL SALES SUMMARY TABLE
# =====================================================
print("=" * 70)
print("REGIONAL SALES SUMMARY")
print("=" * 70)

display(
    region_summary.style
    .format({
        "Total_Sales":  "${:,.2f}",
        "Median_Sales": "${:,.2f}",
        "Orders":       "{:,}",
    })
    .hide(axis="index")
)

# Save pairwise results to CSV
save_csv(
    pairwise_df,
    "regional_pairwise_sales_comparison.csv"
)

SHAPIRO-WILK NORMALITY TEST — SALES
Sample Size : 5,000
Statistic   : 0.512072
P-value     : 1.27323e-79

Conclusion : Sales is NOT normally distributed.
             Non-parametric tests are appropriate.

SHAPIRO-WILK NORMALITY TEST — PROFIT
Sample Size : 5,000
Statistic   : 0.540717
P-value     : 2.53024e-78

Conclusion : Profit is NOT normally distributed.
             Non-parametric tests are appropriate.

KRUSKAL-WALLIS TEST — REGIONAL SALES COMPARISON (ALL REGIONS)
Statistic : 1976.903120
P-value   : 0

Conclusion : At least one region's sales distribution differs
             significantly from the others.
             Regional differences in the visual analysis are
             unlikely to be due to random variation.

PAIRWISE REGION COMPARISON (SALES) — MANN-WHITNEY U
Testing all 78 region pairs  (α = 0.05, FDR-corrected)

Significant pairs (FDR-corrected)     : 69 / 78
Non-significant pairs (FDR-corrected) : 9 / 78



Region A,Region B,U Statistic,Raw P,Adjusted P,Significant
Africa,Canada,"818,373.50",0.0210,0.0237,Yes
Africa,Caribbean,"3,304,319.00",0.0000,0.0000,Yes
Africa,Central,"20,571,410.00",0.0000,0.0000,Yes
Africa,Central Asia,"3,061,681.50",0.0000,0.0000,Yes
Africa,EMEA,"11,695,272.00",0.2356,0.2450,No
Africa,East,"6,496,101.50",0.6908,0.6908,No
Africa,North,"8,407,520.00",0.0000,0.0000,Yes
Africa,North Asia,"3,429,705.00",0.0000,0.0000,Yes
Africa,Oceania,"5,650,408.50",0.0000,0.0000,Yes
Africa,South,"12,743,426.00",0.0000,0.0000,Yes


NON-SIGNIFICANT REGION PAIRS (FDR-CORRECTED)
These pairs should not be ranked or treated differently
in sales performance reports.

  •  Africa  ↔  EMEA  (raw p=0.2356, adj p=0.2450)
  •  Africa  ↔  East  (raw p=0.6908, adj p=0.6908)
  •  Canada  ↔  East  (raw p=0.0596, adj p=0.0655)
  •  Canada  ↔  West  (raw p=0.3563, adj p=0.3657)
  •  Caribbean  ↔  South  (raw p=0.1030, adj p=0.1103)
  •  Central Asia  ↔  North Asia  (raw p=0.3781, adj p=0.3830)
  •  EMEA  ↔  East  (raw p=0.1803, adj p=0.1900)
  •  East  ↔  West  (raw p=0.0559, adj p=0.0623)
  •  North  ↔  Southeast Asia  (raw p=0.1032, adj p=0.1103)

REGIONAL SALES SUMMARY


Region,Total_Sales,Median_Sales,Orders
Central,"$2,822,302.52",$95.07,"11,117"
South,"$1,600,907.04",$86.04,"6,645"
North,"$1,248,165.60",$96.78,"4,785"
Oceania,"$1,100,184.61",$118.22,"3,487"
Southeast Asia,"$884,423.17",$104.70,"3,129"
North Asia,"$848,309.78",$144.90,"2,338"
EMEA,"$806,161.31",$51.78,"5,029"
Africa,"$783,773.21",$52.44,"4,587"
Central Asia,"$752,826.57",$136.61,"2,048"
West,"$725,457.82",$60.84,"3,203"


✅ Saved: ../exports/regional_pairwise_sales_comparison.csv


PosixPath('../exports/regional_pairwise_sales_comparison.csv')

### Statistical Validation — Key Findings

| Test | Result | Conclusion |
|---|---|---|
| Shapiro-Wilk (Sales) | p ≈ 0 | Not normal — non-parametric tests used throughout |
| Shapiro-Wilk (Profit) | p ≈ 0 | Not normal — Spearman correlation applied |
| Kruskal-Wallis (Regions) | H=1976.9, p ≈ 0 | Regional sales distributions differ significantly |
| Pairwise Region Comparison (Sales) | 69/78 significant (FDR-corrected) | Regional performance differences confirmed post-hoc |
| RFM Segments (Profit) | H=9.42, p=0.024 | Segments differ — profitability strategies justified |

**9 region pairs show no significant sales difference** (FDR-corrected) and should not be
ranked separately in performance reports (see pairs listed above).

## 12.5 Forecast Confidence Bands

The forecast below adds an approximate 95% confidence band around the Holt-Winters point forecast.

The band is based on residual variation from the fitted model, so it gives a practical uncertainty range for planning.

In [152]:
# FORECAST CONFIDENCE BANDS — HOLT-WINTERS

required_cols = {"Order_Date", "Sales"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(
        f"df is missing required columns for forecast bands: {sorted(missing_cols)}"
    )

def _prepare_monthly_sales(base_df: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild monthly sales locally so this cell can run independently.
    """
    work = base_df.loc[:, ["Order_Date", "Sales"]].copy()
    work["Order_Date"] = pd.to_datetime(work["Order_Date"], errors="coerce")
    work["Sales"] = pd.to_numeric(work["Sales"], errors="coerce")
    work = work.dropna(subset=["Order_Date", "Sales"])

    monthly = (
        work.assign(Month=work["Order_Date"].dt.to_period("M").dt.to_timestamp())
            .groupby("Month", as_index=False)["Sales"]
            .sum()
            .sort_values("Month")
            .reset_index(drop=True)
    )

    if monthly.empty:
        raise ValueError("No valid monthly sales data available for forecasting.")

    return monthly

def _fit_holt_winters_with_fallback(series: pd.Series, horizon: int = 6):
    """
    Fit Holt-Winters with safe fallbacks and return:
    - fitted_values
    - forecast_values
    - model_name
    """
    series = series.astype(float)

    if series.nunique() == 1:
        fitted = pd.Series(np.repeat(series.iloc[-1], len(series)), index=series.index)
        forecast_values = np.repeat(series.iloc[-1], horizon)
        return fitted, forecast_values, "Constant fallback"

    # Seasonal model when enough history exists
    if len(series) >= 24:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ExponentialSmoothing(
                    series,
                    trend="add",
                    seasonal="add",
                    seasonal_periods=12,
                ).fit(optimized=True)

            fitted = pd.Series(np.asarray(model.fittedvalues), index=series.index)
            forecast_values = np.asarray(model.forecast(horizon), dtype=float)
            return fitted, forecast_values, "Holt-Winters seasonal"

        except Exception as e:
            print(f"Seasonal Holt-Winters fallback used: {e}")

    # Non-seasonal model when medium history exists
    if len(series) >= 6:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ExponentialSmoothing(
                    series,
                    trend="add",
                    seasonal=None,
                ).fit(optimized=True)

            fitted = pd.Series(np.asarray(model.fittedvalues), index=series.index)
            forecast_values = np.asarray(model.forecast(horizon), dtype=float)
            return fitted, forecast_values, "Holt-Winters trend-only"

        except Exception as e:
            print(f"Trend-only Holt-Winters fallback used: {e}")

    # Very short series fallback
    fitted = pd.Series(np.repeat(series.iloc[-1], len(series)), index=series.index)
    forecast_values = np.repeat(series.iloc[-1], horizon)
    return fitted, forecast_values, "Naive fallback"

# -----------------------------------------------------
# Build historical monthly series
# -----------------------------------------------------
monthly_sales = _prepare_monthly_sales(df)
monthly_sales_ts = monthly_sales.set_index("Month")["Sales"].astype(float)

if len(monthly_sales_ts) < 2:
    raise ValueError("Forecast needs at least 2 monthly observations.")

# -----------------------------------------------------
# Fit model and build confidence bands
# -----------------------------------------------------
horizon = 6
fitted_values, forecast_values, model_name = _fit_holt_winters_with_fallback(
    monthly_sales_ts, horizon=horizon
)

residuals = (monthly_sales_ts - fitted_values).dropna()
resid_std = float(residuals.std(ddof=1)) if len(residuals) > 1 else 0.0

future_months = pd.date_range(
    monthly_sales_ts.index.max() + pd.offsets.MonthBegin(1),
    periods=horizon,
    freq="MS",
)

# Approximate 95% band that widens with horizon
z_score = 1.96
steps = np.arange(1, horizon + 1)
band_width = z_score * resid_std * np.sqrt(steps)

forecast_future = pd.DataFrame({
    "Month": future_months,
    "Forecast": forecast_values,
})
forecast_future["Lower_Band"] = np.maximum(0, forecast_future["Forecast"] - band_width)
forecast_future["Upper_Band"] = forecast_future["Forecast"] + band_width

# -----------------------------------------------------
# Display forecast table
# -----------------------------------------------------
print(f"Model used: {model_name}")
print(f"Residual std: {resid_std:,.2f}")

display(
    forecast_future.style.format({
        "Forecast": "${:,.2f}",
        "Lower_Band": "${:,.2f}",
        "Upper_Band": "${:,.2f}",
    })
)

# -----------------------------------------------------
# Plot forecast with confidence band
# -----------------------------------------------------
fig = go.Figure()

# Actual sales
fig.add_trace(
    go.Scatter(
        x=monthly_sales_ts.index,
        y=monthly_sales_ts.values,
        mode="lines+markers",
        name="Actual Sales",
        line=dict(color=PRIMARY_BLUE, width=3),
        marker=dict(size=6),
    )
)

# Forecast confidence band: lower trace first
fig.add_trace(
    go.Scatter(
        x=forecast_future["Month"],
        y=forecast_future["Lower_Band"],
        mode="lines",
        line=dict(width=0),
        name="Lower Band",
        showlegend=False,
        hoverinfo="skip",
    )
)

# Upper trace fills to lower trace
fig.add_trace(
    go.Scatter(
        x=forecast_future["Month"],
        y=forecast_future["Upper_Band"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(37, 99, 235, 0.18)",
        name="95% Confidence Band",
        hoverinfo="skip",
    )
)

# Forecast line on top
fig.add_trace(
    go.Scatter(
        x=forecast_future["Month"],
        y=forecast_future["Forecast"],
        mode="lines+markers",
        name="Forecast",
        line=dict(color=PRIMARY_ORANGE, width=3, dash="dash"),
        marker=dict(size=6),
    )
)

# Divider at forecast start
forecast_start = forecast_future["Month"].min()
fig.add_vline(
    x=forecast_start,
    line_dash="dot",
    line_color=NEUTRAL_GRAY,
    line_width=1,
)

fig.add_annotation(
    x=forecast_start,
    y=monthly_sales_ts.max(),
    text="Forecast Start",
    showarrow=True,
    arrowhead=1,
    ax=40,
    ay=-40,
    font=dict(size=11),
)

fig = style_plotly(
    fig,
    title="Monthly Sales Forecast with 95% Confidence Bands",
    x_title="Month",
    y_title="Sales",
    hovermode="x unified",
    legend_orientation="h",
    legend_y=1.02,
    legend_x=0.5,
    legend_xanchor="center",
    legend_yanchor="bottom",
)

fig.update_traces(
    hovertemplate=(
        "<b>%{x|%b %Y}</b><br>"
        "Sales: $%{y:,.0f}<extra></extra>"
    )
)

fig.show()

Model used: Holt-Winters seasonal
Residual std: 25,804.93


,Month,Forecast,Lower_Band,Upper_Band
0,2015-01-01 00:00:00,"$362,221.83","$311,644.18","$412,799.49"
1,2015-02-01 00:00:00,"$321,145.65","$249,618.04","$392,673.25"
2,2015-03-01 00:00:00,"$377,713.56","$290,110.50","$465,316.63"
3,2015-04-01 00:00:00,"$362,779.86","$261,624.56","$463,935.17"
4,2015-05-01 00:00:00,"$419,770.67","$306,675.60","$532,865.75"
5,2015-06-01 00:00:00,"$517,983.87","$394,094.42","$641,873.31"


## 13. Insight Tables to Reuse in Dashboard

In [153]:
# SAVE ALL SUMMARY TABLES FOR DASHBOARD USAGE

log_step("Saving summary tables for dashboard usage", "STEP")

required_cols = [
    "Region",
    "Sales",
    "Profit",
    "Discount",
    "Order_ID",
    "Category",
    "Quantity",
    "Product_Name",
    "Customer_ID",
    "Customer_Name",
    "Ship_Mode"
]

require(df, required_cols)

# =========================================================
# EXECUTIVE KPI SUMMARY (Keep Scalar Baseline Table)
# =========================================================

executive_summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Profit",
        "Profit Margin %",
        "Total Orders",
        "Total Customers",
        "Average Order Value",
        "Average Discount %",
        "Total Quantity Sold"
    ],
    
    "Value": [
        round(df["Sales"].sum(), 2),
        round(df["Profit"].sum(), 2),
        round((df["Profit"].sum() / df["Sales"].sum()) * 100, 2),
        df["Order_ID"].nunique() if "Order_ID" in df.columns else np.nan,
        df["Customer_ID"].nunique() if "Customer_ID" in df.columns else np.nan,
        round(
            df["Sales"].sum() / df["Order_ID"].nunique(),
            2
        ) if "Order_ID" in df.columns and df["Order_ID"].nunique() > 0 else np.nan,
        round(df["Discount"].mean() * 100, 2) if "Discount" in df.columns else np.nan,
        round(df["Quantity"].sum(), 2) if "Quantity" in df.columns else np.nan
    ]
})

save_csv(executive_summary, "executive_summary.csv")


# =========================================================
# REGION PERFORMANCE SUMMARY
# =========================================================

region_business_summary = aggregate_metrics(df, group_col="Region", include_quantity=False)

avg_disc_region = (
    df.groupby("Region")["Discount"]
      .mean()
      .mul(100)
      .reset_index(name="Avg_Discount_%")
)

region_business_summary = region_business_summary.merge(avg_disc_region, on="Region")
region_business_summary = region_business_summary.rename(columns={"Sales": "Total_Sales", "Profit": "Total_Profit"})
region_business_summary = region_business_summary[
    ["Region", "Total_Sales", "Total_Profit", "Avg_Discount_%", "Orders", "Profit_Margin_%"]
]

save_csv(region_business_summary, "region_business_summary.csv")


# =========================================================
# CATEGORY BUSINESS SUMMARY
# =========================================================

category_business_summary = aggregate_metrics(df, group_col="Category", include_quantity=True)

avg_disc_cat = (
    df.groupby("Category")["Discount"]
      .mean()
      .mul(100)
      .reset_index(name="Avg_Discount_%")
)

category_business_summary = category_business_summary.merge(avg_disc_cat, on="Category")
category_business_summary = category_business_summary[
    ["Category", "Sales", "Profit", "Quantity", "Orders", "Profit_Margin_%", "Avg_Discount_%"]
]

save_csv(category_business_summary, "category_business_summary.csv")


# ============================================================
# SUB-CATEGORY SUMMARY EXPORT
# ============================================================

sub_category_summary = (
    df.groupby(['Category', 'Sub_Category'])
      .agg(
          Sales=('Sales', 'sum'),
          Profit=('Profit', 'sum'),
          Orders=('Order_ID', 'nunique'),
          Customers=('Customer_ID', 'nunique'),
          Quantity=('Quantity', 'sum'),
          Avg_Discount=('Discount', 'mean')
      )
      .reset_index()
)

sub_category_summary['Profit_Margin_%'] = (
    sub_category_summary['Profit']
    / sub_category_summary['Sales'] * 100
).round(2)

sub_category_summary['Avg_Discount'] = (
    sub_category_summary['Avg_Discount'] * 100
).round(2)

sub_category_summary = (
    sub_category_summary
    .sort_values('Sales', ascending=False)
)

save_csv(sub_category_summary, "sub_category_summary.csv")


# =========================================================
# PRODUCT SUMMARY
# =========================================================

product_summary = aggregate_metrics(df, group_col="Product_Name", include_quantity=True)
product_summary = product_summary[["Product_Name", "Sales", "Profit", "Quantity"]]

save_csv(product_summary, "product_summary.csv")


# =========================================================
# PRODUCT ENRICHED SUMMARY
# =========================================================

log_step("Building product enriched summary", "STEP")

required_cols = [
    "Product_Name",
    "Category",
    "Sub_Category",
    "Sales",
    "Profit",
    "Quantity",
    "Order_ID",
    "Customer_ID",
    "Customer_Name",
    "Discount",
]
require(df, required_cols)

work_df = df.copy()

# --- Date handling ---
date_col = get_date_col(work_df)
if not date_col:
    raise ValueError("No valid date column found for product enriched summary.")

work_df[date_col] = pd.to_datetime(work_df[date_col], errors="coerce")

# --- Product ID handling ---
# Use existing Product_ID if available, otherwise create a stable surrogate ID
if "Product_ID" in work_df.columns:
    product_id_col = "Product_ID"
else:
    product_id_col = "__Product_ID__"
    work_df[product_id_col] = (
        work_df["Category"].astype(str).fillna("Unknown")
        + " | "
        + work_df["Sub_Category"].astype(str).fillna("Unknown")
        + " | "
        + work_df["Product_Name"].astype(str).fillna("Unknown")
    )
    work_df[product_id_col] = pd.factorize(work_df[product_id_col])[0] + 1
    work_df[product_id_col] = "P-" + work_df[product_id_col].astype(str).str.zfill(5)

# --- Main product-level aggregation ---
product_enriched_summary = (
    work_df.groupby(
        [
            product_id_col,
            "Product_Name",
            "Category",
            "Sub_Category",
        ],
        as_index=False
    )
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Total_Orders=("Order_ID", "nunique"),
        Avg_Discount_=("Discount", lambda x: x.mean() * 100),
        First_Order_Date=(date_col, "min"),
        Last_Order_Date=(date_col, "max"),
    )
)

# --- Derived metrics ---
product_enriched_summary["Profit_Margin_%"] = np.where(
    product_enriched_summary["Total_Sales"] != 0,
    (product_enriched_summary["Total_Profit"] / product_enriched_summary["Total_Sales"]) * 100,
    np.nan
)

product_enriched_summary["Product_Cost"] = (
    product_enriched_summary["Total_Sales"] - product_enriched_summary["Total_Profit"]
)

# --- Top customer for each product ---
top_customer_rows = (
    work_df.groupby(
        [
            product_id_col,
            "Product_Name",
            "Customer_ID",
            "Customer_Name",
        ],
        as_index=False
    )
    .agg(
        Top_Customer_Orders=("Order_ID", "nunique"),
        Top_Customer_Quantity=("Quantity", "sum"),
        Top_Customer_Sales=("Sales", "sum"),
    )
    .sort_values(
        [
            product_id_col,
            "Top_Customer_Orders",
            "Top_Customer_Sales",
            "Top_Customer_Quantity",
        ],
        ascending=[True, False, False, False]
    )
)

top_customer_rows = (
    top_customer_rows.groupby(product_id_col, as_index=False)
    .head(1)
    .rename(
        columns={
            "Customer_ID": "Top_Ordered_Customer_ID",
            "Customer_Name": "Top_Ordered_Customer_Name",
        }
    )
)

# Keep only the columns needed for merge
top_customer_rows = top_customer_rows[
    [
        product_id_col,
        "Top_Ordered_Customer_ID",
        "Top_Ordered_Customer_Name",
        "Top_Customer_Orders",
        "Top_Customer_Quantity",
        "Top_Customer_Sales",
    ]
]

product_enriched_summary = product_enriched_summary.merge(
    top_customer_rows,
    on=product_id_col,
    how="left"
)

# --- Final column order ---
product_enriched_summary = product_enriched_summary[
    [
        product_id_col,
        "Product_Name",
        "Category",
        "Sub_Category",
        "Total_Sales",
        "Total_Profit",
        "Total_Quantity",
        "Total_Orders",
        "Avg_Discount_",
        "Profit_Margin_%",
        "Product_Cost",
        "First_Order_Date",
        "Last_Order_Date",
        "Top_Ordered_Customer_ID",
        "Top_Ordered_Customer_Name",
        "Top_Customer_Orders",
        "Top_Customer_Quantity",
        "Top_Customer_Sales",
    ]
].copy()

# Rename surrogate column to Product_ID if needed
if product_id_col != "Product_ID":
    product_enriched_summary = product_enriched_summary.rename(
        columns={product_id_col: "Product_ID"}
    )

# --- Formatting ---
product_enriched_summary["Avg_Discount_"] = product_enriched_summary["Avg_Discount_"].round(2)
product_enriched_summary["Profit_Margin_%"] = product_enriched_summary["Profit_Margin_%"].round(2)
product_enriched_summary["Product_Cost"] = product_enriched_summary["Product_Cost"].round(2)
product_enriched_summary["Total_Sales"] = product_enriched_summary["Total_Sales"].round(2)
product_enriched_summary["Total_Profit"] = product_enriched_summary["Total_Profit"].round(2)
product_enriched_summary["Top_Customer_Sales"] = product_enriched_summary["Top_Customer_Sales"].round(2)

# Make dates cleaner for CSV
product_enriched_summary["First_Order_Date"] = pd.to_datetime(
    product_enriched_summary["First_Order_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

product_enriched_summary["Last_Order_Date"] = pd.to_datetime(
    product_enriched_summary["Last_Order_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

# --- Save ---
save_csv(product_enriched_summary, "product_enriched_summary.csv")


# =========================================================
# PRODUCT ORDER HISTORY
# =========================================================

log_step("Building product order history table", "STEP")

required_cols = [
    "Order_ID",
    "Product_Name",
    "Sales",
    "Profit",
    "Quantity",
    "Discount",
    "Customer_ID",
    "Customer_Name"
]
require(df, required_cols)

work_df = df.copy()

# ---------------------------------------------------------
# Date handling
# ---------------------------------------------------------
date_col = get_date_col(work_df)
if not date_col:
    raise ValueError("No valid date column found for product order history.")

work_df[date_col] = pd.to_datetime(work_df[date_col], errors="coerce")
work_df = work_df.dropna(subset=[date_col]).copy()

# Standardize the column name for output
if date_col != "Order_Date":
    work_df = work_df.rename(columns={date_col: "Order_Date"})

# ---------------------------------------------------------
# Product_ID handling
# - Keep existing Product_ID if present
# - Otherwise create a stable surrogate ID from product descriptors
# ---------------------------------------------------------
if "Product_ID" not in work_df.columns:
    product_key = (
        work_df.get("Category", pd.Series("Unknown", index=work_df.index)).astype(str).fillna("Unknown")
        + " | "
        + work_df.get("Sub_Category", pd.Series("Unknown", index=work_df.index)).astype(str).fillna("Unknown")
        + " | "
        + work_df["Product_Name"].astype(str).fillna("Unknown")
    )

    codes, _ = pd.factorize(product_key, sort=True)
    work_df["Product_ID"] = "P-" + (codes + 1).astype(str).str.zfill(5)

# ---------------------------------------------------------
# Product cost at transaction level
# ---------------------------------------------------------
work_df["Product_Cost"] = work_df["Sales"] - work_df["Profit"]

# ---------------------------------------------------------
# Final product order history table
# One row = one order line / transaction record
# ---------------------------------------------------------
product_order_history = work_df[
    [
        "Product_ID",
        "Product_Name",
        "Order_ID",
        "Order_Date",
        "Customer_ID",
        "Customer_Name",
        "Sales",
        "Profit",
        "Quantity",
        "Discount",
        "Product_Cost",
    ]
].copy()

# Sort for clean history browsing
product_order_history = product_order_history.sort_values(
    ["Order_Date", "Order_ID", "Product_Name"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------
product_order_history["Sales"] = product_order_history["Sales"].round(2)
product_order_history["Profit"] = product_order_history["Profit"].round(2)
product_order_history["Product_Cost"] = product_order_history["Product_Cost"].round(2)
product_order_history["Discount"] = (product_order_history["Discount"] * 100).round(2)

product_order_history["Order_Date"] = pd.to_datetime(
    product_order_history["Order_Date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
save_csv(product_order_history, "product_order_history.csv")


# =========================================================
# PRODUCT CATEGORY DISTRIBUTION
# =========================================================

log_step("Building product category distribution summary", "STEP")

require(df, ["Category", "Product_Name", "Sales", "Profit", "Quantity", "Order_ID"])

work_df = df.copy()

# Use Product_ID if it exists; otherwise fall back to Product_Name
product_key_col = "Product_ID" if "Product_ID" in work_df.columns else "Product_Name"

product_category_distribution = (
    work_df.groupby("Category", as_index=False)
    .agg(
        Product_Count=(product_key_col, "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Total_Orders=("Order_ID", "nunique"),
    )
)

total_products = product_category_distribution["Product_Count"].sum()
total_sales = product_category_distribution["Total_Sales"].sum()
total_profit = product_category_distribution["Total_Profit"].sum()

product_category_distribution["Product_Share_%"] = np.where(
    total_products != 0,
    (product_category_distribution["Product_Count"] / total_products) * 100,
    np.nan
)

product_category_distribution["Sales_Share_%"] = np.where(
    total_sales != 0,
    (product_category_distribution["Total_Sales"] / total_sales) * 100,
    np.nan
)

product_category_distribution["Profit_Share_%"] = np.where(
    total_profit != 0,
    (product_category_distribution["Total_Profit"] / total_profit) * 100,
    np.nan
)

product_category_distribution["Product_Share_%"] = product_category_distribution["Product_Share_%"].round(2)
product_category_distribution["Sales_Share_%"] = product_category_distribution["Sales_Share_%"].round(2)
product_category_distribution["Profit_Share_%"] = product_category_distribution["Profit_Share_%"].round(2)

product_category_distribution["Total_Sales"] = product_category_distribution["Total_Sales"].round(2)
product_category_distribution["Total_Profit"] = product_category_distribution["Total_Profit"].round(2)

product_category_distribution = product_category_distribution[
    [
        "Category",
        "Product_Count",
        "Total_Sales",
        "Total_Profit",
        "Total_Quantity",
        "Total_Orders",
        "Product_Share_%",
        "Sales_Share_%",
        "Profit_Share_%",
    ]
].sort_values("Total_Sales", ascending=False)

save_csv(product_category_distribution, "product_category_distribution.csv")


# =========================================================
# PRODUCT SUB-CATEGORY DISTRIBUTION
# =========================================================

log_step("Building product sub-category distribution summary", "STEP")

require(df, ["Category", "Sub_Category", "Sales", "Profit", "Quantity", "Order_ID"])

work_df = df.copy()

# Use Product_ID if available; otherwise fall back to Product_Name
product_key_col = "Product_ID" if "Product_ID" in work_df.columns else "Product_Name"

product_sub_category_distribution = (
    work_df.groupby(["Category", "Sub_Category"], as_index=False)
    .agg(
        Product_Count=(product_key_col, "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Total_Orders=("Order_ID", "nunique"),
    )
)

total_products = product_sub_category_distribution["Product_Count"].sum()
total_sales = product_sub_category_distribution["Total_Sales"].sum()
total_profit = product_sub_category_distribution["Total_Profit"].sum()

product_sub_category_distribution["Product_Share_%"] = np.where(
    total_products != 0,
    (product_sub_category_distribution["Product_Count"] / total_products) * 100,
    np.nan
)

product_sub_category_distribution["Sales_Share_%"] = np.where(
    total_sales != 0,
    (product_sub_category_distribution["Total_Sales"] / total_sales) * 100,
    np.nan
)

product_sub_category_distribution["Profit_Share_%"] = np.where(
    total_profit != 0,
    (product_sub_category_distribution["Total_Profit"] / total_profit) * 100,
    np.nan
)

product_sub_category_distribution["Product_Share_%"] = product_sub_category_distribution["Product_Share_%"].round(2)
product_sub_category_distribution["Sales_Share_%"] = product_sub_category_distribution["Sales_Share_%"].round(2)
product_sub_category_distribution["Profit_Share_%"] = product_sub_category_distribution["Profit_Share_%"].round(2)

product_sub_category_distribution["Total_Sales"] = product_sub_category_distribution["Total_Sales"].round(2)
product_sub_category_distribution["Total_Profit"] = product_sub_category_distribution["Total_Profit"].round(2)

product_sub_category_distribution = product_sub_category_distribution[
    [
        "Category",
        "Sub_Category",
        "Product_Count",
        "Total_Sales",
        "Total_Profit",
        "Total_Quantity",
        "Total_Orders",
        "Product_Share_%",
        "Sales_Share_%",
        "Profit_Share_%",
    ]
].sort_values(["Category", "Total_Sales"], ascending=[True, False])

save_csv(product_sub_category_distribution, "product_sub_category_distribution.csv")


# =========================================================
# CUSTOMER SUMMARY
# =========================================================

customer_summary = aggregate_metrics(
    df,
    group_col=["Customer_ID", "Customer_Name"],
    include_quantity=False
)

customer_summary = customer_summary.rename(columns={"Sales": "Total_Sales", "Profit": "Total_Profit"})
customer_summary["Avg_Order_Value"] = (
    customer_summary["Total_Sales"] / customer_summary["Orders"]
)

customer_summary = customer_summary[
    ["Customer_ID", "Customer_Name", "Total_Sales", "Total_Profit", "Orders", "Avg_Order_Value"]
]

save_csv(customer_summary, "customer_summary.csv")



# =========================================================
# ORDER VALUE DISTRIBUTION SUMMARY
# =========================================================

require(df, ["Order_ID", "Sales"])

# Rebuild order-level sales first so each order appears once
order_value_base = (
    df.groupby("Order_ID", as_index=False)
      .agg(
          Total_Sales=("Sales", "sum")
      )
)

# Band the order values
sales_bins = [0, 100, 250, 500, 1000, np.inf]
sales_labels = ["0-100", "101-250", "251-500", "501-1000", "1000+"]

order_value_base["Sales_Band"] = pd.cut(
    order_value_base["Total_Sales"],
    bins=sales_bins,
    labels=sales_labels,
    include_lowest=True
)

# Summary by band
order_value_distribution = (
    order_value_base.groupby("Sales_Band", observed=False, as_index=False)
    .agg(
        Orders=("Order_ID", "nunique"),
        Total_Sales=("Total_Sales", "sum"),
        Avg_Order_Value=("Total_Sales", "mean")
    )
)

# Share calculations
total_orders = order_value_base["Order_ID"].nunique()
total_sales = order_value_base["Total_Sales"].sum()

order_value_distribution["Order_Share_%"] = np.where(
    total_orders != 0,
    (order_value_distribution["Orders"] / total_orders) * 100,
    np.nan
)

order_value_distribution["Sales_Share_%"] = np.where(
    total_sales != 0,
    (order_value_distribution["Total_Sales"] / total_sales) * 100,
    np.nan
)

# Final formatting
order_value_distribution["Avg_Order_Value"] = order_value_distribution["Avg_Order_Value"].round(2)
order_value_distribution["Order_Share_%"] = order_value_distribution["Order_Share_%"].round(2)
order_value_distribution["Sales_Share_%"] = order_value_distribution["Sales_Share_%"].round(2)

order_value_distribution = order_value_distribution[
    [
        "Sales_Band",
        "Orders",
        "Total_Sales",
        "Avg_Order_Value",
        "Order_Share_%",
        "Sales_Share_%"
    ]
]

save_csv(order_value_distribution, "order_value_distribution.csv")


# =========================================================
# PURCHASE QUANTITY DISTRIBUTION SUMMARY
# =========================================================

require(df, ["Order_ID", "Quantity", "Sales"])

# Rebuild order-level quantity so each order is counted once
purchase_quantity_base = (
    df.groupby("Order_ID", as_index=False)
      .agg(
          Total_Quantity=("Quantity", "sum"),
          Total_Sales=("Sales", "sum")
      )
)

# Band the order quantities
quantity_bins = [0, 1, 2, 3, 5, 10, np.inf]
quantity_labels = ["1", "2", "3", "4-5", "6-10", "10+"]

purchase_quantity_base["Quantity_Band"] = pd.cut(
    purchase_quantity_base["Total_Quantity"],
    bins=quantity_bins,
    labels=quantity_labels,
    include_lowest=True
)

# Summary by band
purchase_quantity_distribution = (
    purchase_quantity_base.groupby("Quantity_Band", observed=False, as_index=False)
    .agg(
        Orders=("Order_ID", "nunique"),
        Total_Quantity=("Total_Quantity", "sum"),
        Total_Sales=("Total_Sales", "sum"),
        Avg_Order_Quantity=("Total_Quantity", "mean")
    )
)

# Share calculations
total_orders = purchase_quantity_base["Order_ID"].nunique()
total_quantity = purchase_quantity_base["Total_Quantity"].sum()

purchase_quantity_distribution["Order_Share_%"] = np.where(
    total_orders != 0,
    (purchase_quantity_distribution["Orders"] / total_orders) * 100,
    np.nan
)

purchase_quantity_distribution["Quantity_Share_%"] = np.where(
    total_quantity != 0,
    (purchase_quantity_distribution["Total_Quantity"] / total_quantity) * 100,
    np.nan
)

# Final formatting
purchase_quantity_distribution["Avg_Order_Quantity"] = purchase_quantity_distribution["Avg_Order_Quantity"].round(2)
purchase_quantity_distribution["Order_Share_%"] = purchase_quantity_distribution["Order_Share_%"].round(2)
purchase_quantity_distribution["Quantity_Share_%"] = purchase_quantity_distribution["Quantity_Share_%"].round(2)

purchase_quantity_distribution = purchase_quantity_distribution[
    [
        "Quantity_Band",
        "Orders",
        "Total_Quantity",
        "Total_Sales",
        "Avg_Order_Quantity",
        "Order_Share_%",
        "Quantity_Share_%"
    ]
]

save_csv(purchase_quantity_distribution, "purchase_quantity_distribution.csv")


# =========================================================
# DISCOUNT SUMMARY
# =========================================================

discount_summary = build_binned_driver_summary(
    df,
    value_col="Discount",
    band_col="Discount_Band",
    bins=[0, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00],
    labels=["0-5%", "5-10%", "10-20%", "20-30%", "30-50%", "50%+"],
)

discount_summary = discount_summary[
    ["Discount_Band", "Sales", "Profit", "Orders", "Profit_Margin_%"]
]

save_csv(discount_summary, "discount_summary.csv")


# =========================================================
# SHIPPING SUMMARY
# =========================================================

ship_mode_summary = aggregate_metrics(df, group_col="Ship_Mode", include_quantity=False)
ship_mode_summary = ship_mode_summary[["Ship_Mode", "Sales", "Profit", "Orders", "Profit_Margin_%"]]

save_csv(ship_mode_summary, "ship_mode_summary.csv")


# =========================================================
# MONTHLY BUSINESS SUMMARY
# =========================================================

if "Order_Date" in df.columns:
    work_df = df.copy()
    work_df["Order_Date"] = pd.to_datetime(work_df["Order_Date"], errors="coerce")

    work_df["Year_Month_Temp"] = work_df["Order_Date"].dt.to_period("M").astype(str)

    monthly_summary = aggregate_metrics(work_df, group_col="Year_Month_Temp", include_quantity=False)
    monthly_summary = monthly_summary.rename(columns={"Year_Month_Temp": "Year_Month"})
    monthly_summary = monthly_summary[["Year_Month", "Sales", "Profit", "Orders", "Profit_Margin_%"]]

    save_csv(monthly_summary, "monthly_summary.csv")


# =========================================================
# MONTHLY GROWTH SUMMARY
# =========================================================

require(df, ["Order_Date", "Sales", "Profit", "Order_ID"])

work_df = df.copy()
work_df["Order_Date"] = pd.to_datetime(work_df["Order_Date"], errors="coerce")
work_df = work_df.dropna(subset=["Order_Date"])

work_df["Year_Month_Temp"] = work_df["Order_Date"].dt.to_period("M").astype(str)

monthly_growth_summary = aggregate_metrics(
    work_df,
    group_col="Year_Month_Temp",
    include_quantity=False
).copy()

monthly_growth_summary = monthly_growth_summary.rename(
    columns={"Year_Month_Temp": "Year_Month"}
)

monthly_growth_summary["Year_Month"] = pd.to_datetime(
    monthly_growth_summary["Year_Month"],
    errors="coerce"
)

monthly_growth_summary = monthly_growth_summary.sort_values("Year_Month").reset_index(drop=True)

monthly_growth_summary["Previous_Month_Sales"] = monthly_growth_summary["Sales"].shift(1)

monthly_growth_summary["MoM_Change"] = (
    monthly_growth_summary["Sales"] - monthly_growth_summary["Previous_Month_Sales"]
)

monthly_growth_summary["MoM_Growth_%"] = np.where(
    monthly_growth_summary["Previous_Month_Sales"].notna() & (monthly_growth_summary["Previous_Month_Sales"] != 0),
    (monthly_growth_summary["MoM_Change"] / monthly_growth_summary["Previous_Month_Sales"]) * 100,
    np.nan
)

total_sales = monthly_growth_summary["Sales"].sum()

monthly_growth_summary["Sales_Share_%"] = np.where(
    total_sales != 0,
    (monthly_growth_summary["Sales"] / total_sales) * 100,
    np.nan
)

monthly_growth_summary["Year_Month"] = monthly_growth_summary["Year_Month"].dt.strftime("%Y-%m")

monthly_growth_summary = monthly_growth_summary[
    [
        "Year_Month",
        "Sales",
        "Profit",
        "Orders",
        "Profit_Margin_%",
        "Previous_Month_Sales",
        "MoM_Change",
        "MoM_Growth_%",
        "Sales_Share_%"
    ]
]

monthly_growth_summary["Sales"] = monthly_growth_summary["Sales"].round(2)
monthly_growth_summary["Profit"] = monthly_growth_summary["Profit"].round(2)
monthly_growth_summary["Profit_Margin_%"] = monthly_growth_summary["Profit_Margin_%"].round(2)
monthly_growth_summary["Previous_Month_Sales"] = monthly_growth_summary["Previous_Month_Sales"].round(2)
monthly_growth_summary["MoM_Change"] = monthly_growth_summary["MoM_Change"].round(2)
monthly_growth_summary["MoM_Growth_%"] = monthly_growth_summary["MoM_Growth_%"].round(2)
monthly_growth_summary["Sales_Share_%"] = monthly_growth_summary["Sales_Share_%"].round(2)

save_csv(monthly_growth_summary, "monthly_growth_summary.csv")


# =========================================================
# WEEKDAY SALES SUMMARY
# =========================================================

require(df, ["Sales", "Order_ID"])

date_col = get_date_col(df)
if not date_col:
    raise ValueError("No valid date column found for weekday sales summary.")

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

# Build order-level sales first so each order is counted once
weekday_base = (
    df.dropna(subset=[date_col])
      .copy()
)

weekday_base["Weekday"] = weekday_base[date_col].dt.day_name()

weekday_order_sales = (
    weekday_base.groupby(["Order_ID", "Weekday"], as_index=False)
    .agg(
        Total_Sales=("Sales", "sum")
    )
)

weekday_sales_summary = (
    weekday_order_sales.groupby("Weekday", as_index=False)
    .agg(
        Orders=("Order_ID", "nunique"),
        Total_Sales=("Total_Sales", "sum"),
        Avg_Order_Value=("Total_Sales", "mean")
    )
)

weekday_sales_summary["Weekday"] = pd.Categorical(
    weekday_sales_summary["Weekday"],
    categories=weekday_order,
    ordered=True
)

weekday_sales_summary = weekday_sales_summary.sort_values("Weekday")

total_sales = weekday_sales_summary["Total_Sales"].sum()

weekday_sales_summary["Sales_Share_%"] = np.where(
    total_sales != 0,
    (weekday_sales_summary["Total_Sales"] / total_sales) * 100,
    np.nan
)

weekday_sales_summary["Avg_Order_Value"] = weekday_sales_summary["Avg_Order_Value"].round(2)
weekday_sales_summary["Sales_Share_%"] = weekday_sales_summary["Sales_Share_%"].round(2)

weekday_sales_summary = weekday_sales_summary[
    [
        "Weekday",
        "Orders",
        "Total_Sales",
        "Avg_Order_Value",
        "Sales_Share_%"
    ]
]

save_csv(weekday_sales_summary, "weekday_sales_summary.csv")


# =========================================================
# LOSS MAKING PRODUCTS
# =========================================================

loss_table = aggregate_sales_profit(df, group_col="Product_Name")

avg_disc_prod = (
    df.groupby("Product_Name")["Discount"]
      .mean()
      .mul(100)
      .reset_index(name="Avg_Discount_%")
)

loss_table = loss_table.merge(avg_disc_prod, on="Product_Name")
loss_table = loss_table[["Product_Name", "Sales", "Profit", "Avg_Discount_%"]]

save_csv(loss_table, "loss_making_products.csv")


# =========================================================
# PRODUCT PROFITABILITY MATRIX
# =========================================================

matrix_table = aggregate_metrics(df, group_col="Product_Name", include_quantity=True)

matrix_table["Category"] = np.where(
    (matrix_table["Sales"] > matrix_table["Sales"].median()) &
    (matrix_table["Profit"] > matrix_table["Profit"].median()),
    "High Value",
    "Needs Attention"
)

matrix_table = matrix_table[["Product_Name", "Sales", "Profit", "Quantity", "Profit_Margin_%", "Category"]]

save_csv(matrix_table, "profitability_matrix.csv")

print("✅ All summary tables saved successfully using utility modules.")
print(f"📁 Target Destination: {OUTPUT_DIR}")

[14:35:45] 🔹 Saving summary tables for dashboard usage
✅ Saved: ../exports/executive_summary.csv
✅ Saved: ../exports/region_business_summary.csv
✅ Saved: ../exports/category_business_summary.csv
✅ Saved: ../exports/sub_category_summary.csv
✅ Saved: ../exports/product_summary.csv
[14:35:46] 🔹 Building product enriched summary
✅ Saved: ../exports/product_enriched_summary.csv
[14:35:46] 🔹 Building product order history table
✅ Saved: ../exports/product_order_history.csv
[14:35:46] 🔹 Building product category distribution summary
✅ Saved: ../exports/product_category_distribution.csv
[14:35:46] 🔹 Building product sub-category distribution summary
✅ Saved: ../exports/product_sub_category_distribution.csv
✅ Saved: ../exports/customer_summary.csv
✅ Saved: ../exports/order_value_distribution.csv
✅ Saved: ../exports/purchase_quantity_distribution.csv
✅ Saved: ../exports/discount_summary.csv
✅ Saved: ../exports/ship_mode_summary.csv
✅ Saved: ../exports/monthly_summary.csv
✅ Saved: ../exports/mont

## 14. Strategic Recommendations

# Executive Summary

- **Strongest customer segment:** Champions / Loyal Customers are the highest-value groups and should remain the primary retention focus.
- **Biggest discount risk:** Higher discount bands are showing weaker profitability and need tighter control.
- **Best-performing product group:** Core Winners should receive the most attention for inventory, promotion, and expansion.
- **Top forecast implication:** Forecasted demand supports proactive planning for inventory, staffing, and campaign timing.
- **Main operational priority:** Protect margin, retain high-value customers, and concentrate effort on the most profitable products.

In [154]:
# STRATEGIC RECOMMENDATIONS — STATELESS REBUILD

log_step("Building strategic recommendations", "STEP")

def build_strategic_recommendations(base_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build recommendation table directly from the base dataframe.
    This avoids hidden dependencies on earlier notebook outputs.
    """
    require(base_df, ["Sales", "Profit", "Discount", "Customer_ID", "Customer_Name", "Order_Date", "Order_ID", "Sub_Category", "Category"])

    recommendations = []

    def add_recommendation(area, evidence, recommendation, impact, priority):
        recommendations.append({
            "Area": area,
            "Evidence": evidence,
            "Recommendation": recommendation,
            "Business_Impact": impact,
            "Priority": priority
        })

    # ---------------------------------------------------------
    # 1) Customer strategy from locally rebuilt RFM
    # ---------------------------------------------------------
    rfm_tbl = build_rfm_table(base_df)

    rfm_segment_summary_local = (
        rfm_tbl.groupby("Segment", as_index=False)
        .agg(
            Customers=("Customer_ID", "nunique"),
            Total_Sales=("Monetary", "sum"),
            Avg_Recency=("Recency", "mean"),
            Avg_Frequency=("Frequency", "mean"),
            Avg_Monetary=("Monetary", "mean"),
        )
    )

    if not rfm_segment_summary_local.empty and rfm_segment_summary_local["Customers"].sum() > 0:
        rfm_segment_summary_local["Revenue_Share_%"] = (
            rfm_segment_summary_local["Total_Sales"]
            / rfm_segment_summary_local["Total_Sales"].sum()
        ) * 100

        top_segment = rfm_segment_summary_local.sort_values("Total_Sales", ascending=False).iloc[0]
        add_recommendation(
            "Customer Strategy",
            f"{top_segment['Segment']} contributes the highest revenue share.",
            f"Protect and grow {top_segment['Segment']} with loyalty rewards, upsell offers, and personalized engagement.",
            "Retention Impact",
            "High"
        )

        if "At Risk" in rfm_segment_summary_local["Segment"].values:
            at_risk = rfm_segment_summary_local[rfm_segment_summary_local["Segment"] == "At Risk"].iloc[0]
            add_recommendation(
                "Customer Recovery",
                f"At Risk customers account for {at_risk['Revenue_Share_%']:.1f}% of segment revenue.",
                "Run targeted win-back campaigns, proactive follow-ups, and time-bound retention offers for At Risk customers.",
                "Revenue Impact",
                "High"
            )

        if "Lost Customers" in rfm_segment_summary_local["Segment"].values:
            lost = rfm_segment_summary_local[rfm_segment_summary_local["Segment"] == "Lost Customers"].iloc[0]
            add_recommendation(
                "Customer Recovery",
                f"Lost Customers contribute {lost['Revenue_Share_%']:.1f}% of segment revenue.",
                "Use low-cost reactivation campaigns only where expected return justifies the acquisition cost.",
                "Revenue Impact",
                "Medium"
            )

    # ---------------------------------------------------------
    # 2) Discount strategy from a locally rebuilt band summary
    # ---------------------------------------------------------
    discount_summary_local = build_binned_driver_summary(
        base_df,
        value_col="Discount",
        band_col="Discount_Band",
        bins=[-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
        labels=["0%", "1-10%", "10-20%", "20-30%", "30-50%", "50%+"],
    )

    if not discount_summary_local.empty and {"Profit_Margin_%", "Discount_Band"}.issubset(discount_summary_local.columns):
        worst_band = discount_summary_local.sort_values("Profit_Margin_%").iloc[0]
        best_band = discount_summary_local.sort_values("Profit_Margin_%", ascending=False).iloc[0]

        add_recommendation(
            "Discount Strategy",
            f"The weakest discount band is {worst_band['Discount_Band']} with margin {worst_band['Profit_Margin_%']:.2f}%.",
            "Reduce discount exposure in weak bands and reserve heavy discounts for clearance or targeted acquisition only.",
            "Margin Impact",
            "High"
        )

        add_recommendation(
            "Discount Strategy",
            f"The strongest discount band is {best_band['Discount_Band']} with margin {best_band['Profit_Margin_%']:.2f}%.",
            "Prioritize controlled discount bands that support conversion without eroding margin.",
            "Margin Impact",
            "Medium"
        )

    # ---------------------------------------------------------
    # 3) Product portfolio strategy from a locally rebuilt efficiency matrix
    # ---------------------------------------------------------
    eff_matrix_local = aggregate_metrics(base_df, ["Category", "Sub_Category"]).copy()

    if not eff_matrix_local.empty:
        eff_total_sales = eff_matrix_local["Sales"].sum()
        eff_matrix_local["Sales Contribution %"] = np.where(
            eff_total_sales != 0,
            (eff_matrix_local["Sales"] / eff_total_sales) * 100,
            0
        )
        eff_matrix_local["Profit Margin %"] = np.where(
            eff_matrix_local["Sales"] != 0,
            (eff_matrix_local["Profit"] / eff_matrix_local["Sales"]) * 100,
            0
        )

        sales_mid = eff_matrix_local["Sales Contribution %"].median()
        margin_mid = eff_matrix_local["Profit Margin %"].median()

        eff_matrix_local["Segment"] = eff_matrix_local.apply(
            profitability_segment,
            axis=1,
            sales_mid=sales_mid,
            margin_mid=margin_mid
        )

        core_winners = eff_matrix_local[eff_matrix_local["Segment"] == "Core Winners"]
        hidden_gems = eff_matrix_local[eff_matrix_local["Segment"] == "Hidden Gems"]
        loss_makers = eff_matrix_local[eff_matrix_local["Segment"] == "Loss Makers"]

        if not core_winners.empty:
            top_core = core_winners.sort_values("Profit Margin %", ascending=False).iloc[0]
            add_recommendation(
                "Product Portfolio",
                f"{len(core_winners)} sub-categories are Core Winners; top margin performer is {top_core['Sub_Category']}.",
                "Prioritize Core Winners in inventory planning, merchandising, and promotional spend.",
                "Revenue Impact",
                "High"
            )

        if not hidden_gems.empty:
            top_hidden = hidden_gems.sort_values("Profit Margin %", ascending=False).iloc[0]
            add_recommendation(
                "Product Portfolio",
                f"{len(hidden_gems)} sub-categories are Hidden Gems; {top_hidden['Sub_Category']} is the strongest scaling candidate.",
                "Increase exposure for Hidden Gems and test controlled scaling with modest promotion support.",
                "Revenue Impact",
                "Medium"
            )

        if not loss_makers.empty:
            worst_loss = loss_makers.sort_values("Profit Margin %").iloc[0]
            add_recommendation(
                "Product Portfolio",
                f"{len(loss_makers)} sub-categories are Loss Makers; weakest performer is {worst_loss['Sub_Category']}.",
                "Review pricing, discounting, bundling, or rationalize the worst-performing Loss Makers if losses persist.",
                "Margin Impact",
                "High"
            )

    # ---------------------------------------------------------
    # 4) Forecast strategy from locally rebuilt forecast table
    # ---------------------------------------------------------
    _, forecast_future_local, _ = build_sales_forecast_table(base_df, horizon=6)

    if not forecast_future_local.empty:
        future_sales_est = forecast_future_local["Forecast"].sum()
        add_recommendation(
            "Forecast Strategy",
            f"The model projects approximately ${future_sales_est:,.0f} total sales over the forecast horizon.",
            "Use the forecast to plan inventory, staffing, and campaign timing in advance.",
            "Operational Impact",
            "Medium"
        )

    # ---------------------------------------------------------
    # 5) Pareto strategy rebuilt locally from sub-category sales
    # ---------------------------------------------------------
    pareto_local = aggregate_sales_profit(base_df, "Sub_Category")[["Sub_Category", "Sales"]].copy()
    pareto_local = pareto_local.sort_values("Sales", ascending=False).reset_index(drop=True)

    if not pareto_local.empty:
        pareto_local["Cumulative Sales"] = pareto_local["Sales"].cumsum()
        pareto_local["Cumulative %"] = (pareto_local["Cumulative Sales"] / pareto_local["Sales"].sum()) * 100
        vital_count = int((pareto_local["Cumulative %"] <= 80).sum())

        add_recommendation(
            "Pareto Strategy",
            f"The top {vital_count} sub-categories contribute roughly 80% of sales.",
            "Concentrate supplier negotiations, inventory attention, and promotional effort on these high-impact sub-categories.",
            "Revenue Impact",
            "High"
        )

    strategic_recommendations_local = pd.DataFrame(recommendations)

    if strategic_recommendations_local.empty:
        strategic_recommendations_local = pd.DataFrame({
            "Area": ["N/A"],
            "Evidence": ["No recommendation signals were generated."],
            "Recommendation": ["Review the upstream analysis sections."],
            "Business_Impact": ["N/A"],
            "Priority": ["Low"]
        })

    priority_order = {"High": 1, "Medium": 2, "Low": 3}
    strategic_recommendations_local["Priority_Order"] = strategic_recommendations_local["Priority"].map(priority_order).fillna(99)

    strategic_recommendations_local = (
        strategic_recommendations_local.sort_values(["Priority_Order", "Area"], ascending=[True, True])
        .drop(columns=["Priority_Order"])
        .reset_index(drop=True)
    )

    return strategic_recommendations_local


strategic_recommendations = build_strategic_recommendations(df)
save_csv(strategic_recommendations, "strategic_recommendations.csv")
display(strategic_recommendations)

[14:35:47] 🔹 Building strategic recommendations
✅ Saved: ../exports/strategic_recommendations.csv


,Area,Evidence,Recommendation,Business_Impact,Priority
0,Customer Recovery,At Risk customers account for 1.9% of segment ...,"Run targeted win-back campaigns, proactive fol...",Revenue Impact,High
1,Customer Strategy,Champions contributes the highest revenue share.,Protect and grow Champions with loyalty reward...,Retention Impact,High
2,Discount Strategy,The weakest discount band is 50%+ with margin ...,Reduce discount exposure in weak bands and res...,Margin Impact,High
3,Pareto Strategy,The top 8 sub-categories contribute roughly 80...,"Concentrate supplier negotiations, inventory a...",Revenue Impact,High
4,Product Portfolio,3 sub-categories are Core Winners; top margin ...,"Prioritize Core Winners in inventory planning,...",Revenue Impact,High
5,Product Portfolio,1 sub-categories are Loss Makers; weakest perf...,"Review pricing, discounting, bundling, or rati...",Margin Impact,High
6,Discount Strategy,The strongest discount band is 0% with margin ...,Prioritize controlled discount bands that supp...,Margin Impact,Medium
7,Forecast Strategy,"The model projects approximately $2,361,615 to...","Use the forecast to plan inventory, staffing, ...",Operational Impact,Medium
8,Product Portfolio,6 sub-categories are Hidden Gems; Paper is the...,Increase exposure for Hidden Gems and test con...,Revenue Impact,Medium


In [155]:
# EXECUTIVE TAKEAWAYS — STATELESS

log_step("Generating executive takeaways", "STEP")

executive_recommendations = build_strategic_recommendations(df)

print("EXECUTIVE TAKEAWAYS")
print("=" * 70)

for _, row in executive_recommendations.iterrows():
    print(f"- [{row['Priority']}] {row['Area']}")
    print(f"  Evidence: {row['Evidence']}")
    print(f"  Action: {row['Recommendation']}")
    print(f"  Impact: {row['Business_Impact']}\n")

[14:35:47] 🔹 Generating executive takeaways
EXECUTIVE TAKEAWAYS
- [High] Customer Recovery
  Evidence: At Risk customers account for 1.9% of segment revenue.
  Action: Run targeted win-back campaigns, proactive follow-ups, and time-bound retention offers for At Risk customers.
  Impact: Revenue Impact

- [High] Customer Strategy
  Evidence: Champions contributes the highest revenue share.
  Action: Protect and grow Champions with loyalty rewards, upsell offers, and personalized engagement.
  Impact: Retention Impact

- [High] Discount Strategy
  Evidence: The weakest discount band is 50%+ with margin -111.02%.
  Action: Reduce discount exposure in weak bands and reserve heavy discounts for clearance or targeted acquisition only.
  Impact: Margin Impact

- [High] Pareto Strategy
  Evidence: The top 8 sub-categories contribute roughly 80% of sales.
  Action: Concentrate supplier negotiations, inventory attention, and promotional effort on these high-impact sub-categories.
  Impact: Reven

## Final Executive Summary

This notebook shows a strong customer base led by Champions and Loyal Customers, clear margin pressure from weaker discount bands, and a product portfolio where a limited set of sub-categories and products drives most of the value. The diagnostic sections also show where profit leakage is concentrated across regions, cities, customers, and discount patterns. The highest-priority actions are to retain high-value customers, control margin-eroding discounts, protect profitable product groups, and use forecasting for operational planning.

## 15. ADVANCED BUSINESS DIAGNOSTICS

In [156]:
# BOTTOM 10 STATES BY PROFIT — ROOT CAUSE ANALYSIS

require(df, ["State", "Sales", "Profit", "Product_Name"])

state_loss_table, state_loss_summary = build_root_cause_table(df, "State", top_n=10)

state_loss_table = state_loss_table[
    [
        "State",
        "Sales",
        "Profit",
        "Profit_Margin_%",
        "Orders",
        "Quantity",
        "Avg_Discount_%",
        "Top_Category",
        "Top_Sub_Category",
        "Top_Products",
        "Worst_Product",
        "Likely_Reason",
    ]
].copy()

save_csv(state_loss_table, "bottom_10_states_root_cause.csv")

display(
    state_loss_table.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
        "Avg_Discount_%": lambda x: "—" if pd.isna(x) else f"{x:.1f}%",
        "Orders": "{:,.0f}",
        "Quantity": "{:,.0f}",
    })
)

fig = px.bar(
    state_loss_table.sort_values("Profit", ascending=True),
    x="Profit",
    y="State",
    orientation="h",
    color="Profit_Margin_%",
    color_continuous_scale=LOSS_SCALE,
    text=state_loss_table.sort_values("Profit", ascending=True)["Profit"].map(lambda x: f"${x:,.0f}"),
)

fig = style_plotly(
    fig,
    "Bottom 10 States by Profit",
    x_title="Profit ($)",
    y_title="State",
    height=DEFAULT_HEIGHT,
    margin=dict(l=120, r=40, t=80, b=40),
)

fig.update_traces(
    textposition="outside",
    hovertemplate=build_hover_template(
        title_template="<b>%{y}</b>",
        lines=[
            {"label": "Profit", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[0]", "format_spec": ".2f", "suffix": "%"},
        ],
    ),
    customdata=np.stack([state_loss_table.sort_values("Profit", ascending=True)["Profit_Margin_%"]], axis=-1),
)

fig.show()

✅ Saved: ../exports/bottom_10_states_root_cause.csv


,State,Sales,Profit,Profit_Margin_%,Orders,Quantity,Avg_Discount_%,Top_Category,Top_Sub_Category,Top_Products,Worst_Product,Likely_Reason
0,Istanbul,"$31,037.46","$-29,033.70",-93.54%,202,930,60.0%,Furniture,Chairs,"Office Star Executive Leather Armchair, Adjustable, Harbour Creations Executive Leather Armchair, Adjustable, Office Star Swivel Stool, Adjustable","Office Star Executive Leather Armchair, Adjustable",average discount is high (60.0%); overall profit margin is negative
1,Lagos,"$17,185.33","$-25,922.51",-150.84%,150,733,70.0%,Technology,Phones,"Breville Microwave, Red, Hamilton Beach Stove, Silver, Bevis Training Table, Fully Assembled","Breville Microwave, Red",average discount is high (70.0%); overall profit margin is negative
2,Texas,"$170,188.05","$-25,729.36",-15.12%,487,"3,724",37.0%,Technology,Phones,"Lexmark MX611dhe Monochrome Laser Printer, High Speed Automatic Electric Letter Opener, HON 5400 Series Task Chairs for Big and Tall",GBC DocuBind P400 Electric Binding System,average discount is high (37.0%); overall profit margin is negative
3,Ohio,"$78,258.14","$-16,971.38",-21.69%,236,"1,759",32.5%,Technology,Phones,"Cubify CubeX 3D Printer Double Head Print, Samsung Galaxy S III - 16GB - pebble blue (T-Mobile), Plantronics Savi W720 Multi-Device Wireless Headset System",Cubify CubeX 3D Printer Double Head Print,average discount is high (32.5%); overall profit margin is negative
4,Izmir,"$15,161.92","$-15,729.80",-103.75%,59,307,60.0%,Technology,Phones,"Motorola Smart Phone, Cordless, SAFCO Executive Leather Armchair, Adjustable, Samsung Audio Dock, with Caller ID","Motorola Smart Phone, Cordless",average discount is high (60.0%); overall profit margin is negative
5,Pennsylvania,"$116,511.91","$-15,559.96",-13.35%,288,"2,153",32.9%,Technology,Phones,"Canon imageCLASS 2200 Advanced Copier, Martin Yale Chadless Opener Electric Letter Opener, HON 5400 Series Task Chairs for Big and Tall",GBC Ibimaster 500 Manual ProClick Binding System,average discount is high (32.9%); overall profit margin is negative
6,Francisco Morazán,"$48,048.90","$-15,007.42",-31.23%,178,"1,334",40.7%,Furniture,Bookcases,"Cuisinart Refrigerator, Black, Safco Library with Doors, Traditional, Canon Wireless Fax, Laser","Cuisinart Refrigerator, Black",average discount is high (40.7%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases
7,Panama,"$41,490.18","$-14,978.50",-36.10%,172,"1,198",40.8%,Furniture,Chairs,"Harbour Creations Executive Leather Armchair, Black, Ikea Classic Bookcase, Pine, Dania Classic Bookcase, Mobile","Bevis Wood Table, with Bottom Storage",average discount is high (40.8%); overall profit margin is negative
8,Punjab,"$60,069.99","$-14,665.05",-24.41%,109,838,36.3%,Technology,Phones,"Apple Smart Phone, Full Size, Samsung Smart Phone, VoIP, SAFCO Executive Leather Armchair, Red","Apple Smart Phone, Full Size",average discount is high (36.3%); overall profit margin is negative
9,Stockholm,"$21,589.75","$-13,806.44",-63.95%,74,561,50.8%,Office Supplies,Appliances,"Sauder Classic Bookcase, Metal, KitchenAid Refrigerator, Black, Barricks Conference Table, Adjustable Height","Sauder Classic Bookcase, Metal",average discount is high (50.8%); overall profit margin is negative


In [157]:
# BOTTOM 10 CITIES BY PROFIT — ROOT CAUSE ANALYSIS

require(df, ["City", "Sales", "Profit", "Product_Name"])

city_loss_table, city_loss_summary = build_root_cause_table(df, "City", top_n=10)

city_loss_table = city_loss_table[
    [
        "City",
        "Sales",
        "Profit",
        "Profit_Margin_%",
        "Orders",
        "Quantity",
        "Avg_Discount_%",
        "Top_Category",
        "Top_Sub_Category",
        "Top_Products",
        "Worst_Product",
        "Likely_Reason",
    ]
].copy()

save_csv(city_loss_table, "bottom_10_cities_root_cause.csv")

display(
    city_loss_table.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
        "Avg_Discount_%": lambda x: "—" if pd.isna(x) else f"{x:.1f}%",
        "Orders": "{:,.0f}",
        "Quantity": "{:,.0f}",
    })
)

fig = px.bar(
    city_loss_table.sort_values("Profit", ascending=True),
    x="Profit",
    y="City",
    orientation="h",
    color="Profit_Margin_%",
    color_continuous_scale=LOSS_SCALE,
    text=city_loss_table.sort_values("Profit", ascending=True)["Profit"].map(lambda x: f"${x:,.0f}"),
)

fig = style_plotly(
    fig,
    "Bottom 10 Cities by Profit",
    x_title="Profit ($)",
    y_title="City",
    height=DEFAULT_HEIGHT,
    margin=dict(l=180, r=40, t=80, b=40),
)

fig.update_traces(
    textposition="outside",
    hovertemplate=build_hover_template(
        title_template="<b>%{y}</b>",
        lines=[
            {"label": "Profit", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[0]", "format_spec": ".2f", "suffix": "%"},
        ],
    ),
    customdata=np.stack([city_loss_table.sort_values("Profit", ascending=True)["Profit_Margin_%"]], axis=-1),
)

fig.show()

✅ Saved: ../exports/bottom_10_cities_root_cause.csv


,City,Sales,Profit,Profit_Margin_%,Orders,Quantity,Avg_Discount_%,Top_Category,Top_Sub_Category,Top_Products,Worst_Product,Likely_Reason
0,Lagos,"$17,185.33","$-25,922.51",-150.84%,150,733,70.0%,Technology,Phones,"Breville Microwave, Red, Hamilton Beach Stove, Silver, Bevis Training Table, Fully Assembled","Breville Microwave, Red",average discount is high (70.0%); overall profit margin is negative
1,Istanbul,"$21,892.57","$-19,960.91",-91.18%,154,700,60.0%,Furniture,Bookcases,"Harbour Creations Executive Leather Armchair, Adjustable, Harbour Creations Swivel Stool, Adjustable, Dania Library with Doors, Mobile","Harbour Creations Swivel Stool, Adjustable",average discount is high (60.0%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases
2,Tegucigalpa,"$48,048.90","$-15,007.42",-31.23%,178,"1,334",40.7%,Furniture,Bookcases,"Cuisinart Refrigerator, Black, Safco Library with Doors, Traditional, Canon Wireless Fax, Laser","Cuisinart Refrigerator, Black",average discount is high (40.7%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases
3,Philadelphia,"$109,077.01","$-13,837.77",-12.69%,265,"1,981",32.7%,Technology,Phones,"Canon imageCLASS 2200 Advanced Copier, Martin Yale Chadless Opener Electric Letter Opener, HON 5400 Series Task Chairs for Big and Tall","Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish",average discount is high (32.7%); overall profit margin is negative
4,Lahore,"$33,772.25","$-13,626.37",-40.35%,50,414,43.7%,Technology,Phones,"Apple Smart Phone, Full Size, Samsung Smart Phone, VoIP, SAFCO Executive Leather Armchair, Red","Apple Smart Phone, Full Size",average discount is high (43.7%); overall profit margin is negative
5,Stockholm,"$17,499.66","$-11,632.89",-66.47%,58,429,50.8%,Office Supplies,Bookcases,"Sauder Classic Bookcase, Metal, KitchenAid Refrigerator, Black, Barricks Conference Table, Adjustable Height","Sauder Classic Bookcase, Metal",average discount is high (50.8%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases
6,Manila,"$120,886.95","$-11,158.56",-9.23%,206,"1,661",34.8%,Furniture,Chairs,"Office Star Executive Leather Armchair, Black, Hoover Stove, Silver, Panasonic Inkjet, White","Lesro Wood Table, Rectangular",average discount is high (34.8%); overall profit margin is negative
7,Kano,"$7,951.87","$-10,916.21",-137.28%,51,254,70.0%,Furniture,Chairs,"SAFCO Executive Leather Armchair, Black, Hamilton Beach Stove, White, Harbour Creations Executive Leather Armchair, Adjustable","Hamilton Beach Stove, White",average discount is high (70.0%); overall profit margin is negative
8,Hanover,"$11,995.07","$-10,440.16",-87.04%,27,175,53.4%,Technology,Phones,"Apple Smart Phone, with Caller ID, Hamilton Beach Stove, Black, Nokia Smart Phone, Full Size","Bevis Conference Table, Fully Assembled",average discount is high (53.4%); overall profit margin is negative
9,Toulouse,"$12,544.14","$-10,382.22",-82.77%,34,280,55.6%,Furniture,Bookcases,"Memorex Router, USB, Bush Library with Doors, Mobile, Ikea Classic Bookcase, Metal","Memorex Router, USB",average discount is high (55.6%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases


In [158]:
# TOP LOSS-MAKING CUSTOMERS — ROOT CAUSE ANALYSIS

require(df, ["Customer_ID", "Customer_Name", "Sales", "Profit", "Product_Name"])

customer_loss_table, customer_loss_summary = build_root_cause_table(df, "Customer_ID", top_n=25)

# Keep only the loss-making customers
customer_loss_table = customer_loss_table[customer_loss_table["Profit"] < 0].head(10).copy()

# Add display name / region context if available
customer_name_map = (
    df.groupby("Customer_ID")["Customer_Name"]
      .agg(dominant_value)
)

if "Region" in df.columns:
    customer_region_map = (
        df.groupby("Customer_ID")["Region"]
          .agg(dominant_value)
    )
    customer_loss_table["Region"] = customer_loss_table["Customer_ID"].map(customer_region_map)
else:
    customer_loss_table["Region"] = "N/A"

customer_loss_table["Customer_Name"] = customer_loss_table["Customer_ID"].map(customer_name_map)

customer_loss_table = customer_loss_table[
    [
        "Customer_ID",
        "Customer_Name",
        "Region",
        "Sales",
        "Profit",
        "Profit_Margin_%",
        "Orders",
        "Quantity",
        "Avg_Discount_%",
        "Top_Category",
        "Top_Sub_Category",
        "Top_Products",
        "Worst_Product",
        "Likely_Reason",
    ]
].copy()

save_csv(customer_loss_table, "top_loss_making_customers_root_cause.csv")

display(
    customer_loss_table.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
        "Avg_Discount_%": lambda x: "—" if pd.isna(x) else f"{x:.1f}%",
        "Orders": "{:,.0f}",
        "Quantity": "{:,.0f}",
    })
)

✅ Saved: ../exports/top_loss_making_customers_root_cause.csv


,Customer_ID,Customer_Name,Region,Sales,Profit,Profit_Margin_%,Orders,Quantity,Avg_Discount_%,Top_Category,Top_Sub_Category,Top_Products,Worst_Product,Likely_Reason
0,CS-12505,Cindy Stewart,Central,"$11,535.25","$-6,437.37",-55.81%,16,128,13.9%,Technology,Machines,"Cubify CubeX 3D Printer Double Head Print, Hon Executive Leather Armchair, Red, Sharp Copy Machine, Color",Cubify CubeX 3D Printer Double Head Print,overall profit margin is negative
1,DM-3345,Denise Monton,Africa,"$4,998.42","$-5,474.61",-109.53%,6,46,36.7%,Technology,Phones,"Motorola Smart Phone, Cordless, Apple Smart Phone, Full Size, Cisco Signal Booster, with Caller ID","Motorola Smart Phone, Cordless",average discount is high (36.7%); overall profit margin is negative
2,GT-14635,Grant Thornton,Central,"$19,080.35","$-3,790.08",-19.86%,20,149,21.0%,Technology,Machines,"Cubify CubeX 3D Printer Triple Head Print, Canon Personal Copier, Color, Hamilton Beach Refrigerator, Black",Cubify CubeX 3D Printer Triple Head Print,average discount is high (21.0%); overall profit margin is negative
3,LF-17185,Luke Foster,Central,"$12,864.72","$-3,700.20",-28.76%,22,231,24.9%,Office Supplies,Tables,"GBC DocuBind P400 Electric Binding System, Lesro Computer Table, Adjustable Height, Apple Smart Phone, Full Size",GBC DocuBind P400 Electric Binding System,average discount is high (24.9%); overall profit margin is negative; mix is concentrated in low-margin sub-category Tables
4,MT-8070,Michelle Tran,EMEA,"$6,851.92","$-2,991.62",-43.66%,7,54,42.0%,Furniture,Bookcases,"Sauder Classic Bookcase, Traditional, SAFCO Executive Leather Armchair, Set of Two, Hoover Stove, White","Hoover Stove, White",average discount is high (42.0%); overall profit margin is negative; mix is concentrated in low-margin sub-category Bookcases
5,JF-5355,Jay Fein,EMEA,"$3,475.76","$-2,891.35",-83.19%,9,56,45.7%,Office Supplies,Chairs,"SAFCO Executive Leather Armchair, Adjustable, Breville Microwave, Red, Smead Lockers, Industrial","SAFCO Executive Leather Armchair, Adjustable",average discount is high (45.7%); overall profit margin is negative
6,CM-11815,Candace McMahon,Central,"$14,362.80","$-2,881.64",-20.06%,23,157,24.1%,Technology,Phones,"Samsung Smart Phone, VoIP, Panasonic Printer, Red, Nokia Smart Phone, Full Size","Nokia Smart Phone, Full Size",average discount is high (24.1%); overall profit margin is negative
7,JC-6105,Julie Creighton,EMEA,"$3,914.94","$-2,601.42",-66.45%,6,43,37.1%,Furniture,Tables,"Barricks Conference Table, Rectangular, Safco Stackable Bookrack, Pine, Smead Trays, Wire Frame","Barricks Conference Table, Rectangular",average discount is high (37.1%); overall profit margin is negative; mix is concentrated in low-margin sub-category Tables
8,SR-20425,Sharelle Roach,North Asia,"$12,721.25","$-2,544.73",-20.00%,19,187,23.0%,Technology,Machines,"Lexmark MX611dhe Monochrome Laser Printer, Dania Classic Bookcase, Pine, Epson Card Printer, Wireless",Lexmark MX611dhe Monochrome Laser Printer,average discount is high (23.0%); overall profit margin is negative
9,SN-20560,Skye Norling,Central,"$12,738.71","$-2,527.12",-19.84%,25,201,14.7%,Technology,Phones,"Apple Smart Phone, Full Size, Office Star Executive Leather Armchair, Adjustable, Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind","Apple Smart Phone, Full Size",overall profit margin is negative


In [159]:
# DISCOUNT IMPACT DEEP DIVE

require(df, ["Discount", "Sales", "Profit"])

discount_df = df.copy()

discount_bins = [-0.001, 0.00, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00]
discount_labels = ["0%", "0-5%", "5-10%", "10-20%", "20-30%", "30-50%", "50%+"]

discount_df["Discount_Band"] = pd.cut(
    discount_df["Discount"],
    bins=discount_bins,
    labels=discount_labels,
    include_lowest=True
)

discount_deep_summary = (
    discount_df.groupby("Discount_Band", observed=False)
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Rows=("Profit", "size"),
        Orders=("Order_ID", "nunique") if "Order_ID" in discount_df.columns else ("Sales", "size"),
        Quantity=("Quantity", "sum") if "Quantity" in discount_df.columns else ("Sales", "size"),
        Avg_Discount=("Discount", lambda s: s.mean() * 100),
        Loss_Rows=("Profit", lambda s: (s < 0).sum()),
    )
    .reset_index()
)

discount_deep_summary = discount_deep_summary.rename(
    columns={"Avg_Discount": "Avg_Discount_%"}
)

discount_deep_summary["Profit_Margin_%"] = np.where(
    discount_deep_summary["Sales"] != 0,
    (discount_deep_summary["Profit"] / discount_deep_summary["Sales"]) * 100,
    np.nan
)

discount_deep_summary["Loss_Rate_%"] = np.where(
    discount_deep_summary["Rows"] != 0,
    (discount_deep_summary["Loss_Rows"] / discount_deep_summary["Rows"]) * 100,
    np.nan
)

save_csv(discount_deep_summary, "discount_impact_deep_dive.csv")

display(
    discount_deep_summary[[
        "Discount_Band",
        "Sales",
        "Profit",
        "Rows",
        "Orders",
        "Quantity",
        "Avg_Discount_%",
        "Loss_Rows",
        "Profit_Margin_%",
        "Loss_Rate_%"
    ]].style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
        "Avg_Discount_%": "{:.2f}%",
        "Loss_Rate_%": "{:.1f}%",
        "Rows": "{:,.0f}",
        "Orders": "{:,.0f}",
        "Quantity": "{:,.0f}",
        "Loss_Rows": "{:,.0f}",
    })
)

worst_band = discount_deep_summary.loc[discount_deep_summary["Profit_Margin_%"].idxmin()]
best_band = discount_deep_summary.loc[discount_deep_summary["Profit_Margin_%"].idxmax()]

high_discount_mask = discount_df["Discount"] >= 0.20
high_discount_sales_share = (
    discount_df.loc[high_discount_mask, "Sales"].sum() / discount_df["Sales"].sum()
) * 100

high_discount_loss_rate = (
    (discount_df.loc[high_discount_mask, "Profit"] < 0).mean()
) * 100

print(f"Best profit margin band: {best_band['Discount_Band']} ({best_band['Profit_Margin_%']:.2f}%)")
print(f"Worst profit margin band: {worst_band['Discount_Band']} ({worst_band['Profit_Margin_%']:.2f}%)")
print(f"Sales share from discounts >=20%: {high_discount_sales_share:.1f}%")
print(f"Loss-making row rate for discounts >=20%: {high_discount_loss_rate:.1f}%")

fig = px.bar(
    discount_deep_summary,
    x="Discount_Band",
    y="Profit_Margin_%",
    text=discount_deep_summary["Profit_Margin_%"].map(lambda x: f"{x:.1f}%"),
    color="Profit_Margin_%",
    color_continuous_scale=LOSS_SCALE,
)

fig = style_plotly(
    fig,
    "Discount Band vs Profit Margin",
    x_title="Discount Band",
    y_title="Profit Margin (%)",
    height=DEFAULT_HEIGHT,
)

fig.update_traces(
    textposition="outside",
    hovertemplate=build_hover_template(
        title_template="<b>%{x}</b>",
        lines=[
            {"label": "Profit Margin", "expr": "y", "format_spec": ".2f", "suffix": "%"},
        ],
    ),
)

fig.show()

✅ Saved: ../exports/discount_impact_deep_dive.csv


,Discount_Band,Sales,Profit,Rows,Orders,Quantity,Avg_Discount_%,Loss_Rows,Profit_Margin_%,Loss_Rate_%
0,0%,"$6,992,410.95","$1,770,695.27","29,009","15,211","98,768",0.00%,0,25.32%,0.0%
1,0-5%,"$261,395.62","$57,976.58",461,437,"1,657",0.20%,13,22.18%,2.8%
2,5-10%,"$1,701,223.22","$280,212.68","4,218","2,591","15,950",9.89%,888,16.47%,21.1%
3,10-20%,"$1,757,261.34","$173,254.84","6,274","4,363","23,394",19.22%,"1,463",9.86%,23.3%
4,20-30%,"$382,554.69","$-21,155.61",967,843,"3,679",27.36%,601,-5.53%,62.2%
5,30-50%,"$1,176,031.43","$-380,944.82","6,189","3,674","22,995",43.56%,"5,407",-32.39%,87.4%
6,50%+,"$371,624.66","$-412,581.66","4,172","2,369","11,869",65.81%,"4,172",-111.02%,100.0%


Best profit margin band: 0% (25.32%)
Worst profit margin band: 50%+ (-111.02%)
Sales share from discounts >=20%: 24.8%
Loss-making row rate for discounts >=20%: 68.6%


In [160]:
# REGION × SUB-CATEGORY PROFIT MARGIN HEATMAP

require(df, ["Region", "Sub_Category", "Sales", "Profit"])

region_sub_summary = aggregate_metrics(
    df,
    ["Region", "Sub_Category"],
    include_quantity=True,
    sort_by="Sales",
    ascending=False
).copy()

region_sub_pivot = region_sub_summary.pivot(
    index="Region",
    columns="Sub_Category",
    values="Profit_Margin_%"
)

fig = px.imshow(
    region_sub_pivot,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    zmin=region_sub_summary["Profit_Margin_%"].min(),
    zmax=region_sub_summary["Profit_Margin_%"].max(),
    aspect="auto",
)

fig = style_plotly(
    fig,
    "Profit Margin Heatmap: Region vs Sub-Category",
    x_title="Sub-Category",
    y_title="Region",
    height=700,
    margin=dict(l=80, r=40, t=90, b=80),
)

fig.show()

best_region_sub = region_sub_summary.sort_values("Profit_Margin_%", ascending=False).head(10)
worst_region_sub = region_sub_summary.sort_values("Profit_Margin_%", ascending=True).head(10)

print("TOP 10 REGION × SUB-CATEGORY COMBINATIONS")
display(
    best_region_sub[["Region", "Sub_Category", "Sales", "Profit", "Profit_Margin_%"]]
    .style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
    })
)

print("BOTTOM 10 REGION × SUB-CATEGORY COMBINATIONS")
display(
    worst_region_sub[["Region", "Sub_Category", "Sales", "Profit", "Profit_Margin_%"]]
    .style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
    })
)

save_csv(region_sub_summary, "region_sub_category_profit_margin_summary.csv")

TOP 10 REGION × SUB-CATEGORY COMBINATIONS


,Region,Sub_Category,Sales,Profit,Profit_Margin_%
205,West,Envelopes,"$4,118.10","$1,908.76",46.35%
127,West,Paper,"$26,663.72","$12,119.24",45.45%
198,West,Labels,"$5,078.73","$2,303.12",45.35%
139,East,Paper,"$20,172.60","$9,015.37",44.69%
210,East,Labels,"$2,602.93","$1,129.28",43.38%
202,East,Envelopes,"$4,375.87","$1,812.41",41.42%
91,West,Copiers,"$49,749.24","$19,327.24",38.85%
183,Canada,Copiers,"$7,465.53","$2,663.64",35.68%
215,Canada,Tables,$849.36,$300.18,35.34%
216,East,Fasteners,$819.72,$263.99,32.21%


BOTTOM 10 REGION × SUB-CATEGORY COMBINATIONS


,Region,Sub_Category,Sales,Profit,Profit_Margin_%
196,Caribbean,Machines,"$5,164.50","$-2,603.66",-50.41%
84,Southeast Asia,Tables,"$52,455.41","$-18,618.31",-35.49%
106,East,Tables,"$39,139.81","$-11,025.38",-28.17%
132,North,Furnishings,"$22,731.49","$-6,324.52",-27.82%
38,South,Tables,"$104,892.47","$-27,012.32",-25.75%
151,Southeast Asia,Supplies,"$15,725.37","$-4,034.22",-25.65%
190,Southeast Asia,Fasteners,"$6,249.66","$-1,602.30",-25.64%
111,Southeast Asia,Accessories,"$36,780.24","$-8,641.53",-23.50%
197,Southeast Asia,Labels,"$5,087.61",$-870.36,-17.11%
164,Southeast Asia,Paper,"$11,382.78","$-1,859.01",-16.33%


✅ Saved: ../exports/region_sub_category_profit_margin_summary.csv


PosixPath('../exports/region_sub_category_profit_margin_summary.csv')

In [161]:
# TOP 10 STATES BY SALES — MARGIN ANALYSIS

require(df, ["State", "Sales", "Profit"])

state_sales_table = (
    aggregate_metrics(df, "State", include_quantity=True, sort_by="Sales", ascending=False)
    .head(10)
    .copy()
)

state_sales_table["Sales_Share_%"] = np.where(
    state_sales_table["Sales"].sum() != 0,
    (state_sales_table["Sales"] / state_sales_table["Sales"].sum()) * 100,
    np.nan
)

state_sales_table = state_sales_table[
    [
        "State",
        "Sales",
        "Profit",
        "Profit_Margin_%",
        "Orders",
        "Quantity",
        "Sales_Share_%",
    ]
].copy()

save_csv(state_sales_table, "top_10_sales_states_margin_analysis.csv")

display(
    state_sales_table.style.format({
        "Sales": "${:,.2f}",
        "Profit": "${:,.2f}",
        "Profit_Margin_%": "{:.2f}%",
        "Sales_Share_%": "{:.1f}%",
        "Orders": "{:,.0f}",
        "Quantity": "{:,.0f}",
    })
)

fig = px.bar(
    state_sales_table.sort_values("Sales", ascending=True),
    x="Sales",
    y="State",
    orientation="h",
    color="Profit_Margin_%",
    color_continuous_scale="Blues",
    text=state_sales_table.sort_values("Sales", ascending=True)["Sales"].map(lambda x: f"${x:,.0f}"),
)

fig = style_plotly(
    fig,
    "Top 10 States by Sales with Profit Margin",
    x_title="Sales ($)",
    y_title="State",
    height=DEFAULT_HEIGHT,
    margin=dict(l=120, r=40, t=80, b=40),
)

fig.update_traces(
    textposition="outside",
    hovertemplate=build_hover_template(
        title_template="<b>%{y}</b>",
        lines=[
            {"label": "Sales", "expr": "x", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit", "expr": "customdata[0]", "format_spec": ",.2f", "prefix": "$"},
            {"label": "Profit Margin", "expr": "customdata[1]", "format_spec": ".2f", "suffix": "%"},
        ],
    ),
    customdata=np.stack([
        state_sales_table.sort_values("Sales", ascending=True)["Profit"],
        state_sales_table.sort_values("Sales", ascending=True)["Profit_Margin_%"],
    ], axis=-1),
)

fig.show()

✅ Saved: ../exports/top_10_sales_states_margin_analysis.csv


,State,Sales,Profit,Profit_Margin_%,Orders,Quantity,Sales_Share_%
0,England,"$485,170.97","$99,907.73",20.59%,699,"5,656",17.5%
1,California,"$457,687.63","$76,381.39",16.69%,"1,021","7,667",16.5%
2,Ile-de-France,"$317,822.54","$44,055.92",13.86%,468,"3,839",11.5%
3,New York,"$310,876.27","$74,038.55",23.82%,562,"4,224",11.2%
4,New South Wales,"$270,487.10","$43,695.98",16.15%,388,"2,921",9.8%
5,Queensland,"$238,312.73","$21,608.75",9.07%,374,"2,665",8.6%
6,North Rhine-Westphalia,"$216,451.85","$42,347.87",19.56%,336,"2,660",7.8%
7,Texas,"$170,188.05","$-25,729.36",-15.12%,487,"3,724",6.1%
8,San Salvador,"$153,639.40","$35,883.38",23.36%,307,"2,273",5.5%
9,National Capital,"$152,175.36","$-13,066.08",-8.59%,281,"2,231",5.5%


In [162]:
# PROFITABILITY EFFICIENCY MATRIX
log_step("Building profitability efficiency matrix", "STEP")

# ---------------------------------------------------------
# Aggregate data
# ---------------------------------------------------------
eff_matrix = aggregate_metrics(df, ["Category", "Sub_Category"]).copy()

# ---------------------------------------------------------
# Create normalized metrics
# ---------------------------------------------------------
eff_total_sales = eff_matrix["Sales"].sum()

eff_matrix["Sales Contribution %"] = (eff_matrix["Sales"] / eff_total_sales) * 100
eff_matrix["Profit Margin %"] = np.where(
    eff_matrix["Sales"] != 0,
    (eff_matrix["Profit"] / eff_matrix["Sales"]) * 100,
    0
)

# Bubble size must be non-negative
eff_matrix["Bubble Size"] = eff_matrix["Profit"].abs()

# ---------------------------------------------------------
# Strategic segmentation
# ---------------------------------------------------------
pem_sales_mid = eff_matrix["Sales Contribution %"].median()
pem_margin_mid = eff_matrix["Profit Margin %"].median()

eff_matrix["Segment"] = eff_matrix.apply(
    profitability_segment,
    axis=1,
    sales_mid=pem_sales_mid,
    margin_mid=pem_margin_mid
)

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
fig = px.scatter(
    eff_matrix,
    x="Sales Contribution %",
    y="Profit Margin %",
    size="Bubble Size",
    color="Segment",
    hover_name="Sub_Category",
    text="Sub_Category",
    size_max=55,
    color_discrete_map={
        "Core Winners": PRIMARY_GREEN,
        "High Volume, Low Efficiency": PRIMARY_ORANGE,
        "Hidden Gems": PRIMARY_BLUE,
        "Weak Performers": PRIMARY_RED,
        "Loss Makers": "#8B0000"
    },
    title="Profitability Efficiency Matrix"
)

# Reference lines
fig.add_vline(x=pem_sales_mid, line_dash="dash", line_color = NEUTRAL_GRAY, opacity=0.7)
fig.add_hline(y=pem_margin_mid, line_dash="dash", line_color = NEUTRAL_GRAY, opacity=0.7)

# Styling
fig.update_traces(
    textposition="top center",
    marker=dict(opacity=0.85, line=dict(width=1, color="white"))
)

fig = style_plotly(
    fig,
    title="Profitability Efficiency Matrix",
    x_title="Sales Contribution (%)",
    y_title="Profit Margin (%)",
    legend_title="Business Segment
)

fig.show()

SyntaxError: unterminated string literal (detected at line 74) (1484030822.py, line 74)

In [ ]:
# PARETO ANALYSIS — 80/20 RULE

log_step("Running Pareto analysis", "STEP")

# ---------------------------------------------------------
# Aggregate sales by sub-category
# ---------------------------------------------------------
pareto_df = aggregate_sales_profit(df, "Sub_Category")[
    ["Sub_Category", "Sales"]
].copy()

# ---------------------------------------------------------
# Cumulative calculations
# ---------------------------------------------------------
pareto_df["Cumulative Sales"] = pareto_df["Sales"].cumsum()
pareto_df["Cumulative %"] = (pareto_df["Cumulative Sales"] / pareto_df["Sales"].sum()) * 100
pareto_df["Rank"] = pareto_df.index + 1

# ---------------------------------------------------------
# Find 80% threshold
# ---------------------------------------------------------
threshold_idx = pareto_df[pareto_df["Cumulative %"] <= 80].shape[0]

# Colors: highlight the vital few
colors = [PRIMARY_BLUE if i < threshold_idx else NEUTRAL_GRAY for i in range(len(pareto_df))]

# ---------------------------------------------------------
# Create Pareto chart
# ---------------------------------------------------------
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bars: Sales
fig.add_trace(
    go.Bar(
        x=pareto_df["Sub_Category"],
        y=pareto_df["Sales"],
        name="Sales",
        marker_color=colors,
        text=pareto_df["Sales"].map(lambda x: f"${x:,.0f}"),
        textposition="outside",
        hovertemplate=build_hover_template(
            title_template="<b>%{x}</b>",
            lines=[
                {"label": "Sales", "expr": "y", "format_spec": ",.2f", "prefix": "$"},
            ],
        ),
    ),
    secondary_y=False,
)

# Line: Cumulative %
fig.add_trace(
    go.Scatter(
        x=pareto_df["Sub_Category"],
        y=pareto_df["Cumulative %"],
        name="Cumulative %",
        mode="lines+markers+text",
        line=dict(color=PRIMARY_RED, width=3),
        marker=dict(size=8),
        text=pareto_df["Cumulative %"].round(1).astype(str) + "%",
        textposition="top center",
        hovertemplate=build_hover_template(
            title_template="<b>%{x}</b>",
            lines=[
                {"label": "Cumulative", "expr": "y", "format_spec": ".2f", "suffix": "%"},
            ],
        ),
    ),
    secondary_y=True,
)

# ---------------------------------------------------------
# Reference line at 80%
# ---------------------------------------------------------
fig.add_hline(
    y=80,
    line_dash="dash",
    line_color="black",
    opacity=0.7,
    secondary_y=True
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------

fig = style_plotly(
    fig,
    title="Pareto Analysis — Sales Contribution by Sub-Category",
    bargap=0.25,
    legend_orientation="h",
    legend_y=1.02,
    legend_x=1,
    legend_xanchor="right",
    legend_yanchor="bottom",
    x_tickangle=-45,
    x_showgrid=False
)

fig.update_yaxes(
    title_text="Sales",
    secondary_y=False,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    tickformat=",.0f"
)

fig.update_yaxes(
    title_text="Cumulative %",
    secondary_y=True,
    range=[0, 110],
    showgrid=False
)

# ---------------------------------------------------------
# Optional annotation
# ---------------------------------------------------------
fig.add_annotation(
    x=pareto_df["Sub_Category"].iloc[min(threshold_idx, len(pareto_df)-1)],
    y=82,
    text=f"Top {threshold_idx} sub-categories contribute ~80% of sales",
    showarrow=False,
    font=dict(size=13, color=TEXT_COLOR),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
)

fig.show()